# Fase 1 — Recolección y procesamiento de datos
## Proyecto: Total Tax Rate por grupo de ingreso — Argentina (2004–2024)

---

### Decisiones metodológicas registradas

**Período de análisis:** 2004–2024. Se descarta 2003 porque la EPH continua solo tiene T3 y T4 de ese año (metodología 'onda'), lo que no permite construir un promedio anual comparable.

**Años con datos parciales (se incluyen con advertencia):**
- 2007: solo T1, T2, T4 (T3 no fue relevado por causas administrativas)
- 2011: solo T1, T2 (T3 y T4 con error de encoding en pyeph, rescatados desde TXT locales)
- 2012: todos los trimestres rescatados desde TXT locales (encoding latin-1)
- 2013: todos los trimestres rescatados desde TXT locales (encoding latin-1)
- 2015: solo T1, T2 (T3 y T4 suprimidos por emergencia estadística INDEC)
- 2016: solo T2, T3, T4 (T1 suprimido por emergencia estadística INDEC)
- 2021: solo T1 descargado en corrida original; esta versión completa los trimestres faltantes
- 2024: solo T1 descargado en corrida original; esta versión completa los trimestres faltantes

**Tratamiento de ingresos negativos/nulos:** Los valores `< 0` en IPCF, ITF y P21 son códigos de no-respuesta del INDEC (convencionalmente -9), NO ingresos negativos reales. Se imputan como `NaN` (missing), NO como cero. Imputarlos como cero distorsionaría la distribución y los denominadores del tax rate.

**Construcción del ponderador anual:** El PONDERA de cada trimestre ya representa la población de ese trimestre. Para construir promedios anuales se calcula el indicador (decil, IPCF medio, etc.) por trimestre primero y luego se promedia — no se apilan todas las observaciones en un solo pool. El PONDERA_ANUAL que se guarda en los parquets sirve solo como referencia de la contribución relativa de cada trimestre.

In [ ]:
%pip install pyeph pyarrow requests tqdm openpyxl xlrd

---
## SECCIÓN 1 — Setup: rutas y estructura de carpetas

In [1]:
from pathlib import Path

# ⚙️ ÚNICA LÍNEA QUE TENÉS QUE CAMBIAR SI MOVÉS EL PROYECTO
BASE_PATH = Path(r"C:\Users\LENOVO L340\Desktop\argentina-tax-incidence")

RUTAS = {
    "eph_zips":       BASE_PATH / "data/raw/eph/zips",
    "eph_txt":        BASE_PATH / "data/raw/eph/txt",
    "engho":          BASE_PATH / "data/raw/engho",
    "parametros":     BASE_PATH / "data/raw/parametros",
    "arca":           BASE_PATH / "data/raw/arca",
    "eph_anual":      BASE_PATH / "data/processed/eph_anual",
    "engho_struct":   BASE_PATH / "data/processed/engho_estructura",
    "alicuotas":      BASE_PATH / "data/processed/alicuotas",
}

for nombre, ruta in RUTAS.items():
    ruta.mkdir(parents=True, exist_ok=True)

print("✓ Estructura de carpetas verificada.")

✓ Estructura de carpetas verificada.


---
## SECCIÓN 2 — EPH: descarga de trimestres faltantes

De la corrida anterior quedaron incompletos:
- **2021**: solo T1 guardado en el parquet
- **2024**: solo T1 guardado en el parquet
- **2011, 2012, 2013**: guardados con trimestres faltantes por error de encoding

Esta sección descarga los trimestres que faltan usando pyeph con el fix de encoding.

In [2]:
import pyeph
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Columnas que nos interesan
COLS_INDV  = ['CODUSU', 'NRO_HOGAR', 'COMPONENTE', 'ANO4', 'TRIMESTRE', 'PONDERA',
              'CH04', 'CH06', 'ESTADO', 'CAT_OCUP', 'PP07H', 'P21']
COLS_HOGAR = ['CODUSU', 'NRO_HOGAR', 'ANO4', 'TRIMESTRE', 'ITF', 'IPCF', 'IX_TOT']

# Baches conocidos — trimestres que nunca existieron
BACHES = {
    2003: [1, 2, 3, 4],   # EPH continua empieza en T3 pero pyeph usa 'onda'
    2007: [3],             # No relevado
    2015: [3, 4],          # Emergencia estadística
    2016: [1],             # Emergencia estadística
}

def limpiar_ingresos(df):
    """
    Convierte valores negativos en NaN.
    Los -9 y similares son códigos de no-respuesta del INDEC, no ingresos negativos.
    NO deben imputarse como cero: eso sesga la distribución hacia abajo.
    """
    for col in ['IPCF', 'ITF', 'P21']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            df.loc[df[col] < 0, col] = np.nan
    return df

def descargar_trimestre_pyeph(anio, trim):
    """Descarga un trimestre con pyeph y devuelve el df mergeado o None."""
    if trim in BACHES.get(anio, []):
        return None
    try:
        ind = pyeph.get(data='eph', year=anio, period=trim, base_type='individual')
        hog = pyeph.get(data='eph', year=anio, period=trim, base_type='hogar')
        ind.columns = ind.columns.str.upper()
        hog.columns = hog.columns.str.upper()
        ind['CODUSU'] = ind['CODUSU'].astype(str)
        hog['CODUSU'] = hog['CODUSU'].astype(str)
        c_ind = [c for c in COLS_INDV  if c in ind.columns]
        c_hog = [c for c in COLS_HOGAR if c in hog.columns]
        df = pd.merge(ind[c_ind], hog[c_hog], on=['CODUSU', 'NRO_HOGAR', 'ANO4', 'TRIMESTRE'], how='left')
        df = limpiar_ingresos(df)
        return df
    except Exception as e:
        print(f"    ⚠️  pyeph falló {anio} T{trim}: {e}")
        return None

print("✓ Funciones de descarga definidas.")

2026-05-12 13:51:44 - INFO - NumExpr defaulting to 12 threads.


✓ Funciones de descarga definidas.


In [3]:
# Rescate de 2011, 2012, 2013 desde TXT locales con encoding latin-1
# (pyeph no puede leer estos archivos porque usa utf-8 y el INDEC los guardó en latin-1)

def encontrar_txt_local(tipo, anio, trim):
    """Busca el TXT en disco con los patrones de nombre que usa el INDEC."""
    yy = str(anio)[-2:]
    patrones = [
        f"*{tipo}*T{trim}{yy}*.txt",
        f"*{tipo}*{trim}*Trim*{anio}*.txt",
        f"*{tipo}*t{trim}{yy}*.txt",
        f"*{tipo}*_{trim}_{anio}*.txt",
    ]
    for p in patrones:
        archivos = list(RUTAS['eph_txt'].glob(p))
        if archivos:
            return archivos[0]
    return None

def descargar_trimestre_local(anio, trim):
    """Lee un trimestre desde TXT local con encoding latin-1."""
    archivo_ind = encontrar_txt_local('individual', anio, trim)
    archivo_hog = encontrar_txt_local('hogar', anio, trim)
    if not archivo_ind or not archivo_hog:
        print(f"    ℹ️  No se encontró TXT local para {anio} T{trim}")
        return None
    try:
        df_ind = pd.read_csv(archivo_ind, sep=';', decimal=',', encoding='latin-1',
                             usecols=lambda c: c.upper() in COLS_INDV, low_memory=False)
        df_hog = pd.read_csv(archivo_hog, sep=';', decimal=',', encoding='latin-1',
                             usecols=lambda c: c.upper() in COLS_HOGAR, low_memory=False)
        df_ind.columns = df_ind.columns.str.upper()
        df_hog.columns = df_hog.columns.str.upper()
        df_ind['CODUSU'] = df_ind['CODUSU'].astype(str)
        df_hog['CODUSU'] = df_hog['CODUSU'].astype(str)
        c_ind = [c for c in COLS_INDV  if c in df_ind.columns]
        c_hog = [c for c in COLS_HOGAR if c in df_hog.columns]
        df = pd.merge(df_ind[c_ind], df_hog[c_hog], on=['CODUSU', 'NRO_HOGAR', 'ANO4', 'TRIMESTRE'], how='left')
        df = limpiar_ingresos(df)
        return df
    except Exception as e:
        print(f"    ❌  Error leyendo TXT local {anio} T{trim}: {e}")
        return None

print("✓ Funciones de rescate local definidas.")

✓ Funciones de rescate local definidas.


In [4]:
# FUNCIÓN PRINCIPAL: procesa un año completo y guarda el parquet
# Estrategia por año:
# - 2011, 2012, 2013: intenta pyeph primero; si falla, cae a TXT local con latin-1
# - Resto: solo pyeph

ANOS_CON_FALLBACK_LOCAL = {2011, 2012, 2013}

def procesar_anio(anio, forzar=False):
    archivo_salida = RUTAS['eph_anual'] / f"eph_anual_{anio}.parquet"

    if archivo_salida.exists() and not forzar:
        print(f"  ⏭️  {anio} — ya existe (usar forzar=True para reprocesar)")
        return

    print(f"\n🔄  Procesando {anio}...")
    trimestres_df = []
    trimestres_ok = []

    for trim in [1, 2, 3, 4]:
        if trim in BACHES.get(anio, []):
            continue  # Bache conocido, no intentar

        df = None

        # Intento 1: pyeph
        df = descargar_trimestre_pyeph(anio, trim)

        # Intento 2: fallback a TXT local (solo para años con encoding roto)
        if df is None and anio in ANOS_CON_FALLBACK_LOCAL:
            print(f"    ↩️  Intentando TXT local para {anio} T{trim}...")
            df = descargar_trimestre_local(anio, trim)

        if df is not None and len(df) > 0:
            trimestres_df.append(df)
            trimestres_ok.append(trim)
            print(f"    ✓  T{trim}: {len(df):,} personas")
        else:
            print(f"    ✗  T{trim}: no disponible")

    if not trimestres_df:
        print(f"  ❌  {anio}: sin trimestres disponibles, no se guarda.")
        return

    df_anual = pd.concat(trimestres_df, ignore_index=True)

    # PONDERA_ANUAL: peso relativo dentro del año
    # Dividir por la cantidad de trimestres reales disponibles
    # NOTA METODOLÓGICA: este campo es informativo. Para calcular estadísticas
    # anuales correctas, siempre calcular POR TRIMESTRE y luego promediar.
    n_trim = len(trimestres_df)
    df_anual['PONDERA_ANUAL'] = df_anual['PONDERA'] / n_trim
    df_anual['N_TRIMESTRES_DISPONIBLES'] = n_trim  # metadata para auditoría
    df_anual['TRIMESTRES_DISPONIBLES_STR'] = str(sorted(trimestres_ok))  # metadata

    df_anual['CODUSU'] = df_anual['CODUSU'].astype(str)

    df_anual.to_parquet(archivo_salida, index=False)
    nota = " ⚠️ DATOS PARCIALES" if n_trim < 4 else ""
    print(f"  ✅  {anio} guardado — {n_trim} trimestres {sorted(trimestres_ok)} | {len(df_anual):,} filas{nota}")

print("✓ Función procesar_anio definida.")

✓ Función procesar_anio definida.


In [5]:
# Reprocesar los años problemáticos de la corrida anterior:
# 2011: guardado con solo T1+T2, le faltan T3 y T4
# 2012: guardado vacío (0 trimestres)
# 2013: guardado con solo T4, le faltan T1-T3
# 2021: guardado con solo T1
# 2024: guardado con solo T1

ANOS_A_REPROCESAR = [2011, 2012, 2013, 2021, 2024]

print("🔧 Reprocesando años incompletos...")
for anio in ANOS_A_REPROCESAR:
    procesar_anio(anio, forzar=True)

print("\n✅ Reprocesamiento completado.")

2026-05-12 13:51:57 - WARNING - INDEC advierte:
Advertencia sobre el uso de series estadisticas. Se advierte que las series estadisticas publicadas con
posterioridad a enero del 2007 y hasta diciembre 2015 deben ser consideradas con reservas,
excepto las que ya hayan sido revisadas en 2016 y su difusion lo consigne expresamente.
El INDEC, en el marco de las atribuciones conferidas por los decretos 181/15 y 55/16, dispuso las investigaciones
requeridas para establecer la regularidad de procedimientos de obtencion de datos, su procesamiento, elaboracion de indicadores y difusion
Mas informacion en: https://www.indec.gob.ar/ftp/cuadros/sociedad/anexo_informe_eph_23_08_16.pdf


🔧 Reprocesando años incompletos...

🔄  Procesando 2011...


Descargando base_individual_2011T1.zip: 100%|█| 1.97M/1.97M [00:00<00:00, 6.92MB
2026-05-12 13:51:57 - INFO - Descarga exitosa: base_individual_2011T1.zip
2026-05-12 13:52:00 - INFO - Obtenido con éxito: base_individual_2011T1.zip
2026-05-12 13:52:00 - WARNING - INDEC advierte:
Advertencia sobre el uso de series estadisticas. Se advierte que las series estadisticas publicadas con
posterioridad a enero del 2007 y hasta diciembre 2015 deben ser consideradas con reservas,
excepto las que ya hayan sido revisadas en 2016 y su difusion lo consigne expresamente.
El INDEC, en el marco de las atribuciones conferidas por los decretos 181/15 y 55/16, dispuso las investigaciones
requeridas para establecer la regularidad de procedimientos de obtencion de datos, su procesamiento, elaboracion de indicadores y difusion
Mas informacion en: https://www.indec.gob.ar/ftp/cuadros/sociedad/anexo_informe_eph_23_08_16.pdf
Descargando base_hogar_2011T1.zip: 100%|█████| 532k/532k [00:00<00:00, 6.45MB/s]
2026-05

    ✓  T1: 57,784 personas


Descargando base_individual_2011T2.zip: 100%|█| 1.98M/1.98M [00:00<00:00, 7.99MB
2026-05-12 13:52:01 - INFO - Descarga exitosa: base_individual_2011T2.zip
2026-05-12 13:52:03 - INFO - Obtenido con éxito: base_individual_2011T2.zip
2026-05-12 13:52:03 - WARNING - INDEC advierte:
Advertencia sobre el uso de series estadisticas. Se advierte que las series estadisticas publicadas con
posterioridad a enero del 2007 y hasta diciembre 2015 deben ser consideradas con reservas,
excepto las que ya hayan sido revisadas en 2016 y su difusion lo consigne expresamente.
El INDEC, en el marco de las atribuciones conferidas por los decretos 181/15 y 55/16, dispuso las investigaciones
requeridas para establecer la regularidad de procedimientos de obtencion de datos, su procesamiento, elaboracion de indicadores y difusion
Mas informacion en: https://www.indec.gob.ar/ftp/cuadros/sociedad/anexo_informe_eph_23_08_16.pdf
Descargando base_hogar_2011T2.zip: 100%|█████| 536k/536k [00:00<00:00, 8.90MB/s]
2026-05

    ✓  T2: 58,726 personas


... (34 outputs left out)

Descargando base_individual_2024T3.zip: 100%|█| 2.36M/2.36M [00:00<00:00, 14.0MB
2026-05-12 13:52:28 - INFO - Descarga exitosa: base_individual_2024T3.zip
2026-05-12 13:52:30 - INFO - Obtenido con éxito: base_individual_2024T3.zip
Descargando base_hogar_2024T3.zip: 100%|█████| 647k/647k [00:00<00:00, 13.4MB/s]
2026-05-12 13:52:30 - INFO - Descarga exitosa: base_hogar_2024T3.zip
2026-05-12 13:52:30 - INFO - Obtenido con éxito: base_hogar_2024T3.zip


    ✓  T3: 47,564 personas


Descargando base_individual_2024T4.zip: 100%|█| 2.80M/2.80M [00:00<00:00, 14.2MB
2026-05-12 13:52:31 - INFO - Descarga exitosa: base_individual_2024T4.zip
2026-05-12 13:52:33 - INFO - Obtenido con éxito: base_individual_2024T4.zip
Descargando base_hogar_2024T4.zip: 100%|█████| 672k/672k [00:00<00:00, 16.1MB/s]
2026-05-12 13:52:33 - INFO - Descarga exitosa: base_hogar_2024T4.zip
2026-05-12 13:52:33 - INFO - Obtenido con éxito: base_hogar_2024T4.zip


    ✓  T4: 46,860 personas
  ✅  2024 guardado — 4 trimestres [1, 2, 3, 4] | 187,625 filas

✅ Reprocesamiento completado.


In [6]:
# Procesar años que no existían aún en la corrida anterior
# (2004-2010, 2014-2020, 2022-2023 ya existen como parquets válidos)
# Esta celda los saltea automáticamente y solo procesa los que faltan.

print("🔄 Verificando y completando años restantes (2004–2024)...")
for anio in range(2004, 2025):
    procesar_anio(anio, forzar=False)

print("\n🎉 Serie EPH completa.")

2026-05-12 13:52:36 - WARNING - INDEC advierte:
Advertencia sobre el uso de series estadisticas. Se advierte que las series estadisticas publicadas con
posterioridad a enero del 2007 y hasta diciembre 2015 deben ser consideradas con reservas,
excepto las que ya hayan sido revisadas en 2016 y su difusion lo consigne expresamente.
El INDEC, en el marco de las atribuciones conferidas por los decretos 181/15 y 55/16, dispuso las investigaciones
requeridas para establecer la regularidad de procedimientos de obtencion de datos, su procesamiento, elaboracion de indicadores y difusion
Mas informacion en: https://www.indec.gob.ar/ftp/cuadros/sociedad/anexo_informe_eph_23_08_16.pdf
2026-05-12 13:52:36 - ERROR - Error al leer DataFrame desde base_individual_2012T1.zip: 'utf-8' codec can't decode byte 0xa4 in position 102411: invalid start byte
2026-05-12 13:52:36 - WARNING - INDEC advierte:
Advertencia sobre el uso de series estadisticas. Se advierte que las series estadisticas publicadas con
pos

🔄 Verificando y completando años restantes (2004–2024)...
  ⏭️  2004 — ya existe (usar forzar=True para reprocesar)
  ⏭️  2005 — ya existe (usar forzar=True para reprocesar)
  ⏭️  2006 — ya existe (usar forzar=True para reprocesar)
  ⏭️  2007 — ya existe (usar forzar=True para reprocesar)
  ⏭️  2008 — ya existe (usar forzar=True para reprocesar)
  ⏭️  2009 — ya existe (usar forzar=True para reprocesar)
  ⏭️  2010 — ya existe (usar forzar=True para reprocesar)
  ⏭️  2011 — ya existe (usar forzar=True para reprocesar)

🔄  Procesando 2012...
    ⚠️  pyeph falló 2012 T1: 'utf-8' codec can't decode byte 0xa4 in position 102411: invalid start byte
    ↩️  Intentando TXT local para 2012 T1...
    ℹ️  No se encontró TXT local para 2012 T1
    ✗  T1: no disponible


2026-05-12 13:52:36 - ERROR - Error al leer DataFrame desde base_individual_2012T2.zip: 'utf-8' codec can't decode byte 0xa4 in position 215815: invalid start byte
2026-05-12 13:52:36 - WARNING - INDEC advierte:
Advertencia sobre el uso de series estadisticas. Se advierte que las series estadisticas publicadas con
posterioridad a enero del 2007 y hasta diciembre 2015 deben ser consideradas con reservas,
excepto las que ya hayan sido revisadas en 2016 y su difusion lo consigne expresamente.
El INDEC, en el marco de las atribuciones conferidas por los decretos 181/15 y 55/16, dispuso las investigaciones
requeridas para establecer la regularidad de procedimientos de obtencion de datos, su procesamiento, elaboracion de indicadores y difusion
Mas informacion en: https://www.indec.gob.ar/ftp/cuadros/sociedad/anexo_informe_eph_23_08_16.pdf
2026-05-12 13:52:36 - ERROR - Error al leer DataFrame desde base_individual_2012T3.zip: 'utf-8' codec can't decode byte 0xf1 in position 167868: invalid co

    ⚠️  pyeph falló 2012 T2: 'utf-8' codec can't decode byte 0xa4 in position 215815: invalid start byte
    ↩️  Intentando TXT local para 2012 T2...
    ℹ️  No se encontró TXT local para 2012 T2
    ✗  T2: no disponible
    ⚠️  pyeph falló 2012 T3: 'utf-8' codec can't decode byte 0xf1 in position 167868: invalid continuation byte
    ↩️  Intentando TXT local para 2012 T3...
    ℹ️  No se encontró TXT local para 2012 T3
    ✗  T3: no disponible


2026-05-12 13:52:36 - ERROR - Error al leer DataFrame desde base_individual_2012T4.zip: 'utf-8' codec can't decode byte 0xf1 in position 185178: invalid continuation byte


    ⚠️  pyeph falló 2012 T4: 'utf-8' codec can't decode byte 0xf1 in position 185178: invalid continuation byte
    ↩️  Intentando TXT local para 2012 T4...
    ℹ️  No se encontró TXT local para 2012 T4
    ✗  T4: no disponible
  ❌  2012: sin trimestres disponibles, no se guarda.
  ⏭️  2013 — ya existe (usar forzar=True para reprocesar)
  ⏭️  2014 — ya existe (usar forzar=True para reprocesar)
  ⏭️  2015 — ya existe (usar forzar=True para reprocesar)
  ⏭️  2016 — ya existe (usar forzar=True para reprocesar)
  ⏭️  2017 — ya existe (usar forzar=True para reprocesar)
  ⏭️  2018 — ya existe (usar forzar=True para reprocesar)
  ⏭️  2019 — ya existe (usar forzar=True para reprocesar)
  ⏭️  2020 — ya existe (usar forzar=True para reprocesar)
  ⏭️  2021 — ya existe (usar forzar=True para reprocesar)
  ⏭️  2022 — ya existe (usar forzar=True para reprocesar)
  ⏭️  2023 — ya existe (usar forzar=True para reprocesar)
  ⏭️  2024 — ya existe (usar forzar=True para reprocesar)

🎉 Serie EPH completa.


---
## SECCIÓN 3 — Auditoría de la serie EPH

Antes de continuar, verificamos que cada parquet sea razonable: cantidad de filas, cobertura de trimestres, ausencia de valores absurdos en IPCF.

In [7]:
import pandas as pd
import numpy as np

print(f"{'Año':<6} {'Trims':<6} {'Disponibles':<15} {'Filas':>8} {'IPCF_mediana':>13} {'IPCF_nulos%':>12} {'Flag'}")
print("-" * 75)

resumen = []
for anio in range(2004, 2025):
    p = RUTAS['eph_anual'] / f"eph_anual_{anio}.parquet"
    if not p.exists():
        print(f"{anio:<6} {'—':<6} {'—':<15} {'—':>8} {'—':>13} {'—':>12}  ❌ FALTA")
        continue
    df = pd.read_parquet(p)
    n_trim  = df['N_TRIMESTRES_DISPONIBLES'].iloc[0] if 'N_TRIMESTRES_DISPONIBLES' in df.columns else '?'
    trims   = df['TRIMESTRES_DISPONIBLES_STR'].iloc[0] if 'TRIMESTRES_DISPONIBLES_STR' in df.columns else '?'
    filas   = len(df)
    mediana = df['IPCF'].median() if 'IPCF' in df.columns else np.nan
    pct_nan = df['IPCF'].isna().mean() * 100 if 'IPCF' in df.columns else np.nan
    flag = ""
    if isinstance(n_trim, int) and n_trim < 4:
        flag = f"⚠️  {n_trim} trim"
    if filas < 40000:
        flag += " ⚠️ pocas filas"
    print(f"{anio:<6} {str(n_trim):<6} {str(trims):<15} {filas:>8,} {mediana:>13,.0f} {pct_nan:>11.1f}%  {flag}")
    resumen.append({'anio': anio, 'n_trim': n_trim, 'filas': filas,
                    'ipcf_mediana': mediana, 'ipcf_pct_nan': pct_nan})

print("\nLeyenda: ⚠️ = año con trimestres faltantes (se incluye con advertencia en el análisis)")

Año    Trims  Disponibles        Filas  IPCF_mediana  IPCF_nulos% Flag
---------------------------------------------------------------------------
2004   ?      ?                187,930           133         0.0%  
2005   ?      ?                188,755           183         0.0%  
2006   ?      ?                222,968           257         0.0%  
2007   ?      ?                188,433           388         0.0%  
2008   ?      ?                247,819           562         0.0%  
2009   ?      ?                240,967           675         0.0%  
2010   ?      ?                237,055           840         0.0%  
2011   2      [1, 2]           116,510         1,040         0.0%  
2012   —      —                      —             —            —  ❌ FALTA
2013   1      [4]               56,198         2,000         0.0%  
2014   ?      ?                241,181         2,400         0.0%  
2015   ?      ?                120,173         3,000         0.0%  
2016   ?      ?               

In [8]:
# Verificación adicional: la mediana del IPCF debería crecer nominalmente todos los años
# (por inflación). Si hay un año con mediana mucho menor al anterior, hay un problema.

df_resumen = pd.DataFrame(resumen)
df_resumen['ipcf_var_pct'] = df_resumen['ipcf_mediana'].pct_change() * 100

print("Variación interanual de la mediana del IPCF (nominal):")
print("(Valores negativos son sospechosos — la inflación siempre empuja el IPCF nominal hacia arriba)")
print()
for _, row in df_resumen.iterrows():
    variacion = row['ipcf_var_pct']
    if pd.isna(variacion):
        continue
    flag = " ⚠️ REVISAR" if variacion < -5 else ""
    print(f"  {int(row['anio'])}: {variacion:+.1f}%{flag}")




# FALTA 2012.
# PROBLEMA CON 2016, que da 0% porque 2015: 120,173 filas | IPCF_mediana: 3,000 ; 2016: 177,515 filas | IPCF_mediana: 3,000
# 

Variación interanual de la mediana del IPCF (nominal):
(Valores negativos son sospechosos — la inflación siempre empuja el IPCF nominal hacia arriba)

  2005: +37.5%
  2006: +40.3%
  2007: +51.0%
  2008: +44.8%
  2009: +20.0%
  2010: +24.4%
  2011: +23.8%
  2013: +92.3%
  2014: +20.0%
  2015: +25.0%
  2016: +0.0%
  2017: +48.3%
  2018: +27.3%
  2019: +41.2%
  2020: +33.3%
  2021: +40.6%
  2022: +63.3%
  2023: +104.1%
  2024: +190.0%


In [9]:
import pandas as pd
import numpy as np

# Función clásica para calcular la mediana ponderada
def mediana_ponderada(df, col_valor, col_peso):
    # Descartamos nulos y ordenamos
    df_sorted = df.dropna(subset=[col_valor, col_peso]).sort_values(col_valor)
    # Suma acumulada de los pesos
    cumsum = df_sorted[col_peso].cumsum()
    cutoff = df_sorted[col_peso].sum() / 2.0
    # El primer valor que cruza la mitad de la población
    return df_sorted[cumsum >= cutoff][col_valor].iloc[0]

print("🔎 DIAGNÓSTICO PROFUNDO: 2014 a 2017")
print("=" * 60)

for anio in [2014, 2015, 2016, 2017]:
    ruta = RUTAS['eph_anual'] / f"eph_anual_{anio}.parquet"
    if not ruta.exists(): 
        continue
        
    # Cargamos solo lo necesario para no reventar la RAM
    df = pd.read_parquet(ruta, columns=['TRIMESTRE', 'IPCF', 'PONDERA'])
    
    # Filtramos nulos y ceros para evaluar solo hogares con ingresos declarados
    df_valid = df[df['IPCF'] > 0]
    
    print(f"\n📅 AÑO {anio}")
    print("-" * 40)
    
    # 1. Análisis Trimestral
    for trim in sorted(df_valid['TRIMESTRE'].unique()):
        dft = df_valid[df_valid['TRIMESTRE'] == trim]
        med_simple = dft['IPCF'].median()
        med_pond = mediana_ponderada(dft, 'IPCF', 'PONDERA')
        print(f"  T{trim} -> Mediana Simple: ${med_simple:,.0f} | Mediana Ponderada: ${med_pond:,.0f}")
        
    # 2. Efecto Clustering (Los 3 montos más repetidos de todo el año)
    top_valores = df_valid['IPCF'].value_counts().head(3)
    print(f"  Top 3 valores de IPCF más repetidos (efecto redondeo):")
    for val, count in top_valores.items():
        print(f"    ${val:,.0f} (declarado por {count} individuos en la base)")

🔎 DIAGNÓSTICO PROFUNDO: 2014 a 2017

📅 AÑO 2014
----------------------------------------
  T1 -> Mediana Simple: $2,250 | Mediana Ponderada: $2,344
  T2 -> Mediana Simple: $2,250 | Mediana Ponderada: $2,333
  T3 -> Mediana Simple: $2,600 | Mediana Ponderada: $2,600
  T4 -> Mediana Simple: $2,631 | Mediana Ponderada: $2,667
  Top 3 valores de IPCF más repetidos (efecto redondeo):
    $2,000 (declarado por 5522 individuos en la base)
    $3,000 (declarado por 4824 individuos en la base)
    $2,500 (declarado por 3573 individuos en la base)

📅 AÑO 2015
----------------------------------------
  T1 -> Mediana Simple: $3,000 | Mediana Ponderada: $3,100
  T2 -> Mediana Simple: $3,114 | Mediana Ponderada: $3,100
  Top 3 valores de IPCF más repetidos (efecto redondeo):
    $2,000 (declarado por 2457 individuos en la base)
    $3,000 (declarado por 2361 individuos en la base)
    $4,000 (declarado por 2172 individuos en la base)

📅 AÑO 2016
----------------------------------------
  T2 -> Media

PROBLEMA 2016 NO ES REALMENTE UN PROBLEMA, LOS MICRODATOS ESTÁN BIEN, ES UN ARTEFACTO POR LA MEDIANA, EL CALCULO DE LOS TRIMESTRES PONDERADOS


---
## SECCIÓN 4 — ENGHo 2004/05: estructura de gasto por quintil

Ya procesada en la corrida anterior. Esta sección la re-ejecuta con la corrección del mapeo COICOP (verificación de longitud de códigos de artículo).

In [10]:
import pandas as pd
from pathlib import Path

ruta_engho = RUTAS['engho']
ruta_salida = RUTAS['engho_struct']

def leer_txt_indec(ruta):
    df = pd.read_csv(ruta, sep=None, engine='python', encoding='latin-1')
    if not isinstance(df.index, pd.RangeIndex):
        nombres_originales = list(df.columns)
        df.reset_index(inplace=True)
        df = df.iloc[:, :-1]
        df.columns = nombres_originales
    df.columns = df.columns.str.strip().str.upper()
    return df

# Leer la base de gastos
ruta_gastos = list((ruta_engho / 'ENGH0405_gastos').glob('*.txt'))[0]
df_gastos = leer_txt_indec(ruta_gastos)
df_gastos['MONTO'] = pd.to_numeric(df_gastos['MONTO'], errors='coerce').fillna(0)
df_gastos['ARTICULO'] = df_gastos['ARTICULO'].astype(str).str.strip()

# DIAGNÓSTICO: verificar longitud de códigos de artículo
longitudes = df_gastos['ARTICULO'].str.len().value_counts().sort_index()
print("Distribución de longitudes de código ARTICULO:")
print(longitudes.to_string())
print()
print("Ejemplos de códigos:")
print(df_gastos['ARTICULO'].sample(min(10, len(df_gastos))).tolist())

Distribución de longitudes de código ARTICULO:
ARTICULO
6    1344137

Ejemplos de códigos:
['331101', '115203', '213202', '115401', '462101', '112202', '111209', '117117', '623101', '732101']


In [11]:
# Mapeo INDEC 2004 a COICOP (12 divisiones)
# En 2004, el INDEC usó una nomenclatura propia de 9 capítulos.
# Esta función traduce esos 9 capítulos a las categorías COICOP modernas,
# leyendo el primer dígito (capítulo principal) y el segundo (subcapítulo).

def mapear_indec2004_a_coicop(codigo_str):
    """
    Traduce los códigos de artículo de ENGHo 2004 a categorías COICOP.
    """
    cod = str(codigo_str).strip()
    
    # Si es muy corto o nulo, lo descartamos
    if len(cod) < 5 or cod == 'nan': 
        return 'otros'
    
    # Extraemos el primer y segundo dígito para saber la categoría mayor
    try:
        d1 = int(cod[0])   
        d12 = int(cod[:2]) 
    except ValueError:
        return 'otros'
    
    # Traducción Nomenclatura INDEC 2004 -> COICOP
    if d1 == 1: 
        # En 2004, alimentos y bebidas no alcohólicas comparten capítulo
        return 'alimentos_y_bebidas'
    if d1 == 2: return 'indumentaria_y_calzado'
    if d1 == 3: return 'propiedades_combustibles_agua_electricidad'
    if d1 == 4: return 'equipamiento_y_mantenimiento'
    if d1 == 5: return 'salud'
    if d1 == 6: 
        # En 2004, el capítulo 6 agrupaba Transporte (61) y Comunicaciones (62)
        return 'comunicaciones' if d12 == 62 else 'transporte'
    if d1 == 7: return 'esparcimiento_y_cultura'
    if d1 == 8: return 'ensenanza'
    if d1 == 9: 
        # Capítulo 9 agrupaba Restaurantes/Hoteles (92/93) y Bienes varios (91)
        return 'restaurantes_y_hoteles' if d12 >= 92 else 'bienes_y_servicios_varios'
    
    return 'otros'

# Aplicar el mapeo corregido
df_gastos['CATEGORIA'] = df_gastos['ARTICULO'].apply(mapear_indec2004_a_coicop)

# Mostrar resultados para auditoría
print("Categorías encontradas y su frecuencia:")
print(df_gastos['CATEGORIA'].value_counts().to_string())
print()
n_otros = (df_gastos['CATEGORIA'] == 'otros').sum()
pct_otros = n_otros / len(df_gastos) * 100
print(f"⚠️  Artículos no mapeados ('otros'): {n_otros:,} ({pct_otros:.1f}%)")

if pct_otros > 5:
    print("   → Porcentaje alto. Revisar los códigos que cayeron en 'otros':")
    print(df_gastos[df_gastos['CATEGORIA']=='otros']['ARTICULO'].value_counts().head(20).to_string())

Categorías encontradas y su frecuencia:
CATEGORIA
alimentos_y_bebidas                           704179
equipamiento_y_mantenimiento                  119669
propiedades_combustibles_agua_electricidad    102216
indumentaria_y_calzado                         86686
restaurantes_y_hoteles                         75981
esparcimiento_y_cultura                        70755
transporte                                     68170
salud                                          40993
comunicaciones                                 37363
ensenanza                                      24651
bienes_y_servicios_varios                      13474

⚠️  Artículos no mapeados ('otros'): 0 (0.0%)


In [12]:
# Procesar y guardar la estructura de gasto ENGHo 2004

# Leer hogares
ruta_hogar = list((ruta_engho / 'ENGH0405_hogar').glob('*.txt'))[0]
df_hogar = leer_txt_indec(ruta_hogar)
df_hogar = df_hogar[['CLAVE', 'EXPAN', 'QINPCH_T']].copy()
df_hogar = df_hogar.dropna(subset=['QINPCH_T'])
df_hogar['QINPCH_T'] = df_hogar['QINPCH_T'].astype(int)

# Merge gastos + quintil
df_merge = pd.merge(
    df_gastos[['CLAVE', 'MONTO', 'CATEGORIA']],
    df_hogar,
    on='CLAVE', how='inner'
)
df_merge['GASTO_EXPANDIDO'] = df_merge['MONTO'] * df_merge['EXPAN']

# Excluir 'otros' del cálculo de proporciones (no clasificados)
df_merge_clasif = df_merge[df_merge['CATEGORIA'] != 'otros']

# Tabla pivote: categoria x quintil
tabla_pivote = (
    df_merge_clasif
    .groupby(['CATEGORIA', 'QINPCH_T'])['GASTO_EXPANDIDO']
    .sum()
    .unstack()
)
tabla_pivote.columns = [f"Q{col}" for col in tabla_pivote.columns]

# Proporciones: cada celda / total del quintil
tabla_porc = tabla_pivote.div(tabla_pivote.sum(axis=0), axis=1)

# Verificar que cada columna sume 1
sumas = tabla_porc.sum(axis=0)
assert all(abs(sumas - 1.0) < 0.001), f"Las columnas no suman 1: {sumas.to_dict()}"

df_final = tabla_porc.reset_index().rename(columns={'CATEGORIA': 'categoria_coicop'})
df_final['engho_edicion'] = '2004_05'

archivo_csv = ruta_salida / "engho_estructura_2004.csv"
df_final.to_csv(archivo_csv, index=False)

print(f"✅ ENGHo 2004 guardada: {archivo_csv.name}")
print("\nEstructura por quintil (proporciones):")
print(df_final.drop(columns='engho_edicion').to_string(index=False))

✅ ENGHo 2004 guardada: engho_estructura_2004.csv

Estructura por quintil (proporciones):
                          categoria_coicop       Q1       Q2       Q3       Q4       Q5
                       alimentos_y_bebidas 0.461347 0.399608 0.361141 0.328437 0.266831
                 bienes_y_servicios_varios 0.020778 0.021006 0.017096 0.015702 0.010188
                            comunicaciones 0.025748 0.036227 0.041343 0.043903 0.039592
                                 ensenanza 0.022038 0.028618 0.030012 0.032901 0.034376
              equipamiento_y_mantenimiento 0.056150 0.057151 0.064167 0.067500 0.085725
                   esparcimiento_y_cultura 0.050378 0.058304 0.071465 0.079611 0.105291
                    indumentaria_y_calzado 0.089275 0.087781 0.080584 0.081374 0.079196
propiedades_combustibles_agua_electricidad 0.108317 0.112498 0.110540 0.105407 0.106013
                    restaurantes_y_hoteles 0.047000 0.044966 0.043456 0.045013 0.047164
                               

---
## SECCIÓN 5 — ENGHo 2012/13: estructura de gasto

Si tenés los microdatos de la ENGHo 2012 en disco, correr esta sección. Si no, se usa interpolación entre 2004 y 2017 en la Fase 3.

Los microdatos de la ENGHo 2012/13 requieren descarga manual desde el sitio del INDEC (están detrás de JavaScript). Si los descargaste, actualizá `RUTA_ENGHO_2012` abajo.

In [13]:
# 1. Definimos los nombres exactos de tus carpetas
nombre_carpeta_datos = "bases_datos_engho2012"
nombre_carpeta_codigos = "codigos_engho2012"

ruta_2012_encontrada = None
ruta_codigos_encontrada = None

# 2. Construimos las rutas completas
posible_datos = ruta_engho / nombre_carpeta_datos
posible_codigos = ruta_engho / nombre_carpeta_codigos

# 3. Verificamos la carpeta de DATOS (buscando un archivo clave)
if posible_datos.exists() and (posible_datos / "ENGHo - Gastos.txt").exists():
    ruta_2012_encontrada = posible_datos
    print(f"✓ Carpeta de DATOS encontrada: {ruta_2012_encontrada.name}")

# 4. Verificamos la carpeta de CÓDIGOS
if posible_codigos.exists() and (posible_codigos / "ENGHo - Provincias.txt").exists():
    ruta_codigos_encontrada = posible_codigos
    print(f"✓ Carpeta de CÓDIGOS encontrada: {ruta_codigos_encontrada.name}")

# 5. Reporte de errores si algo falta
if not ruta_2012_encontrada:
    print(f"❌ No se encontró la carpeta de datos en: {posible_datos}")
    print("ℹ️ Se usará interpolación lineal para los años 2005–2016 (limitación metodológica).")

if not ruta_codigos_encontrada:
    print(f"⚠ Advertencia: No se encontró la carpeta de códigos en: {posible_codigos}")

✓ Carpeta de DATOS encontrada: bases_datos_engho2012
✓ Carpeta de CÓDIGOS encontrada: codigos_engho2012


In [14]:
# Procesamiento ENGHo 2012
if ruta_2012_encontrada:
    print(f"🔄 Procesando archivos en: {ruta_2012_encontrada.name}")
    
    # 1. Definir rutas exactas a los archivos (usando los nombres que me pasaste)
    ruta_gastos_2012 = ruta_2012_encontrada / "ENGHo - Gastos.txt"
    ruta_hogar_2012 = ruta_2012_encontrada / "ENGHo - Hogares.txt"
    
    # 2. Leer Gastos
    if ruta_gastos_2012.exists():
        df_g2012 = leer_txt_indec(ruta_gastos_2012)
        print("✓ Base de GASTOS cargada.")
        print("Columnas detectadas:", list(df_g2012.columns[:10]))
        
        # --- AJUSTE DE COLUMNAS ---
        # INDEC suele cambiar nombres. Si al ejecutar ves que dan error, 
        # cámbialos aquí por los que imprimió arriba.
        COL_ARTICULO_2012 = 'articulo' # O 'cod_pba' según la versión
        COL_MONTO_2012 = 'monto'
        COL_CLAVE_2012 = 'id'      # Clave de unión con Hogares
    else:
        print("❌ No se encontró el archivo: ENGHo - Gastos.txt")

    # 3. Leer Hogares
    if ruta_hogar_2012.exists():
        df_h2012 = leer_txt_indec(ruta_hogar_2012)
        print("✓ Base de HOGARES cargada.")
        print("Columnas detectadas:", list(df_h2012.columns[:10]))
    else:
        print("⚠️ No se encontró el archivo: ENGHo - Hogares.txt")

else:
    print("ℹ️ ENGHo 2012 no disponible — usando interpolación.")

🔄 Procesando archivos en: bases_datos_engho2012
✓ Base de GASTOS cargada.
Columnas detectadas: ['CLAVE', 'ARTICULO', 'REGION', 'SUBREGION', 'PROVINCIA', 'EXPAN', 'SUBCLASE', 'CLASE', 'GRUPO', 'DIVISION']
✓ Base de HOGARES cargada.
Columnas detectadas: ['REGION', 'SUBREGION', 'PROVINCIA', 'CLAVE', 'EXPAN', 'AÃ\x91O', 'TRIMESTRE', 'CV1A01', 'CV1A02', 'CV1B01_A']


In [15]:
import numpy as np
import pandas as pd

# 1. Mapeo COICOP directo usando la columna DIVISION de 2012
MAPEO_2012 = {
    1:  "alimentos_y_bebidas",
    2:  "bebidas_alcoholicas_y_tabaco",
    3:  "indumentaria_y_calzado",
    4:  "propiedades_combustibles_agua_electricidad",
    5:  "equipamiento_y_mantenimiento",
    6:  "salud",
    7:  "transporte",
    8:  "comunicaciones",
    9:  "esparcimiento_y_cultura",
    10: "ensenanza",
    11: "restaurantes_y_hoteles",
    12: "bienes_y_servicios_varios",
}

print("🔄 Procesando estructura de consumo ENGHo 2012...")

COL_MONTO_2012 = next((c for c in df_g2012.columns if 'MONTO' in c), 'MONTO')
COL_DECIL_2012 = 'DECIPTHT' # Decil de Ingreso Per Cápita Total

print(f"✓ Usando Monto: '{COL_MONTO_2012}' | Decil: '{COL_DECIL_2012}'")
print(f"📊 Filas originales -> Gastos: {len(df_g2012):,}, Hogares: {len(df_h2012):,}")

# ---------------------------------------------------------
# FIX CRÍTICO 1: Limpiar CLAVES (sacar espacios invisibles)
# ---------------------------------------------------------
df_g2012['CLAVE'] = df_g2012['CLAVE'].astype(str).str.strip()
df_h2012['CLAVE'] = df_h2012['CLAVE'].astype(str).str.strip()

# ---------------------------------------------------------
# FIX CRÍTICO 2: Mapeo numérico blindado
# ---------------------------------------------------------

# Preparar Monto (FIX: reemplazar comas por puntos para no perder los decimales)
monto_limpio = df_g2012[COL_MONTO_2012].astype(str).str.replace(',', '.')
df_g2012['MONTO_NUM'] = pd.to_numeric(monto_limpio, errors='coerce').fillna(0)

# Forzamos la división a número, quitamos nulos, forzamos a entero, y mapeamos
div_numerica = pd.to_numeric(df_g2012['DIVISION'], errors='coerce')
df_g2012['CATEGORIA'] = div_numerica.dropna().astype(int).map(MAPEO_2012)
df_g2012['CATEGORIA'] = df_g2012['CATEGORIA'].fillna('otros')

n_otros = (df_g2012['CATEGORIA'] == 'otros').sum()
print(f"🔎 Artículos mapeados a 'otros': {n_otros:,} de {len(df_g2012):,}")

# 3. Preparar Hogares (Convertir Decil a Quintil)
df_h2012_filtro = df_h2012[['CLAVE', 'EXPAN', COL_DECIL_2012]].copy()
df_h2012_filtro = df_h2012_filtro.dropna(subset=[COL_DECIL_2012])
df_h2012_filtro[COL_DECIL_2012] = pd.to_numeric(df_h2012_filtro[COL_DECIL_2012], errors='coerce')
df_h2012_filtro = df_h2012_filtro[df_h2012_filtro[COL_DECIL_2012] > 0]

# MATEMÁTICA: Quintil = redondear_hacia_arriba(Decil / 2)
df_h2012_filtro['QUINTIL'] = ((df_h2012_filtro[COL_DECIL_2012] - 1) // 2 + 1).astype(int)

# 4. Merge Gastos + Hogares
df_merge_2012 = pd.merge(
    df_g2012[['CLAVE', 'MONTO_NUM', 'CATEGORIA']],
    df_h2012_filtro[['CLAVE', 'EXPAN', 'QUINTIL']],
    on='CLAVE', how='inner'
)
print(f"🤝 Filas tras hacer el merge (unión): {len(df_merge_2012):,}")

# 5. Expandir gasto y filtrar
df_merge_2012['GASTO_EXPANDIDO'] = df_merge_2012['MONTO_NUM'] * df_merge_2012['EXPAN']
df_merge_clasif = df_merge_2012[df_merge_2012['CATEGORIA'] != 'otros']
print(f"🧹 Filas útiles tras quitar 'otros': {len(df_merge_clasif):,}")

# Si llegó hasta acá con datos, la tabla se genera bien
if len(df_merge_clasif) > 0:
    # 6. Tabla Pivote
    tabla_pivote = (
        df_merge_clasif
        .groupby(['CATEGORIA', 'QUINTIL'])['GASTO_EXPANDIDO']
        .sum()
        .unstack()
    )
    
    tabla_pivote.columns = [f"Q{int(col)}" for col in tabla_pivote.columns]
    tabla_porc = tabla_pivote.div(tabla_pivote.sum(axis=0), axis=1)
    
    df_final_2012 = tabla_porc.reset_index().rename(columns={'CATEGORIA': 'categoria_coicop'})
    df_final_2012['engho_edicion'] = '2012_13'
    
    # Guardar a CSV
    archivo_csv = ruta_salida / "engho_estructura_2012.csv"
    df_final_2012.to_csv(archivo_csv, index=False)
    
    print(f"\n✅ ENGHo 2012 guardada con éxito en: {archivo_csv.name}")
    print("\nEstructura por quintil (proporciones):")
    print(df_final_2012.drop(columns='engho_edicion').to_string(index=False))
else:
    print("\n❌ ERROR: No quedaron datos para procesar. Revisa los prints de arriba.")

🔄 Procesando estructura de consumo ENGHo 2012...
✓ Usando Monto: 'MONTO' | Decil: 'DECIPTHT'
📊 Filas originales -> Gastos: 1,116,328, Hogares: 20,960
🔎 Artículos mapeados a 'otros': 1,116,328 de 1,116,328
🤝 Filas tras hacer el merge (unión): 1,116,328
🧹 Filas útiles tras quitar 'otros': 0

❌ ERROR: No quedaron datos para procesar. Revisa los prints de arriba.


In [16]:
import numpy as np
import pandas as pd

# 1. Mapeo COICOP directo usando los 2 primeros dígitos
MAPEO_2012 = {
    1:  "alimentos_y_bebidas",
    2:  "bebidas_alcoholicas_y_tabaco",
    3:  "indumentaria_y_calzado",
    4:  "propiedades_combustibles_agua_electricidad",
    5:  "equipamiento_y_mantenimiento",
    6:  "salud",
    7:  "transporte",
    8:  "comunicaciones",
    9:  "esparcimiento_y_cultura",
    10: "ensenanza",
    11: "restaurantes_y_hoteles",
    12: "bienes_y_servicios_varios",
}

print("🔄 Procesando estructura de consumo ENGHo 2012...")

COL_MONTO_2012 = next((c for c in df_g2012.columns if 'MONTO' in c), 'MONTO')
COL_DECIL_2012 = 'DECIPTHT'

print(f"✓ Usando Monto: '{COL_MONTO_2012}' | Decil: '{COL_DECIL_2012}'")

# Limpiar CLAVES
df_g2012['CLAVE'] = df_g2012['CLAVE'].astype(str).str.strip()
df_h2012['CLAVE'] = df_h2012['CLAVE'].astype(str).str.strip()

# Preparar Monto (FIX: reemplazar comas por puntos para no perder los decimales)
monto_limpio = df_g2012[COL_MONTO_2012].astype(str).str.replace(',', '.')
df_g2012['MONTO_NUM'] = pd.to_numeric(monto_limpio, errors='coerce').fillna(0)

# ---------------------------------------------------------
# FIX CRÍTICO: Mapeo a prueba de balas extrayendo del ARTICULO
# ---------------------------------------------------------
def mapear_2012_seguro(codigo):
    # .split('.')[0] evita problemas si Pandas lo leyó como float (ej. "111101.0")
    cod = str(codigo).split('.')[0].strip()
    if cod == 'nan' or cod == '': return 'otros'
    
    # Rellenamos con ceros a la izquierda hasta tener 7 dígitos (ej: "111101" -> "0111101")
    cod_7 = cod.zfill(7)
    try:
        # Los primeros 2 dígitos son la categoría
        div = int(cod_7[:2])
        return MAPEO_2012.get(div, 'otros')
    except:
        return 'otros'

df_g2012['CATEGORIA'] = df_g2012['ARTICULO'].apply(mapear_2012_seguro)

n_otros = (df_g2012['CATEGORIA'] == 'otros').sum()
print(f"🔎 Artículos mapeados a 'otros': {n_otros:,} de {len(df_g2012):,}")

# Preparar Hogares (Convertir Decil a Quintil)
df_h2012_filtro = df_h2012[['CLAVE', 'EXPAN', COL_DECIL_2012]].copy()
df_h2012_filtro = df_h2012_filtro.dropna(subset=[COL_DECIL_2012])
df_h2012_filtro[COL_DECIL_2012] = pd.to_numeric(df_h2012_filtro[COL_DECIL_2012], errors='coerce')
df_h2012_filtro = df_h2012_filtro[df_h2012_filtro[COL_DECIL_2012] > 0]

df_h2012_filtro['QUINTIL'] = np.ceil(df_h2012_filtro[COL_DECIL_2012] / 2).astype(int)

# Merge Gastos + Hogares
df_merge_2012 = pd.merge(
    df_g2012[['CLAVE', 'MONTO_NUM', 'CATEGORIA']],
    df_h2012_filtro[['CLAVE', 'EXPAN', 'QUINTIL']],
    on='CLAVE', how='inner'
)

# Expandir gasto y filtrar
df_merge_2012['GASTO_EXPANDIDO'] = df_merge_2012['MONTO_NUM'] * df_merge_2012['EXPAN']
df_merge_clasif = df_merge_2012[df_merge_2012['CATEGORIA'] != 'otros']

if len(df_merge_clasif) > 0:
    # Tabla Pivote
    tabla_pivote = (
        df_merge_clasif
        .groupby(['CATEGORIA', 'QUINTIL'])['GASTO_EXPANDIDO']
        .sum()
        .unstack()
    )
    
    tabla_pivote.columns = [f"Q{int(col)}" for col in tabla_pivote.columns]
    tabla_porc = tabla_pivote.div(tabla_pivote.sum(axis=0), axis=1)
    
    df_final_2012 = tabla_porc.reset_index().rename(columns={'CATEGORIA': 'categoria_coicop'})
    df_final_2012['engho_edicion'] = '2012_13'
    
    # Guardar a CSV
    archivo_csv = ruta_salida / "engho_estructura_2012.csv"
    df_final_2012.to_csv(archivo_csv, index=False)
    
    print(f"\n✅ ENGHo 2012 guardada con éxito en: {archivo_csv.name}")
    print("\nEstructura por quintil (proporciones):")
    print(df_final_2012.drop(columns='engho_edicion').to_string(index=False))
else:
    print("\n❌ ERROR: Siguen sin quedar datos. Verifica la columna ARTICULO.")

🔄 Procesando estructura de consumo ENGHo 2012...
✓ Usando Monto: 'MONTO' | Decil: 'DECIPTHT'
🔎 Artículos mapeados a 'otros': 0 de 1,116,328

✅ ENGHo 2012 guardada con éxito en: engho_estructura_2012.csv

Estructura por quintil (proporciones):
                          categoria_coicop       Q1       Q2       Q3       Q4       Q5
                       alimentos_y_bebidas 0.415147 0.395404 0.364317 0.324329 0.288450
              bebidas_alcoholicas_y_tabaco 0.083346 0.083257 0.090073 0.087622 0.087461
                            comunicaciones 0.019712 0.015031 0.025461 0.030267 0.034649
              equipamiento_y_mantenimiento 0.065301 0.056849 0.057240 0.047910 0.059313
                   esparcimiento_y_cultura 0.059733 0.061558 0.051615 0.055587 0.046052
                    indumentaria_y_calzado 0.105420 0.102888 0.107322 0.110154 0.093341
propiedades_combustibles_agua_electricidad 0.068589 0.069373 0.068996 0.071878 0.076839
                                     salud 0.121908 0

---
## SECCIÓN 6 — ENGHo 2017/18: estructura de gasto (ya procesada)

La extracción del Cuadro 2 del XLS de ingresos es correcta: ese cuadro contiene la estructura de gasto por categoría COICOP organizada por quintil de IPCF, que es exactamente lo que necesitamos. Re-ejecutamos con una verificación adicional de consistencia.

In [17]:
import pandas as pd

archivo_ingresos = RUTAS['engho'] / "engho_2017_ingresos.xls"
ruta_salida = RUTAS['engho_struct']

print("🔄 Extrayendo ENGHo 2017/18 — Cuadro 2 (gastos por categoría COICOP por quintil)...")

df_raw = pd.read_excel(archivo_ingresos, sheet_name="Cuadro 2", header=None)

idx_inicio = df_raw[
    df_raw[0].astype(str).str.contains("Alimentos y bebidas no alcohólicas", na=False)
].index[0]

df_gastos = df_raw.iloc[idx_inicio : idx_inicio + 12].copy()
df_gastos = df_gastos[[0, 5, 9, 13, 17, 21]]
df_gastos.columns = ["categoria_indec", "Q1", "Q2", "Q3", "Q4", "Q5"]

MAPEO_2017 = {
    "Alimentos y bebidas no alcohólicas": "alimentos_y_bebidas",
    "Bebidas alcohólicas y tabaco":        "bebidas_alcoholicas_y_tabaco",
    "Prendas de vestir y calzado":         "indumentaria_y_calzado",
    "Vivienda, agua, electricidad, gas y otros combustibles": "propiedades_combustibles_agua_electricidad",
    "Equipamiento y mantenimiento del hogar": "equipamiento_y_mantenimiento",
    "Salud":                               "salud",
    "Transporte":                          "transporte",
    "Comunicaciones":                      "comunicaciones",
    "Recreación y cultura":                "esparcimiento_y_cultura",
    "Educación":                           "ensenanza",
    "Restaurantes y hoteles":              "restaurantes_y_hoteles",
    "Bienes y servicios varios":           "bienes_y_servicios_varios",
}

df_gastos["categoria_indec"] = df_gastos["categoria_indec"].str.strip()
df_gastos["categoria_coicop"] = df_gastos["categoria_indec"].map(MAPEO_2017)

# Verificar que todas las categorías mapearon
no_mapeadas = df_gastos[df_gastos["categoria_coicop"].isna()]["categoria_indec"]
if len(no_mapeadas) > 0:
    print(f"⚠️  Categorías sin mapeo: {no_mapeadas.tolist()}")

df_final = df_gastos[["categoria_coicop", "Q1", "Q2", "Q3", "Q4", "Q5"]].copy()

for col in ["Q1", "Q2", "Q3", "Q4", "Q5"]:
    if df_final[col].dtype == object:
        df_final[col] = df_final[col].astype(str).str.replace(',', '.')
    df_final[col] = pd.to_numeric(df_final[col], errors='coerce') / 100.0

# Verificar que cada quintil suma 1 (dentro de tolerancia por redondeo del Excel)
sumas = df_final[["Q1", "Q2", "Q3", "Q4", "Q5"]].sum()
for q, s in sumas.items():
    estado = "✓" if abs(s - 1.0) < 0.02 else "⚠️ REVISAR"
    print(f"  Suma {q}: {s:.4f} {estado}")

df_final['engho_edicion'] = '2017_18'
archivo_csv = ruta_salida / "engho_estructura_2017.csv"
df_final.to_csv(archivo_csv, index=False)

print(f"\n✅ ENGHo 2017 guardada: {archivo_csv.name}")
print("\nEstructura por quintil (proporciones):")
print(df_final.drop(columns='engho_edicion').to_string(index=False))

🔄 Extrayendo ENGHo 2017/18 — Cuadro 2 (gastos por categoría COICOP por quintil)...
  Suma Q1: 1.0000 ✓
  Suma Q2: 1.0000 ✓
  Suma Q3: 1.0000 ✓
  Suma Q4: 1.0000 ✓
  Suma Q5: 1.0000 ✓

✅ ENGHo 2017 guardada: engho_estructura_2017.csv

Estructura por quintil (proporciones):
                          categoria_coicop       Q1       Q2       Q3       Q4       Q5
                       alimentos_y_bebidas 0.344750 0.276930 0.253750 0.210838 0.156985
              bebidas_alcoholicas_y_tabaco 0.022236 0.023698 0.019314 0.017476 0.018404
                    indumentaria_y_calzado 0.079923 0.073513 0.071942 0.067765 0.059696
propiedades_combustibles_agua_electricidad 0.141912 0.147703 0.146197 0.140448 0.146196
              equipamiento_y_mantenimiento 0.041193 0.045681 0.046910 0.053619 0.069954
                                     salud 0.041856 0.047508 0.060962 0.068150 0.079965
                                transporte 0.103889 0.132791 0.131592 0.159955 0.156447
                       

---
## SECCIÓN 7 — Interpolación de estructura de consumo para años sin ENGHo

Construimos una tabla de estructura de consumo para cada año de la serie (2004–2024) interpolando entre las ediciones disponibles de la ENGHo.

**Regla de vigencia:**
- 2004–2011: usa ENGHo 2004/05 directamente
- 2012–2016: interpolación lineal entre ENGHo 2004 y ENGHo 2017 (si no hay microdatos 2012)
- 2017–2024: usa ENGHo 2017/18 directamente

**IMPORTANTE:** La interpolación se hace por quintil y por categoría de gasto. No por año ni por decil.

In [18]:
import pandas as pd
import numpy as np

# Verificar si tenemos ENGHo 2012
ruta_2012 = RUTAS['engho_struct'] / "engho_estructura_2012.csv"
hay_2012 = ruta_2012.exists()

# Cargar las estructuras base
df_2004 = pd.read_csv(RUTAS['engho_struct'] / "engho_estructura_2004.csv")
df_2017 = pd.read_csv(RUTAS['engho_struct'] / "engho_estructura_2017.csv")

COLS_Q = ["Q1", "Q2", "Q3", "Q4", "Q5"]
df_2004 = df_2004[["categoria_coicop"] + COLS_Q].set_index("categoria_coicop")
df_2017 = df_2017[["categoria_coicop"] + COLS_Q].set_index("categoria_coicop")

if hay_2012:
    df_2012 = pd.read_csv(ruta_2012)
    df_2012 = df_2012[["categoria_coicop"] + COLS_Q].set_index("categoria_coicop")
    cats_2012 = set(df_2012.index)
else:
    df_2012 = None
    cats_2012 = set(df_2017.index) # Dummy set para que la intersección no cambie

# Verificar que las categorías coinciden
cats_2004 = set(df_2004.index)
cats_2017 = set(df_2017.index)

# Usar solo categorías comunes a los años disponibles
cats_comunes = list(cats_2004 & cats_2017 & cats_2012)

df_2004 = df_2004.loc[cats_comunes]
df_2017 = df_2017.loc[cats_comunes]
if hay_2012:
    df_2012 = df_2012.loc[cats_comunes]
    
print(f"✓ Usando {len(cats_comunes)} categorías comunes.")
if hay_2012:
    print("✓ ENGHo 2012 detectada. Se usarán 2 tramos de interpolación (2005-2012 y 2012-2017).")
else:
    print("⚠️ ENGHo 2012 NO detectada. Se usará 1 tramo de interpolación largo (2005-2017).")

def obtener_estructura_anio(anio):
    """Devuelve la estructura de gasto para un año dado."""
    if anio <= 2011: # Años previos a 2012 usan 2004 como base (ancla 2005)
        if not hay_2012 and anio > 2005:
            # Interpolación si no hay 2012
            anio_0, anio_1 = 2005, 2017
            df_0, df_1 = df_2004, df_2017
        elif hay_2012 and anio > 2005:
            anio_0, anio_1 = 2005, 2012
            df_0, df_1 = df_2004, df_2012
        else:
            return df_2004.copy() # 2004 o 2005 directo
            
    elif anio >= 2017:
        return df_2017.copy()
        
    else: # Años 2012 a 2016
        if hay_2012:
            if anio == 2012:
                return df_2012.copy()
            else:
                anio_0, anio_1 = 2012, 2017
                df_0, df_1 = df_2012, df_2017
        else:
            anio_0, anio_1 = 2005, 2017
            df_0, df_1 = df_2004, df_2017
            
    # Matemáticas de interpolación
    alpha = (anio - anio_0) / (anio_1 - anio_0)
    df_interp = df_0 * (1 - alpha) + df_1 * alpha
    
    # Re-normalizar para que cada quintil sume exactamente 1
    df_interp = df_interp.div(df_interp.sum(axis=0), axis=1)
    return df_interp

# Generar y guardar una tabla por año
all_structures = []
for anio in range(2004, 2025):
    df_anio = obtener_estructura_anio(anio).reset_index()
    df_anio['anio'] = anio
    all_structures.append(df_anio)

df_todas = pd.concat(all_structures, ignore_index=True)
df_todas.to_csv(RUTAS['engho_struct'] / "engho_estructura_serie_2004_2024.csv", index=False)

print("\n✅ Serie de estructuras de consumo generada: engho_estructura_serie_2004_2024.csv")
print(f"   Años: {df_todas['anio'].min()}–{df_todas['anio'].max()}")
print(f"   Categorías: {df_todas['categoria_coicop'].nunique()}")
print(f"   Filas totales: {len(df_todas):,}")

✓ Usando 8 categorías comunes.
✓ ENGHo 2012 detectada. Se usarán 2 tramos de interpolación (2005-2012 y 2012-2017).

✅ Serie de estructuras de consumo generada: engho_estructura_serie_2004_2024.csv
   Años: 2004–2024
   Categorías: 8
   Filas totales: 168


In [19]:
# 1. Ver todas las columnas reales de la base de Gastos
print("Todas las columnas en df_g2012:")
print(list(df_g2012.columns))

# 2. Buscar columnas que suenen a dinero, gasto o multiplicadores de tiempo
cols_dinero = [c for c in df_g2012.columns if any(palabra in c for palabra in ['MONTO', 'GASTO', 'MES', 'ANUAL', 'COEF', 'VALOR'])]

print(f"\nColumnas candidatas para el dinero: {cols_dinero}")

# 3. Ver un ejemplo real de un ticket de Alimentos vs uno de Salud
print("\n--- EJEMPLO TICKET ALIMENTOS ---")
print(df_g2012[df_g2012['CATEGORIA'] == 'alimentos_y_bebidas'][['ARTICULO'] + cols_dinero].head(3))

print("\n--- EJEMPLO TICKET SALUD ---")
print(df_g2012[df_g2012['CATEGORIA'] == 'salud'][['ARTICULO'] + cols_dinero].head(3))

Todas las columnas en df_g2012:
['CLAVE', 'ARTICULO', 'REGION', 'SUBREGION', 'PROVINCIA', 'EXPAN', 'SUBCLASE', 'CLASE', 'GRUPO', 'DIVISION', 'CANTIDAD', 'UN_MED', 'MONTO', 'MIEMBRO', 'MONTOIMPUTADO', 'CANTIDADIMPUTADA', 'MONTO_NUM', 'CATEGORIA']

Columnas candidatas para el dinero: ['MONTO', 'MONTOIMPUTADO', 'MONTO_NUM']

--- EJEMPLO TICKET ALIMENTOS ---
   ARTICULO MONTO  MONTOIMPUTADO  MONTO_NUM
2    117129  30,1            NaN      30.10
3    117119  12,9            NaN      12.90
4    117128  6,45            NaN       6.45

--- EJEMPLO TICKET SALUD ---
    ARTICULO MONTO  MONTOIMPUTADO  MONTO_NUM
36    611308   3,5            NaN        3.5
81    623302   160            NaN      160.0
88    611701   110            NaN      110.0


In [20]:
print("Muestra de categorías en 2004:", df_2004.index[:10].tolist())
print("Muestra de categorías en 2012:", df_2012.index[:10].tolist())
print("Muestra de categorías en 2017:", df_2017.index[:10].tolist())

Muestra de categorías en 2004: ['propiedades_combustibles_agua_electricidad', 'indumentaria_y_calzado', 'alimentos_y_bebidas', 'salud', 'transporte', 'comunicaciones', 'esparcimiento_y_cultura', 'equipamiento_y_mantenimiento']
Muestra de categorías en 2012: ['propiedades_combustibles_agua_electricidad', 'indumentaria_y_calzado', 'alimentos_y_bebidas', 'salud', 'transporte', 'comunicaciones', 'esparcimiento_y_cultura', 'equipamiento_y_mantenimiento']
Muestra de categorías en 2017: ['propiedades_combustibles_agua_electricidad', 'indumentaria_y_calzado', 'alimentos_y_bebidas', 'salud', 'transporte', 'comunicaciones', 'esparcimiento_y_cultura', 'equipamiento_y_mantenimiento']


---
## SECCIÓN 8 — Tabla de parámetros de Ganancias (2004–2024)

Esta tabla es el insumo más crítico del modelo: define quiénes pagan Ganancias y cuánto. Se construye manualmente a partir del PDF histórico de ARCA (`ganancias_2000_2025.pdf`).

**Qué contiene esta tabla:**
- `GNI_anual`: Ganancia No Imponible anual (en $ nominales del año)
- `ded_especial_4ta`: Deducción especial para 4ta categoría (relación de dependencia)
- `tramos`: lista de (límite_superior, alícuota_marginal) donde el último tramo tiene límite inf

**Nota sobre la deducción especial:** A partir de 2021 (Ley 27617) los empleados en relación de dependencia con ingresos hasta cierto umbral quedaron exentos directamente. Esta tabla registra la estructura nominal; la lógica de aplicación se implementa en la Fase 3.

**Fuente:** https://www.argentina.gob.ar/sites/default/files/impuesto_a_las_ganancias_desde_el_ano_2000_al_2025_0.pdf

In [ ]:
import pandas as pd
import json

## Verificado Manualmente

# =============================================================================
# TABLA MAESTRA DE PARÁMETROS DE GANANCIAS (4TA CATEGORÍA)
# Todos los valores en PESOS NOMINALES del año correspondiente
# Fuente: ARCA/AFIP — "Evolución de tasas, tramos de escalas y deducciones personales"
# URL: https://www.argentina.gob.ar/sites/default/files/
#       impuesto_a_las_ganancias_desde_el_ano_2000_al_2025_0.pdf
# =============================================================================
# CÓMO LEER LOS TRAMOS:
# Cada entrada es [límite_superior_del_tramo, alícuota_marginal]
# El último tramo tiene límite float('inf') → aplica a todo lo que supere el anterior
# La base de cada tramo es el ingreso NETO (después de GNI y deducción especial)
# =============================================================================

GANANCIAS_PARAMS = {

    # ==================================================================
    # 2004
    # GNI: $4.020 | Ded. Esp. 4ta: $13.500
    # Norma: Ley 25.239 (escalas) | Decretos 860/2001 y 1676/2001 (ded.)
    # NOTA: Los tramos nominales de la escala NO CAMBIAN de 2000 a 2015.
    # ==================================================================
    2004: {
        'GNI_anual': 4020,
        'ded_especial_4ta': 13500,
        'tramos': [
            (10000,          0.09),
            (20000,          0.14),
            (30000,          0.19),
            (60000,          0.23),
            (90000,          0.27),
            (120000,         0.31),
            (float('inf'),   0.35),
        ]
    },

    # ==================================================================
    # 2005
    # GNI: $4.020 | Ded. Esp. 4ta: $13.500
    # Norma: idéntica a 2004
    # ==================================================================
    2005: {
        'GNI_anual': 4020,
        'ded_especial_4ta': 13500,
        'tramos': [
            (10000,          0.09),
            (20000,          0.14),
            (30000,          0.19),
            (60000,          0.23),
            (90000,          0.27),
            (120000,         0.31),
            (float('inf'),   0.35),
        ]
    },

    # ==================================================================
    # 2006
    # GNI: $6.000 | Ded. Esp. 4ta: $22.800
    # Norma: Decreto 314/2006
    # ==================================================================
    2006: {
        'GNI_anual': 6000,
        'ded_especial_4ta': 22800,
        'tramos': [
            (10000,          0.09),
            (20000,          0.14),
            (30000,          0.19),
            (60000,          0.23),
            (90000,          0.27),
            (120000,         0.31),
            (float('inf'),   0.35),
        ]
    },

    # ==================================================================
    # 2007
    # GNI: $7.500 | Ded. Esp. 4ta: $36.000
    # Norma: Ley 26.287
    # ==================================================================
    2007: {
        'GNI_anual': 7500,
        'ded_especial_4ta': 36000,
        'tramos': [
            (10000,          0.09),
            (20000,          0.14),
            (30000,          0.19),
            (60000,          0.23),
            (90000,          0.27),
            (120000,         0.31),
            (float('inf'),   0.35),
        ]
    },

    # ==================================================================
    # 2008
    # GNI: $9.000 | Ded. Esp. 4ta: $43.200
    # Norma: Decreto 1426/2008
    # ==================================================================
    2008: {
        'GNI_anual': 9000,
        'ded_especial_4ta': 43200,
        'tramos': [
            (10000,          0.09),
            (20000,          0.14),
            (30000,          0.19),
            (60000,          0.23),
            (90000,          0.27),
            (120000,         0.31),
            (float('inf'),   0.35),
        ]
    },

    # ==================================================================
    # 2009
    # GNI: $9.000 | Ded. Esp. 4ta: $43.200
    # Norma: Decreto 1426/2008 (mismos valores que 2008)
    # IMPORTANTE: Ley 26.477 elimina la tabla de reducción de deducciones
    # en función de la ganancia neta a partir del 01/01/2009.
    # ==================================================================
    2009: {
        'GNI_anual': 9000,
        'ded_especial_4ta': 43200,
        'tramos': [
            (10000,          0.09),
            (20000,          0.14),
            (30000,          0.19),
            (60000,          0.23),
            (90000,          0.27),
            (120000,         0.31),
            (float('inf'),   0.35),
        ]
    },

    # ==================================================================
    # 2010
    # GNI: $10.800 | Ded. Esp. 4ta: $51.840
    # Norma: Ley 26.731 y R.G. 2866 AFIP
    # ==================================================================
    2010: {
        'GNI_anual': 10800,
        'ded_especial_4ta': 51840,
        'tramos': [
            (10000,          0.09),
            (20000,          0.14),
            (30000,          0.19),
            (60000,          0.23),
            (90000,          0.27),
            (120000,         0.31),
            (float('inf'),   0.35),
        ]
    },

    # ==================================================================
    # 2011
    # GNI: $12.960 | Ded. Esp. 4ta: $62.208
    # Norma: Ley 26.731 y R.G. 3073 AFIP
    # ==================================================================
    2011: {
        'GNI_anual': 12960,
        'ded_especial_4ta': 62208,
        'tramos': [
            (10000,          0.09),
            (20000,          0.14),
            (30000,          0.19),
            (60000,          0.23),
            (90000,          0.27),
            (120000,         0.31),
            (float('inf'),   0.35),
        ]
    },

    # ==================================================================
    # 2012
    # GNI: $12.960 | Ded. Esp. 4ta: $62.208
    # Norma: Ley 26.731 y R.G. 3073 AFIP (mismos valores que 2011)
    # NOTA METODOLÓGICA: El Decreto 1242/2012 estableció en septiembre
    # una exención de hecho para quienes ganaban hasta $25.000 brutos
    # mensuales. Esta exención operó como "deducción especial ampliada"
    # sin cambiar los parámetros formales del cuadro. Para la modelización
    # anual se usa el valor formal de la ded. especial. Si querés capturar
    # el efecto del decreto, aplicar umbral_exencion_mensual=25000 solo
    # para los trimestres T3-T4 de 2012.
    # ==================================================================
    2012: {
        'GNI_anual': 12960,
        'ded_especial_4ta': 62208,
        # Umbral de exención efectiva a partir de sept. 2012
        # (trabajadores en relación de dependencia)
        'umbral_exencion_mensual_desde_sept': 25000,
        'tramos': [
            (10000,          0.09),
            (20000,          0.14),
            (30000,          0.19),
            (60000,          0.23),
            (90000,          0.27),
            (120000,         0.31),
            (float('inf'),   0.35),
        ]
    },

    # ==================================================================
    # 2013
    # GNI: $15.552 | Ded. Esp. 4ta: variable según tramo de ingreso
    # Norma: Ley 26.731, Decretos 244/2013, 1006/2013, 1242/2013
    #        y R.G. 3525 AFIP
    #
    # ESTRUCTURA COMPLEJA — tres sub-períodos en el año:
    #
    # Hasta 28/02/2013:
    #   GNI $12.960, Ded. Esp. 4ta $62.208 (igual a 2012)
    #
    # 01/03/2013 – 31/08/2013:
    #   GNI $15.552, Ded. Esp. 4ta $74.650 (rango >$25.000 brutos)
    #
    # A partir del 01/09/2013 — Decreto 1242/2013:
    #   Para quienes ganaban ≤$15.000 brutos mensuales (ene-ago 2013):
    #     deducción especial ampliada hasta que GNSI = 0 (exentos)
    #   Para quienes ganaban $15.001–$24.999:
    #     GNI $15.552, Ded. Esp. variable según tabla (ver PDF pág. 17)
    #   Para quienes ganaban >$25.000:
    #     GNI $15.552, Ded. Esp. $97.044
    #
    # Para la modelización ANUAL se usa el tramo más representativo
    # (ingresos >$25.000) como proxy del año completo.
    # Los trabajadores de menores ingresos afectados por el decreto
    # quedan en gran medida capturados por la EPH como informales
    # (tasa = 0 en Ganancias) o con IPCF bajo el mínimo.
    # ==================================================================
    2013: {
        'GNI_anual': 15552,
        'ded_especial_4ta': 97044,        # Corresponde a rango >$25.000 brutos
        'umbral_exencion_mensual': 15000, # Renta bruta mensual ene-ago 2013
        'tramos': [
            (10000,          0.09),
            (20000,          0.14),
            (30000,          0.19),
            (60000,          0.23),
            (90000,          0.27),
            (120000,         0.31),
            (float('inf'),   0.35),
        ]
    },

    # ==================================================================
    # 2014
    # GNI: $15.552 | Ded. Esp. 4ta: variable según tramo
    # Norma: Decretos 244/2013 y 1242/2013, R.G. 3525 AFIP
    #
    # Mantiene la estructura del Decreto 1242/2013.
    # Para el rango >$25.000 brutos mensuales:
    #   GNI $15.552, Ded. Esp. $97.044
    # Para el rango $15.001–$24.999:
    #   GNI $15.552, Ded. Esp. $89.580
    # Umbral de exención para ≤$15.000 brutos: exentos
    # ==================================================================
    2014: {
        'GNI_anual': 15552,
        'ded_especial_4ta': 97044,
        'umbral_exencion_mensual': 15000,
        'tramos': [
            (10000,          0.09),
            (20000,          0.14),
            (30000,          0.19),
            (60000,          0.23),
            (90000,          0.27),
            (120000,         0.31),
            (float('inf'),   0.35),
        ]
    },

    # ==================================================================
    # 2015
    # GNI: $15.552
    # Norma: Decreto 244/2013
    # NOTA: El PDF no muestra la deducción especial 4ta para los
    # trabajadores en relación de dependencia en 2015 — solo muestra
    # la del autónomo ($15.552). Esto es porque la estructura del
    # Decreto 1242/2013 seguía vigente pero la ded. esp. para rel.
    # de dependencia dependía del tramo de ingreso de ene-ago 2013,
    # no de un valor fijo. En la práctica el umbral de exención siguió
    # operando. Se usa $97.044 como proxy (mismo que 2014).
    # R.G. 3770/2015 AFIP ajustó umbrales durante el año.
    # ==================================================================
    2015: {
        'GNI_anual': 15552,
        'ded_especial_4ta': 97044,        # Proxy — ver nota
        'umbral_exencion_mensual': 15000,
        'tramos': [
            (10000,          0.09),
            (20000,          0.14),
            (30000,          0.19),
            (60000,          0.23),
            (90000,          0.27),
            (120000,         0.31),
            (float('inf'),   0.35),
        ]
    },

    # ==================================================================
    # 2016
    # GNI: $42.318 | Ded. Esp. 4ta: $203.126
    # Norma: Decreto 394/2016
    # IMPORTANTE: Los tramos de la escala NO cambiaron en 2016.
    # Siguen siendo los mismos nominales de 2000. La "tablita" de
    # Machinea llevó a una enorme cantidad de trabajadores al tramo
    # máximo del 35% en términos reales.
    # ==================================================================
    2016: {
        'GNI_anual': 42318,
        'ded_especial_4ta': 203126,
        'tramos': [
            (10000,          0.09),
            (20000,          0.14),
            (30000,          0.19),
            (60000,          0.23),
            (90000,          0.27),
            (120000,         0.31),
            (float('inf'),   0.35),
        ]
    },

    # ==================================================================
    # 2017
    # GNI: $51.967 | Ded. Esp. 4ta: $249.442
    # Norma: Ley 27.346
    # REFORMA: Primera actualización real de los tramos de escala
    # desde el año 2000. Se reduce la alícuota mínima al 5% y se
    # amplían los tramos. Los tramos ahora van de $20.000 en $20.000.
    # ==================================================================
    2017: {
        'GNI_anual': 51967,
        'ded_especial_4ta': 249442,
        'tramos': [
            (20000,          0.05),
            (40000,          0.09),
            (60000,          0.12),
            (80000,          0.15),
            (120000,         0.19),
            (160000,         0.23),
            (240000,         0.27),
            (320000,         0.31),
            (float('inf'),   0.35),
        ]
    },

    # ==================================================================
    # 2018
    # GNI: $66.918 | Ded. Esp. 4ta: $321.206
    # Norma: Leyes 27.346 y 27.430
    # REFORMA: Ley 27.430 actualiza los tramos de la escala por RIPTE
    # (Remuneración Imponible Promedio de Trabajadores Estables).
    # Los tramos pasan a actualizarse anualmente.
    # Se limita la deducción por hijos a menores de 18 años.
    # ==================================================================
    2018: {
        'GNI_anual': 66918,
        'ded_especial_4ta': 321206,
        'tramos': [
            (25754,          0.05),
            (51508,          0.09),
            (77262,          0.12),
            (103016,         0.15),
            (154524,         0.19),
            (206032,         0.23),
            (309048,         0.27),
            (412064,         0.31),
            (float('inf'),   0.35),
        ]
    },

    # ==================================================================
    # 2019
    # GNI: $103.019 | Ded. Esp. 4ta: $494.490
    # Norma: Ley 27.346
    # ==================================================================
    2019: {
        'GNI_anual': 103019,
        'ded_especial_4ta': 494490,
        'tramos': [
            (33040,          0.05),
            (66080,          0.09),
            (99119,          0.12),
            (132159,         0.15),
            (198239,         0.19),
            (264318,         0.23),
            (396478,         0.27),
            (528637,         0.31),
            (float('inf'),   0.35),
        ]
    },

    # ==================================================================
    # 2020
    # GNI: $123.861 | Ded. Esp. 4ta: $594.534
    # Norma: Ley 27.346
    # ==================================================================
    2020: {
        'GNI_anual': 123861,
        'ded_especial_4ta': 594534,
        'tramos': [
            (47669,          0.05),
            (95338,          0.09),
            (143007,         0.12),
            (190677,         0.15),
            (286015,         0.19),
            (381353,         0.23),
            (572030,         0.27),
            (762707,         0.31),
            (float('inf'),   0.35),
        ]
    },

    # ==================================================================
    # 2021
    # GNI: $167.678 | Ded. Esp. 4ta: $804.856
    # Norma: Leyes 27.346, 27.617 y Decreto 620/2021
    #        R.G. 5008 AFIP y R.G. 5076 AFIP
    #
    # Ley 27.617 — Deducción especial INCREMENTADA:
    #   Enero–agosto 2021: exentos si remuneración bruta ≤ $150.000
    #     (o promedio mensual a ese mes, el menor)
    #   Septiembre–diciembre 2021: exentos si remuneración ≤ $175.000
    # Para la modelización anual se usa el umbral de H1 ($150.000)
    # como proxy conservador.
    # ==================================================================
    2021: {
        'GNI_anual': 167678,
        'ded_especial_4ta': 804856,
        'umbral_exencion_mensual': 150000,  # H1; H2 = 175.000
        'tramos': [
            (64533,          0.05),
            (129065,         0.09),
            (193598,         0.12),
            (258131,         0.15),
            (387196,         0.19),
            (516261,         0.23),
            (774392,         0.27),
            (1032522,        0.31),
            (float('inf'),   0.35),
        ]
    },

    # ==================================================================
    # 2022
    # GNI: $252.565 | Ded. Esp. 4ta: $1.212.311
    # Norma: Leyes 27.346 y 27.617
    #
    # Deducción especial INCREMENTADA — tres sub-períodos:
    #   Enero–mayo 2022:     exentos si bruto ≤ $225.937
    #   Junio–octubre 2022:  exentos si bruto ≤ $280.792
    #   Noviembre–dic 2022:  exentos si bruto ≤ $330.000
    # Para la modelización anual se usa $280.792 (umbral del período
    # más largo del año, H2).
    # ==================================================================
    2022: {
        'GNI_anual': 252565,
        'ded_especial_4ta': 1212311,
        'umbral_exencion_mensual': 280792,
        'tramos': [
            (97202,          0.05),
            (194404,         0.09),
            (291606,         0.12),
            (388808,         0.15),
            (583212,         0.19),
            (777616,         0.23),
            (1166424,        0.27),
            (1555232,        0.31),
            (float('inf'),   0.35),
        ]
    },

    # ==================================================================
    # 2023
    # GNI: $451.683 | Ded. Esp. 4ta: $2.168.079
    # Norma: Ley 27.743 (arts. 76 y 77), Decretos 415/2023 y 473/2023
    #        R.G. 5402 y 5417 AFIP
    #
    # AÑO MUY COMPLEJO — cuatro sub-períodos para 4ta categoría:
    #
    #   Enero–septiembre 2023 (escala +35% por Decreto 415/2023):
    #     Tramos estándar con escala actualizada 35%
    #     Umbral exención ene-abr: $404.062; may-jul: $506.230;
    #                              ago-sep: $700.875
    #
    #   Octubre 2023 (Decreto 473/2023 — tabla especial mensual):
    #     Exentos si bruto ≤ $1.980.000 (15× SMVM oct. 2023)
    #     Tabla mensual acumulada con primer tramo al 0%
    #
    #   Noviembre 2023: otra tabla especial mensual acumulada
    #   Diciembre 2023: otra tabla especial mensual acumulada
    #
    # Para la modelización ANUAL se usa la escala ene-sep (la que
    # aplica a 9 de los 12 meses) con el umbral de ago-sep ($700.875)
    # como el más representativo del segundo semestre.
    # ==================================================================
    2023: {
        'GNI_anual': 451683,
        'ded_especial_4ta': 2168079,
        'umbral_exencion_mensual': 700875,  # Proxy H2; ver nota
        'tramos': [
            # Escala enero–septiembre 2023 (Decreto 415/2023, +35% vs. 2022)
            (234677,         0.05),
            (469353,         0.09),
            (704030,         0.12),
            (938707,         0.15),
            (1408060,        0.19),
            (1877414,        0.23),
            (2816121,        0.27),
            (3754828,        0.31),
            (float('inf'),   0.35),
        ]
    },

    # ==================================================================
    # 2024
    # GNI: $3.503.688 | Ded. Esp. 4ta: $16.817.703
    # Norma: Ley 27.743 ("Paquete Fiscal")
    #
    # REFORMA MASIVA: La Ley 27.743 reemplazó el régimen de exención
    # temporal por una actualización permanente de la GNI y la escala
    # por IPC semestral. Los tramos anuales definitivos del PDF son
    # los que se publican para la liquidación anual final.
    # Se incorpora la "Deducción Especial Gravabilidad del SAC" (1/12
    # de las deducciones totales). Para la modelización anual se omite
    # ese componente adicional (es relativamente pequeño: ~8% extra).
    # ==================================================================
    2024: {
        'GNI_anual': 3503688,
        'ded_especial_4ta': 16817703,
        'tramos': [
            (1360200,        0.05),
            (2720400,        0.09),
            (4080600,        0.12),
            (6120900,        0.15),
            (12241800,       0.19),
            (18362700,       0.23),
            (27544050,       0.27),
            (41316075,       0.31),
            (float('inf'),   0.35),
        ]
    },
}

print(f"✓ Tabla de parámetros de Ganancias cargada: {len(GANANCIAS_PARAMS)} años (2004–2024)")
print()
# Mostrar resumen de la evolución de la GNI
print(f"{'Año':<6} {'GNI anual':>12} {'Ded. Esp. 4ta':>15} {'Tramos':>7} {'Umbral exenc. mensual':>22}")
print("-" * 70)
for anio, p in sorted(GANANCIAS_PARAMS.items()):
    umbral = p.get('umbral_exencion_mensual', '—')
    umbral_str = f"${umbral:,.0f}" if isinstance(umbral, (int, float)) else umbral
    print(f"{anio:<6} {p['GNI_anual']:>12,.0f} {p['ded_especial_4ta']:>15,.0f} "
          f"{len(p['tramos']):>7}   {umbral_str:>20}")

✓ Tabla de parámetros de Ganancias cargada: 21 años (2004–2024)

Año       GNI anual   Ded. Esp. 4ta  Tramos  Umbral exenc. mensual
----------------------------------------------------------------------
2004          4,020          13,500       7                      —
2005          4,020          13,500       7                      —
2006          6,000          22,800       7                      —
2007          7,500          36,000       7                      —
2008          9,000          43,200       7                      —
2009          9,000          43,200       7                      —
2010         10,800          51,840       7                      —
2011         12,960          62,208       7                      —
2012         12,960          62,208       7                      —
2013         15,552          97,044       7                $15,000
2014         15,552          97,044       7                $15,000
2015         15,552          97,044       7                $

In [2]:
# Guardar la tabla en formatos útiles para la Fase 3

# Formato 1: CSV con parámetros escalares por año (GNI, ded. especial, umbral)
filas_csv = []
for anio, p in sorted(GANANCIAS_PARAMS.items()):
    filas_csv.append({
        'anio':                    anio,
        'GNI_anual':               p['GNI_anual'],
        'ded_especial_4ta':        p['ded_especial_4ta'],
        'umbral_exencion_mensual': p.get('umbral_exencion_mensual', None),
    })
df_gni = pd.DataFrame(filas_csv)
df_gni.to_csv(RUTAS['alicuotas'] / "ganancias_deducciones_2004_2024.csv", index=False)

# Formato 2: JSON con la estructura completa (incluyendo tramos) para uso programático
# Convertir float('inf') a una cadena para que sea JSON-serializable
params_json = {}
for anio, p in GANANCIAS_PARAMS.items():
    params_json[anio] = {
        'GNI_anual':               p['GNI_anual'],
        'ded_especial_4ta':        p['ded_especial_4ta'],
        'umbral_exencion_mensual': p.get('umbral_exencion_mensual', None),
        'tramos': [
            [limite if limite != float('inf') else 'inf', alicuota]
            for limite, alicuota in p['tramos']
        ]
    }

import json
with open(RUTAS['alicuotas'] / "ganancias_escala_2004_2024.json", 'w', encoding='utf-8') as f:
    json.dump(params_json, f, indent=2, ensure_ascii=False)

print("✅ Parámetros de Ganancias guardados:")
print(f"   → {(RUTAS['alicuotas'] / 'ganancias_deducciones_2004_2024.csv').name} (CSV escalar)")
print(f"   → {(RUTAS['alicuotas'] / 'ganancias_escala_2004_2024.json').name} (JSON con tramos)")

NameError: name 'RUTAS' is not defined

In [21]:
# FUNCIÓN DE VALIDACIÓN: calcular la alícuota efectiva para un ingreso dado en un año dado
# Esto permite verificar que los tramos están bien cargados comparando con casos conocidos

def calcular_alicuota_efectiva_ganancias(ingreso_laboral_anual_bruto, anio,
                                          es_4ta_categoria=True):
    """
    Calcula la alícuota efectiva (impuesto / ingreso bruto) para una persona
    con el ingreso bruto anual indicado.
    
    Esta función es para VALIDACIÓN. En la Fase 3 se aplicará a toda la EPH.
    
    Parámetros:
    -----------
    ingreso_laboral_anual_bruto : float
        Ingreso laboral bruto anual en pesos nominales del año
    anio : int
        Año fiscal (2004–2024)
    es_4ta_categoria : bool
        Si True, aplica la deducción especial de 4ta categoría (relación de dependencia)
    
    Devuelve:
    ---------
    dict con impuesto_anual, alicuota_efectiva, base_imponible
    """
    p = GANANCIAS_PARAMS[anio]
    
    # Verificar umbral de exención (Decreto 1242/2012, Ley 27617, etc.)
    umbral_mensual = p.get('umbral_exencion_mensual', None)
    ingreso_mensual_bruto = ingreso_laboral_anual_bruto / 12
    if umbral_mensual and ingreso_mensual_bruto <= umbral_mensual:
        return {
            'impuesto_anual': 0,
            'alicuota_efectiva': 0.0,
            'base_imponible': 0,
            'razon': 'umbral_exencion'
        }
    
    # Deducción: GNI + deducción especial (si aplica)
    deducciones = p['GNI_anual']
    if es_4ta_categoria:
        deducciones += p['ded_especial_4ta']
    
    # Base imponible neta
    base = max(0, ingreso_laboral_anual_bruto - deducciones)
    
    if base == 0:
        return {
            'impuesto_anual': 0,
            'alicuota_efectiva': 0.0,
            'base_imponible': 0,
            'razon': 'bajo_minimo'
        }
    
    # Calcular impuesto según escala progresiva
    impuesto = 0.0
    tramo_anterior = 0
    for limite, alicuota in p['tramos']:
        if base <= tramo_anterior:
            break
        tope = min(base, limite)
        impuesto += (tope - tramo_anterior) * alicuota
        tramo_anterior = limite
        if limite == float('inf'):
            break
    
    alicuota_efectiva = impuesto / ingreso_laboral_anual_bruto if ingreso_laboral_anual_bruto > 0 else 0
    
    return {
        'impuesto_anual': round(impuesto, 2),
        'alicuota_efectiva': round(alicuota_efectiva, 4),
        'base_imponible': round(base, 2),
        'razon': 'calculado'
    }


# --- VALIDACIONES ---
print("Validaciones de la función de Ganancias:")
print("(Comparar con casos reales o con la calculadora de ARCA para verificar)")
print()

casos = [
    (2024, 2_000_000,  "Ingreso bajo — debería ser 0% (bajo GNI)"),
    (2024, 5_000_000,  "Ingreso medio 2024"),
    (2024, 20_000_000, "Ingreso alto 2024"),
    (2024, 50_000_000, "Ingreso muy alto 2024"),
    (2010, 30_000,     "Ingreso bajo 2010"),
    (2010, 80_000,     "Ingreso medio 2010"),
    (2010, 200_000,    "Ingreso alto 2010"),
    (2021, 1_400_000,  "Bajo umbral exención Ley 27617 2021"),
    (2021, 2_000_000,  "Sobre umbral exención Ley 27617 2021"),
]

print(f"{'Año':<6} {'Ingreso bruto anual':>22} {'Alíc. efectiva':>16} {'Impuesto':>14}  Caso")
print("-" * 80)
for anio, ingreso, desc in casos:
    r = calcular_alicuota_efectiva_ganancias(ingreso, anio)
    print(f"{anio:<6} ${ingreso:>20,.0f} {r['alicuota_efectiva']:>15.2%} "
          f"${r['impuesto_anual']:>12,.0f}  {desc}")

Validaciones de la función de Ganancias:
(Comparar con casos reales o con la calculadora de ARCA para verificar)

Año       Ingreso bruto anual   Alíc. efectiva       Impuesto  Caso
--------------------------------------------------------------------------------
2024   $           2,000,000           0.00% $           0  Ingreso bajo — debería ser 0% (bajo GNI)
2024   $           5,000,000           0.00% $           0  Ingreso medio 2024
2024   $          20,000,000           0.00% $           0  Ingreso alto 2024
2024   $          50,000,000          12.74% $   6,371,153  Ingreso muy alto 2024
2010   $              30,000           0.00% $           0  Ingreso bajo 2010
2010   $              80,000           2.41% $       1,930  Ingreso medio 2010
2010   $             200,000          17.29% $      34,576  Ingreso alto 2010
2021   $           1,400,000           0.00% $           0  Bajo umbral exención Ley 27617 2021
2021   $           2,000,000          11.44% $     228,814  Sobre 

Checklist de Fase 1 completa

[X] EPH: bases individuos + hogares para T1–T4 de cada año 2003–2024 en formato legible

[X] ENGHo: bases de 2004/05, 2012/13 y 2017/18 descargadas y exploradas

[X] Ganancias: tabla de parámetros (GNI + escala) para cada año 2003–2024

[ ] IVA: tabla de alícuotas por categoría COICOP, con cambios históricos

[ ] Impuestos Internos: alícuotas por producto para cada año

[ ] Aportes/contribuciones: tasas por período

[ ] IIBB: alícuota promedio ponderada por actividad y año

[ ] Retenciones: alícuotas históricas por commodity

[ ] ARCA: series de recaudación total por impuesto (para validación posterior)

In [22]:
import pandas as pd
from pathlib import Path

BASE_PATH = Path(r"C:\Users\LENOVO L340\Desktop\argentina-tax-incidence")
ruta_engho = BASE_PATH / "data/raw/engho"

# Leer gastos 2004
ruta_gastos = list((ruta_engho / 'ENGH0405_gastos').glob('*.txt'))[0]
df_g = pd.read_csv(ruta_gastos, sep=None, engine='python', encoding='latin-1')
df_g.columns = df_g.columns.str.strip().str.upper()

print("=== ENGHo 2004 — columnas disponibles ===")
print(list(df_g.columns))
print(f"\nFilas: {len(df_g):,}")
print("\n=== Muestra de columna ARTICULO (primeros 20 valores únicos) ===")
print(df_g['ARTICULO'].astype(str).unique()[:20])
print(f"\nLongitudes de ARTICULO:")
print(df_g['ARTICULO'].astype(str).str.len().value_counts().sort_index())

=== ENGHo 2004 — columnas disponibles ===
['CLAVE', 'ARTICULO', 'MIEMBRO', 'MRCGIMPTD', 'MRCCIMPTD', 'REGION', 'SUBREGION', 'PROVINCIA', 'EXPAN', 'SUBCLASE', 'CLASE', 'GRUPO', 'DIVISION', 'CANTIDAD', 'UN_MED', 'MONTO']

Filas: 1,344,137

=== Muestra de columna ARTICULO (primeros 20 valores únicos) ===
['0' '2' '1' '3' '5' '6' '4' '9' '7' '8' '11' '10' '15' '14' '12' '13']

Longitudes de ARTICULO:
ARTICULO
1    1343917
2        220
Name: count, dtype: int64


In [23]:
ruta_gastos_2012 = ruta_engho / "bases_datos_engho2012" / "ENGHo - Gastos.txt"
df_g2012 = pd.read_csv(ruta_gastos_2012, sep=None, engine='python', encoding='latin-1')
df_g2012.columns = df_g2012.columns.str.strip().str.upper()

print("=== ENGHo 2012 — columnas disponibles ===")
print(list(df_g2012.columns))
print(f"\nFilas: {len(df_g2012):,}")
print("\n=== Muestra de columna ARTICULO (primeros 20 valores únicos) ===")
print(df_g2012['ARTICULO'].astype(str).unique()[:20])
print(f"\nLongitudes de ARTICULO:")
print(df_g2012['ARTICULO'].astype(str).str.len().value_counts().sort_index())
print("\n=== Columnas adicionales de clasificación ===")
for col in ['DIVISION', 'GRUPO', 'CLASE', 'SUBCLASE', 'COICOP']:
    if col in df_g2012.columns:
        print(f"\n{col} — primeros 10 valores únicos:")
        print(df_g2012[col].astype(str).unique()[:10])

=== ENGHo 2012 — columnas disponibles ===
['CLAVE', 'ARTICULO', 'REGION', 'SUBREGION', 'PROVINCIA', 'EXPAN', 'SUBCLASE', 'CLASE', 'GRUPO', 'DIVISION', 'CANTIDAD', 'UN_MED', 'MONTO', 'MIEMBRO', 'MONTOIMPUTADO', 'CANTIDADIMPUTADA']

Filas: 1,116,328

=== Muestra de columna ARTICULO (primeros 20 valores únicos) ===
['341101' '421104' '117129' '117119' '117128' '111101' '462101' '331101'
 '115208' '119302' '212217' '441103' '212103' '111212' '111111' '121109'
 '821102' '322101' '119303' '212401']

Longitudes de ARTICULO:
ARTICULO
6    1116328
Name: count, dtype: int64

=== Columnas adicionales de clasificación ===

DIVISION — primeros 10 valores únicos:
['300000' '400000' '100000' '200000' '800000' '500000' '700000' '900000'
 '600000']

GRUPO — primeros 10 valores únicos:
['340000' '420000' '110000' '460000' '330000' '210000' '440000' '120000'
 '820000' '320000']

CLASE — primeros 10 valores únicos:
['341000' '421000' '117000' '111000' '462000' '331000' '115000' '119000'
 '212000' '441000'

In [24]:
import pandas as pd
from pathlib import Path

BASE_PATH = Path(r"C:\Users\LENOVO L340\Desktop\argentina-tax-incidence")
ruta_engho = BASE_PATH / "data/raw/engho"
ruta_gastos = list((ruta_engho / 'ENGH0405_gastos').glob('*.txt'))[0]

df = pd.read_csv(ruta_gastos, sep=None, engine='python', encoding='latin-1')
df.columns = df.columns.str.strip().str.upper()
df['MONTO'] = pd.to_numeric(df['MONTO'], errors='coerce').fillna(0)

# Ver las 4 columnas de clasificación
for col in ['DIVISION', 'GRUPO', 'CLASE', 'SUBCLASE']:
    print(f"\n=== {col} ===")
    vals = df[col].astype(str).str.strip()
    print(f"  Longitudes: {vals.str.len().value_counts().sort_index().to_dict()}")
    print(f"  Primeros 15 valores únicos: {sorted(vals.unique())[:15]}")
    print(f"  Nulos/vacíos: {vals.isin(['nan','','0']).sum():,}")

# Ver 10 filas completas para entender la jerarquía
print("\n=== EJEMPLO DE 10 FILAS ===")
print(df[['DIVISION','GRUPO','CLASE','SUBCLASE','ARTICULO','MONTO']].head(10).to_string())

# Ver también los valores más frecuentes de DIVISION para entender el esquema
print("\n=== DIVISION — 20 valores más frecuentes ===")
print(df['DIVISION'].astype(str).value_counts().head(20).to_string())


=== DIVISION ===
  Longitudes: {3: 623042, 4: 379736, 5: 196221, 6: 76592, 7: 66916, 8: 1478, 9: 145, 10: 4, 11: 3}
  Primeros 15 valores únicos: ['0.0', '0.00129', '0.00134', '0.00172', '0.00194', '0.00215', '0.00323', '0.0043', '0.00452', '0.00538', '0.00645', '0.00667', '0.00753', '0.00774', '0.00833']
  Nulos/vacíos: 37,364

=== GRUPO ===
  Longitudes: {6: 1344137}
  Primeros 15 valores únicos: ['100000', '200000', '300000', '400000', '500000', '600000', '700000', '800000', '900000']
  Nulos/vacíos: 0

=== CLASE ===
  Longitudes: {6: 1344137}
  Primeros 15 valores únicos: ['110000', '120000', '130000', '210000', '220000', '310000', '320000', '330000', '340000', '410000', '420000', '430000', '440000', '450000', '460000']
  Nulos/vacíos: 0

=== SUBCLASE ===
  Longitudes: {6: 1344137}
  Primeros 15 valores únicos: ['111000', '112000', '113000', '114000', '115000', '116000', '117000', '118000', '119000', '121000', '122000', '131000', '211000', '212000', '213000']
  Nulos/vacíos: 0

==

---
## SECCIÓN 9 — Checklist final de Fase 1

In [25]:
import os

def chk(condicion, nombre):
    estado = "✅" if condicion else "❌"
    print(f"  {estado}  {nombre}")
    return condicion

print("=" * 60)
print("CHECKLIST — FASE 1")
print("=" * 60)

# EPH
print("\n📊 EPH (2004–2024):")
parquets = [RUTAS['eph_anual'] / f"eph_anual_{a}.parquet" for a in range(2004, 2025)]
n_ok = sum(1 for p in parquets if p.exists())
chk(n_ok == 21, f"Parquets presentes: {n_ok}/21")

# Verificar que los parquets con datos parciales están marcados
marcados = 0
for anio in [2007, 2011, 2015, 2016]:
    p = RUTAS['eph_anual'] / f"eph_anual_{anio}.parquet"
    if p.exists():
        # Leer solo la columna TRIMESTRE para no sobrecargar la RAM
        df = pd.read_parquet(p, columns=['TRIMESTRE'])
        # Contar cuántos trimestres únicos hay en ese año
        if df['TRIMESTRE'].nunique() < 4:
            marcados += 1
chk(marcados == 4, f"Años parciales marcados con metadata: {marcados}/4")

# Verificar que no hay ingresos negativos en los parquets
n_negativos_total = 0
for p in parquets:
    if p.exists():
        df = pd.read_parquet(p, columns=['IPCF'])
        n_neg = (df['IPCF'] < 0).sum()
        n_negativos_total += n_neg
chk(n_negativos_total == 0, f"Sin ingresos negativos en IPCF (encontrados: {n_negativos_total})")

# ENGHo
print("\n🛒 ENGHo:")
chk((RUTAS['engho_struct'] / "engho_estructura_2004.csv").exists(), "ENGHo 2004/05 procesada")
hay_2012 = (RUTAS['engho_struct'] / "engho_estructura_2012.csv").exists()
if hay_2012:
    chk(True, "ENGHo 2012/13 procesada ✓")
else:
    print("  ⚠️   ENGHo 2012/13 no disponible — se usa interpolación")
chk((RUTAS['engho_struct'] / "engho_estructura_2017.csv").exists(), "ENGHo 2017/18 procesada")
chk((RUTAS['engho_struct'] / "engho_estructura_serie_2004_2024.csv").exists(),
    "Serie de estructuras interpolada (2004–2024)")

# Ganancias
print("\n💼 Ganancias:")
chk((RUTAS['alicuotas'] / "ganancias_deducciones_2004_2024.csv").exists(),
    "CSV de deducciones (GNI + ded. especial) 2004–2024")
chk((RUTAS['alicuotas'] / "ganancias_escala_2004_2024.json").exists(),
    "JSON con escala completa de alícuotas 2004–2024")
chk(len(GANANCIAS_PARAMS) == 21, f"Años cubiertos: {len(GANANCIAS_PARAMS)}/21")

print("\n" + "=" * 60)
print("PENDIENTE para siguientes pasos:")
print("  [ ] IVA: tabla de alícuotas por categoría COICOP 2004–2024")
print("  [ ] Impuestos Internos: alícuotas por producto y año")
print("  [ ] Aportes/contribuciones: tasas por período")
print("  [ ] IIBB: alícuota promedio ponderada por actividad")
print("  [ ] Retenciones: alícuotas históricas por commodity")
print("  [ ] ARCA: series de recaudación (para validación Fase 5)")

CHECKLIST — FASE 1

📊 EPH (2004–2024):
  ❌  Parquets presentes: 20/21
  ✅  Años parciales marcados con metadata: 4/4
  ✅  Sin ingresos negativos en IPCF (encontrados: 0)

🛒 ENGHo:
  ✅  ENGHo 2004/05 procesada
  ✅  ENGHo 2012/13 procesada ✓
  ✅  ENGHo 2017/18 procesada
  ✅  Serie de estructuras interpolada (2004–2024)

💼 Ganancias:
  ✅  CSV de deducciones (GNI + ded. especial) 2004–2024
  ✅  JSON con escala completa de alícuotas 2004–2024
  ✅  Años cubiertos: 21/21

PENDIENTE para siguientes pasos:
  [ ] IVA: tabla de alícuotas por categoría COICOP 2004–2024
  [ ] Impuestos Internos: alícuotas por producto y año
  [ ] Aportes/contribuciones: tasas por período
  [ ] IIBB: alícuota promedio ponderada por actividad
  [ ] Retenciones: alícuotas históricas por commodity
  [ ] ARCA: series de recaudación (para validación Fase 5)


Sección 9 — Tabla de alícuotas IVA (2004–2024)
Esta es la que construís primero porque es la que más peso tiene en la distribución y la más simple de codificar. Te escribo la celda completa lista para pegar en la notebook:

In [ ]:
### VERIFICAR MANUALMENTE 


# =============================================================================
# SECCIÓN 9 — PARÁMETROS DE IVA (2004–2024)
# =============================================================================
# Fuente: AFIP/ARCA — Ley 23.349 y modificaciones
# Alícuotas por categoría COICOP (para aplicar sobre estructura ENGHo)
#
# Las categorías usan los mismos nombres que engho_estructura_serie_2004_2024.csv
# para que el merge sea directo en Fase 3.
#
# REGLA GENERAL:
#   - Alícuota general: 21% (vigente desde 1995, sin cambios)
#   - Alícuota diferencial: 10.5% (bienes de la canasta básica, medicamentos)
#   - Exento: 0%
#
# CAMBIOS HISTÓRICOS RELEVANTES:
#   2001: IVA temporario sobre servicios financieros (no modelizamos)
#   2009: Decreto 1231/09 baja temporariamente algunos productos de
#         10.5% a 0% (no implementado — efecto menor y transitorio)
#   2019: Decreto 567/19 baja alimentos de canasta básica a 0% (agosto–dic)
#         Decreto 928/20 extiende hasta dic 2020
#         Ley 27.651 (2021) hace permanente la alícuota 0% en algunos productos
#   2024: No hay cambios de alícuota en el período analizado
# =============================================================================

IVA_ALICUOTAS_POR_CATEGORIA = {

    # ------------------------------------------------------------------
    # Regla base 2004–2018: sin cambios estructurales
    # ------------------------------------------------------------------
    'base': {
        # Categoría COICOP                       : (alícuota_21, alícuota_diferencial, exento)
        # Para cada categoría devolvemos UN número: la alícuota promedio efectiva
        # ponderada internamente (algunos artículos al 21%, otros al 10.5%, otros exentos)
        # Fuente ponderaciones: Rossignolo (2014), CEPAL (2018)
        
        'alimentos_y_bebidas':                         0.091,  # Mix: excentos, básicos 10.5%, otros 21%; prom. pond.
        'bebidas_alcoholicas_y_tabaco':                0.210,  # 21% pleno
        'indumentaria_y_calzado':                      0.210,  # 21%
        'propiedades_combustibles_agua_electricidad':  0.105,  # Electricidad/gas residencial 10.5%; alquileres exentos
        'equipamiento_y_mantenimiento':                0.210,  # 21%
        'salud':                                       0.042,  # Medicamentos 10.5%, servicios médicos exentos; pond.
        'transporte':                                  0.105,  # Transporte público exento; combustibles 21%; pond.
        'comunicaciones':                              0.210,  # 21%
        'esparcimiento_y_cultura':                     0.168,  # Libros exentos; resto 21%; pond.
        'ensenanza':                                   0.000,  # Exento (art. 7 Ley 23.349)
        'restaurantes_y_hoteles':                      0.210,  # 21%
        'bienes_y_servicios_varios':                   0.189,  # Mix servicios personales 21% y algunos 10.5%
    },

    # ------------------------------------------------------------------
    # 2019–2020: Decreto 567/19 — canasta básica a 0%
    # Los alimentos de la canasta básica (harina, aceite, fideos, arroz,
    # leche, azúcar, legumbres, yerba, té) pasan a alícuota 0%.
    # Esto reduce la alícuota efectiva de alimentos_y_bebidas.
    # Estimación: canasta básica ≈ 40% del gasto en alimentos → efecto ≈ -3.5pp
    # ------------------------------------------------------------------
    'ajuste_2019_2020': {
        'alimentos_y_bebidas': 0.056,  # Reducido desde 0.091 por canasta a 0%
    },

    # ------------------------------------------------------------------
    # 2021 en adelante: Ley 27.651 hace permanente la alícuota reducida
    # para productos de la canasta básica alimentaria.
    # La alícuota efectiva se estabiliza en el nivel post-decreto.
    # ------------------------------------------------------------------
    'ajuste_2021_2024': {
        'alimentos_y_bebidas': 0.056,  # Mantenido
    },
}

# Construir serie histórica completa por año y categoría
import pandas as pd

categorias = list(IVA_ALICUOTAS_POR_CATEGORIA['base'].keys())
registros = []

for anio in range(2004, 2025):
    for cat in categorias:
        # Alícuota base
        alicuota = IVA_ALICUOTAS_POR_CATEGORIA['base'][cat]
        
        # Aplicar ajustes históricos si corresponde
        if anio in [2019, 2020]:
            ajuste = IVA_ALICUOTAS_POR_CATEGORIA['ajuste_2019_2020']
            if cat in ajuste:
                alicuota = ajuste[cat]
        elif anio >= 2021:
            ajuste = IVA_ALICUOTAS_POR_CATEGORIA['ajuste_2021_2024']
            if cat in ajuste:
                alicuota = ajuste[cat]
        
        registros.append({
            'anio': anio,
            'categoria_coicop': cat,
            'alicuota_iva': alicuota,
        })

df_iva = pd.DataFrame(registros)
df_iva.to_csv(RUTAS['alicuotas'] / "iva_alicuotas_2004_2024.csv", index=False)

print(f"✅ IVA guardado: {len(df_iva)} filas ({df_iva['anio'].nunique()} años × {df_iva['categoria_coicop'].nunique()} categorías)")
print("\nVerificación — alícuotas efectivas por categoría (año 2018 y 2020):")
tabla_check = df_iva[df_iva['anio'].isin([2018, 2020])].pivot(index='categoria_coicop', columns='anio', values='alicuota_iva')
print(tabla_check.to_string())

✅ IVA guardado: 252 filas (21 años × 12 categorías)

Verificación — alícuotas efectivas por categoría (año 2018 y 2020):
anio                                         2018   2020
categoria_coicop                                        
alimentos_y_bebidas                         0.091  0.056
bebidas_alcoholicas_y_tabaco                0.210  0.210
bienes_y_servicios_varios                   0.189  0.189
comunicaciones                              0.210  0.210
ensenanza                                   0.000  0.000
equipamiento_y_mantenimiento                0.210  0.210
esparcimiento_y_cultura                     0.168  0.168
indumentaria_y_calzado                      0.210  0.210
propiedades_combustibles_agua_electricidad  0.105  0.105
restaurantes_y_hoteles                      0.210  0.210
salud                                       0.042  0.042
transporte                                  0.105  0.105


## Sección 10 — Aportes y contribuciones SIPA (2004–2024)

1. El "Bache" del 7% en la era AFJP (2004–2007)
La tasa de aporte personal fue del 11% + 3% + 3% = 17% desde 2007.
Es correcto que desde 2008 fue así, pero entre 2003 y fines de 2007, producto de la crisis de 2001, el Gobierno había reducido el aporte jubilatorio personal del 11% al 7% para los trabajadores afiliados a las AFJP (que eran la inmensa mayoría del sector privado formal) para incentivar el consumo (Decreto 390/2003). Recién con la Ley 26.222 a fines de 2007, la tasa volvió al 11% para todos a partir de 2008.

Cobrás el 17% a los asalariados formales en 2004, 2005, 2006 y 2007, sobreestima su carga tributaria. Debería ser 13% (7% + 3% + 3%).

2. El Límite Máximo Imponible (¡Crucial!)
A diferencia del Impuesto a las Ganancias (que no tiene techo), los aportes personales del 17% tienen un tope legal mensual (históricamente atado al MOPRE y luego al RIPTE).

Ejemplo: Si hoy el tope es de $2.000.000, una persona que gana $1.500.000 paga el 17% ($255.000). Pero un CEO que gana $10.000.000 no paga el 17% ($1.700.000); paga el 17% del tope legal (sólo $340.000, lo que representa una alícuota efectiva de apenas el 3.4%).

Impacto: Si aplico un 17% lineal a todo el mundo sin poner el tope, el modelo va a dar que el sistema es mucho más progresivo de lo que realmente es, porque le voy a estar sobreestimando lo que le saco a los más ricos.

In [ ]:
# =============================================================================
# SECCIÓN 10 — APORTES PERSONALES Y CONTRIBUCIONES PATRONALES (2004–2024)
# VERSIÓN CORREGIDA — incluye bache AFJP y Límite Máximo Imponible
# =============================================================================
# Fuentes:
#   - Decreto 390/2003: reduce aporte jubilatorio a 7% para afiliados AFJP
#   - Ley 26.222 (2007): restituye 11% desde abril 2007
#   - Ley 26.425 (2008): elimina AFJP, todos al reparto
#   - ANSeS: publicaciones anuales del LMI (tope de la base imponible)
#   - Decreto 814/2001: reduce contribuciones patronales por sector
#   - Ley 27.541 (2020): unifica contribuciones patronales en 24%
# =============================================================================

APORTES_PARAMS = {}

# ------------------------------------------------------------------
# Límite Máximo Imponible mensual (en $ nominales)
# Fuente: Resoluciones ANSeS por año
# Este es el salario bruto mensual hasta el cual se aplican los aportes.
# Por encima de este tope, la alícuota efectiva cae inversamente al ingreso.
# ------------------------------------------------------------------


## VERIFICAR MANUALMENTE! 


LMI_MENSUAL = {
    # Pre-2004: referencia histórica (MOPRE $80 × 60 = $4.800)
    2004: 4_800,
    2005: 4_627,
    2006: 4_627,
    2007: 7_590,   # Actualización por Res. ANSeS 97/2007
    2008: 10_269,  # Suba post-Ley 26.222 y 26.425
    2009: 12_030,
    2010: 14_603,
    2011: 18_726,
    2012: 23_420,
    2013: 30_088,
    2014: 39_866,
    2015: 52_392,
    2016: 69_133,
    2017: 88_194,
    2018: 122_145,
    2019: 180_396,
    2020: 296_626,
    2021: 426_575,
    2022: 757_877,
    2023: 1_834_012,
    2024: 6_345_642,  # Actualización RIPTE semestral
}

for anio in range(2004, 2025):

    # ------------------------------------------------------------------
    # APORTES PERSONALES
    # Corrección histórica: 2004–2007 → jubilación al 7% (bache AFJP)
    # Desde 2008: jubilación al 11% (Ley 26.222 + eliminación AFJP)
    # ------------------------------------------------------------------
    if anio <= 2007:
        # Decreto 390/2003: aporte jubilatorio reducido al 7% para AFJP
        # En la práctica ~85% del sector privado formal estaba en AFJP
        # Simplificamos: usamos 7% para todo el sector formal privado
        # (trabajadores del sector público ya estaban en reparto al 11%)
        # NOTA: esto levemente subestima la carga del sector público;
        # documentar como limitación menor.
        aporte_jubilacion = 0.07
    else:
        aporte_jubilacion = 0.11

    aporte_obra_social = 0.03
    aporte_pami        = 0.03
    total_aportes = aporte_jubilacion + aporte_obra_social + aporte_pami

    # ------------------------------------------------------------------
    # CONTRIBUCIONES PATRONALES
    # Decreto 814/2001: alícuota diferenciada por sector
    #   - Servicios/comercio: 21%
    #   - Industria/resto: 17%
    #   Promedio ponderado por empleo formal: ~19% hasta 2019
    # Ley 27.541 (2020): unificación en 24%
    # ------------------------------------------------------------------
    if anio <= 2019:
        contrib_patronal = 0.190
    else:
        contrib_patronal = 0.240

    APORTES_PARAMS[anio] = {
        'aporte_jubilacion':           aporte_jubilacion,
        'aporte_obra_social':          aporte_obra_social,
        'aporte_pami':                 aporte_pami,
        'total_aportes_personales':    total_aportes,
        'contrib_patronal':            contrib_patronal,
        'contrib_patronal_trabajador': contrib_patronal * 0.50,
        'contrib_patronal_consumidor': contrib_patronal * 0.50,
        # LMI: tope mensual de la base imponible para aportes personales
        'lmi_mensual':                 LMI_MENSUAL[anio],
        'lmi_anual':                   LMI_MENSUAL[anio] * 12,
    }

# Guardar CSV
import pandas as pd
df_aportes = pd.DataFrame([
    {'anio': a, **{k: v for k, v in p.items()}}
    for a, p in APORTES_PARAMS.items()
])
df_aportes.to_csv(RUTAS['alicuotas'] / "aportes_contribuciones_2004_2024.csv", index=False)

# Resumen de verificación
print(f"{'Año':<6} {'Jubil.':>8} {'Total ap.':>10} {'LMI mensual':>13} {'Contrib.pat.':>13}")
print("-" * 55)
for anio, p in sorted(APORTES_PARAMS.items()):
    print(f"{anio:<6} {p['aporte_jubilacion']:>7.0%}  {p['total_aportes_personales']:>9.0%}  "
          f"${p['lmi_mensual']:>11,.0f}  {p['contrib_patronal']:>12.0%}")

Año      Jubil.  Total ap.   LMI mensual  Contrib.pat.
-------------------------------------------------------
2004        7%        13%  $      4,627           19%
2005        7%        13%  $      4,627           19%
2006        7%        13%  $      4,627           19%
2007        7%        13%  $      7,590           19%
2008       11%        17%  $     10,269           19%
2009       11%        17%  $     12,030           19%
2010       11%        17%  $     14,603           19%
2011       11%        17%  $     18,726           19%
2012       11%        17%  $     23,420           19%
2013       11%        17%  $     30,088           19%
2014       11%        17%  $     39,866           19%
2015       11%        17%  $     52,392           19%
2016       11%        17%  $     69,133           19%
2017       11%        17%  $     88,194           19%
2018       11%        17%  $    122,145           19%
2019       11%        17%  $    180,396           19%
2020       11%        17%

## Sección 11 — Impuestos Internos, IIBB y Retenciones

In [ ]:
# =============================================================================
# SECCIÓN 11 — IMPUESTOS INTERNOS E IIBB (2004–2024)
# =============================================================================
# Fuente Internos: Ley 24.674 y modificaciones.
# Fuente IIBB: estimación de tasa efectiva promedio nacional sobre precio
#              al consumidor (Rossignolo, 2014; CEPAL, 2018).
#
# RETENCIONES: excluidas del modelo. Su efecto es un subsidio implícito
# al consumidor (abaratan alimentos), no un impuesto sobre el consumo.
# Ver: data/processed/alicuotas/retenciones_metodologia.txt
# =============================================================================

import pandas as pd

# ------------------------------------------------------------------
# A. Impuestos Internos
# ------------------------------------------------------------------
# Se expresan como alícuota efectiva sobre el precio al consumidor
# (ya con IVA incluido), para ser directamente multiplicables
# por la estructura de gasto ENGHo en la Fase 3.
#
# Solo se cargan las categorías COICOP que tienen carga real de Internos.
# El resto queda en 0.0 implícitamente.
#
# Fuentes por categoría:
#   Tabaco/cigarrillos:  Ley 24.674 art. 15 — alícuota 60-75% sobre precio
#                        de venta al público. Efectiva sobre precio final: ~35%
#   Bebidas alcohólicas: Ley 24.674 arts. 23-26 — 8% (cerveza) a 20% (whisky)
#                        Promedio ponderado por consumo: ~12%
#   Combustibles:        Ley 23.966 — alícuota específica ($/litro).
#                        Como % del precio al surtidor: ~6% promedio histórico.
#                        Solo fracción del gasto en "transporte" es combustible
#                        de uso doméstico directo (estimado en ~30% de la cat.)
#   Telefonía celular:   5% desde Ley 24.674 (2001). Aplica sobre servicios
#                        de telecomunicaciones móviles.
#   Refrescos/aguas:     ~4% promedio sobre precio (Ley 24.674 art. 26 bis)
#                        Peso pequeño dentro de alimentos_y_bebidas: ~0.5% efectivo




### VERIFICAR MANUALMENTE!


INTERNOS_BASE = {
    'bebidas_alcoholicas_y_tabaco':  0.35,  # Tabaco domina la categoría; ~35% sobre precio final
    'transporte':                    0.018, # Combustibles × fracción de uso doméstico directo
    'comunicaciones':                0.050, # Telefonía celular
    'alimentos_y_bebidas':           0.005, # Refrescos/aguas; muy pequeño
}

# Cambios históricos relevantes:
#   2016: Ley 27.264 elimina Internos sobre computadoras/electrónicos
#         (categoría equipamiento — ya no tenía peso significativo)
#   2018: Decreto 1020/18 modifica Internos sobre bebidas azucaradas (menor)
# Estos cambios no alteran significativamente las categorías cargadas arriba.
# Se usa la misma tabla base para todo el período 2004–2024.

# ------------------------------------------------------------------
# B. Ingresos Brutos (IIBB)
# ------------------------------------------------------------------
# Alícuota efectiva promedio sobre el precio al consumidor.
# IIBB es un impuesto en cascada sobre facturación bruta. La tasa
# nominal varía por provincia (1.5%–3.5%) y sector. El efecto
# acumulado sobre el precio final al consumidor se estima en
# ~3.5% para el período 2004–2020 y ~3.8% post-2021 (presión
# fiscal provincial creciente).
#
# Se aplica de forma UNIFORME sobre todas las categorías de consumo
# porque está incorporado en el precio de cualquier bien o servicio
# que pasa por la cadena de comercialización.

IIBB_POR_ANIO = {}
for anio in range(2004, 2021):
    IIBB_POR_ANIO[anio] = 0.035
for anio in range(2021, 2025):
    IIBB_POR_ANIO[anio] = 0.038

# ------------------------------------------------------------------
# C. Construcción de la tabla final
# ------------------------------------------------------------------

CATEGORIAS_COICOP = [
    'alimentos_y_bebidas',
    'bebidas_alcoholicas_y_tabaco',
    'indumentaria_y_calzado',
    'propiedades_combustibles_agua_electricidad',
    'equipamiento_y_mantenimiento',
    'salud',
    'transporte',
    'comunicaciones',
    'esparcimiento_y_cultura',
    'ensenanza',
    'restaurantes_y_hoteles',
    'bienes_y_servicios_varios',
]

registros = []
for anio in range(2004, 2025):
    for cat in CATEGORIAS_COICOP:
        registros.append({
            'anio':               anio,
            'categoria_coicop':   cat,
            'alicuota_internos':  INTERNOS_BASE.get(cat, 0.0),
            'alicuota_iibb':      IIBB_POR_ANIO[anio],
        })

df_otros = pd.DataFrame(registros)
df_otros.to_csv(RUTAS['alicuotas'] / "internos_iibb_2004_2024.csv", index=False)

# ------------------------------------------------------------------
# D. Documentar exclusión de retenciones
# ------------------------------------------------------------------

RETENCIONES_NOTA = """
RETENCIONES AGROPECUARIAS — EXCLUIDAS DEL MODELO

Las retenciones a las exportaciones agropecuarias no se incorporan
al total tax rate porque su incidencia NO opera como impuesto sobre
el consumidor. Su mecanismo es el inverso:

  - Para el CONSUMIDOR: subsidio implícito. Al gravar la exportación,
    el precio doméstico de los commodities (trigo, soja, maíz) se
    desacopla del precio internacional y baja. El consumidor paga
    menos por alimentos básicos de lo que pagaría sin las retenciones.
    Este efecto beneficia proporcionalmente más a los deciles bajos
    (mayor fracción del ingreso destinada a alimentos).

  - Para el PRODUCTOR: el impuesto real recae sobre la renta
    agropecuaria. Los dueños de tierra y capital agropecuario
    (concentrados en deciles 9-10) reciben un precio menor por
    su producción.

Incluirlas como impuesto sobre el consumo (alícuota positiva en
alimentos) produciría el resultado incorrecto: el modelo indicaría
que castigan a los pobres, cuando la literatura muestra lo opuesto.

Para modelizarlas correctamente se requiere:
  1. Serie de precios de paridad FOB × tipo de cambio (SENASA/FAO)
     para construir el precio contrafactual sin retenciones.
  2. Distribución de la renta agropecuaria por decil (datos AFIP).

Ver: Lustig et al. — CEQ Assessment Argentina
     Gaggero & Rossignolo — Progresividad del sistema tributario argentino
"""

with open(RUTAS['alicuotas'] / "retenciones_metodologia.txt", 'w', encoding='utf-8') as f:
    f.write(RETENCIONES_NOTA)

# ------------------------------------------------------------------
# E. Verificación
# ------------------------------------------------------------------

print("✅ Sección 11 completa.")
print(f"   → internos_iibb_2004_2024.csv: {len(df_otros):,} filas")
print(f"   → retenciones_metodologia.txt: documentación de exclusión")
print()

# Spot check: mostrar la carga total sobre consumo para 2 años y 3 categorías
check_anios  = [2004, 2016, 2024]
check_cats   = ['alimentos_y_bebidas', 'bebidas_alcoholicas_y_tabaco', 'transporte']
check = df_otros[
    df_otros['anio'].isin(check_anios) &
    df_otros['categoria_coicop'].isin(check_cats)
].copy()
check['total_sobre_consumo'] = check['alicuota_internos'] + check['alicuota_iibb']

print("Spot check — carga de Internos + IIBB sobre precio al consumidor:")
print(check[['anio','categoria_coicop','alicuota_internos','alicuota_iibb','total_sobre_consumo']].to_string(index=False))
print()
print("Nota: IVA está en tabla separada (iva_alicuotas_2004_2024.csv).")
print("      La carga total sobre consumo en Fase 3 = IVA + Internos + IIBB.")

✅ Sección 11 completa.
   → internos_iibb_2004_2024.csv: 252 filas
   → retenciones_metodologia.txt: documentación de exclusión

Spot check — carga de Internos + IIBB sobre precio al consumidor:
 anio             categoria_coicop  alicuota_internos  alicuota_iibb  total_sobre_consumo
 2004          alimentos_y_bebidas              0.005          0.035                0.040
 2004 bebidas_alcoholicas_y_tabaco              0.350          0.035                0.385
 2004                   transporte              0.018          0.035                0.053
 2016          alimentos_y_bebidas              0.005          0.035                0.040
 2016 bebidas_alcoholicas_y_tabaco              0.350          0.035                0.385
 2016                   transporte              0.018          0.035                0.053
 2024          alimentos_y_bebidas              0.005          0.038                0.043
 2024 bebidas_alcoholicas_y_tabaco              0.350          0.038                0

### Seccion 12 - Bienes Personales

In [29]:
# =============================================================================
# SECCIÓN 12 — BIENES PERSONALES: DICCIONARIO CORREGIDO (2004–2024)
# VERSIÓN FINAL — verificada contra fuentes primarias
#
# Fuentes:
#   - Ley 23.966 Título VI (texto original y modificaciones)
#   - Ley 26.317 (2007): MNI $305.000
#   - Ley 27.260 (2016): MNI a excedente, alícuotas planas 2016-2018
#   - Ley 27.541 (2019): escala progresiva, tope 1.25%
#   - Ley 27.667 (dic. 2021): agrega tramos 1.50% y 1.75%
#   - Ley 27.743 (jul. 2024): retroactiva a 2023, tope 1.50% en 2023,
#     1.25% en 2024, MNI indexado por IPC
#   - ARCA (tablas oficiales publicadas): tramos exactos 2023 y 2024
#     https://www.afip.gob.ar/gananciasYBienes/bienes-personales/
#                              conceptos-basicos/alicuotas.asp
#
# CLAVE METODOLÓGICA — dos regímenes distintos:
#   PRE-2016  → "mínimo EXENTO": si patrimonio > mínimo, el impuesto
#               aplica sobre la TOTALIDAD del patrimonio (no el excedente)
#   POST-2016 → "mínimo NO IMPONIBLE": el impuesto aplica solo sobre
#               el EXCEDENTE del mínimo (estándar internacional)
#   Bandera: 'tipo_minimo': 'exento' | 'no_imponible'
# ACLARACIÓN CRÍTICA SOBRE EL MECANISMO POST-2019:
#   A partir de 2019, la escala NO aplica sobre el excedente del MNI.
#   Aplica sobre el VALOR TOTAL de los bienes (igual que el régimen
#   pre-2016 de mínimo exento), pero con una diferencia:
#   el MNI funciona como umbral de entrada (si patrimonio <= MNI → exento),
#   y si supera, los tramos se miden desde $0 sobre el total.
#   Esto está explicitamente en el PDF: "Más de $0 a $3.000.000 → 0.50%"
#
#   La función de cálculo necesita una tercera modalidad:
#   'tipo_minimo': 'total_desde_cero'
#
# TRES REGÍMENES:
#   'exento'          (2004-2015): mínimo exento; si supera → impuesto
#                     sobre la TOTALIDAD con escala desde el primer peso
#   'flat'            (2016-2018): MNI real; impuesto flat sobre excedente
#   'total_desde_cero'(2019-2022): MNI como umbral; si supera → impuesto
#                     sobre la TOTALIDAD con escala desde $0
#   'no_imponible'    (2023-2024): MNI real; impuesto progresivo sobre
#                     el excedente (los tramos de ARCA son del excedente)
# =============================================================================

BP_PARAMS = {

    # ------------------------------------------------------------------
    # 1999–2006: Ley 25.239 — dos tramos
    # Mínimo EXENTO $102.300
    # Escala: hasta $200k → 0.50% | más de $200k → 0.75%
    # CORRECCIÓN: antes tenía tres tramos incorrectos
    # ------------------------------------------------------------------
    2004: {
        'tipo_minimo': 'exento',
        'mni': 102_300,
        'vivienda_exenta': True,
        'tramos': [
            (200_000,       0.0050),
            (float('inf'), 0.0075),
        ]
    },
    2005: {
        'tipo_minimo': 'exento',
        'mni': 102_300,
        'vivienda_exenta': True,
        'tramos': [
            (200_000,       0.0050),
            (float('inf'), 0.0075),
        ]
    },
    2006: {
        'tipo_minimo': 'exento',
        'mni': 102_300,
        'vivienda_exenta': True,
        'tramos': [
            (200_000,       0.0050),
            (float('inf'), 0.0075),
        ]
    },

    # ------------------------------------------------------------------
    # 2007–2015: Ley 26.317 — cuatro tramos
    # Mínimo EXENTO $305.000
    # Escala correcta verificada en PDF oficial
    # ------------------------------------------------------------------
    2007: {
        'tipo_minimo': 'exento',
        'mni': 305_000,
        'vivienda_exenta': True,
        'tramos': [
            (750_000,       0.0050),
            (2_000_000,    0.0075),
            (5_000_000,    0.0100),
            (float('inf'), 0.0125),
        ]
    },
    2008: {
        'tipo_minimo': 'exento',
        'mni': 305_000,
        'vivienda_exenta': True,
        'tramos': [
            (750_000,       0.0050),
            (2_000_000,    0.0075),
            (5_000_000,    0.0100),
            (float('inf'), 0.0125),
        ]
    },
    2009: {
        'tipo_minimo': 'exento',
        'mni': 305_000,
        'vivienda_exenta': True,
        'tramos': [
            (750_000,       0.0050),
            (2_000_000,    0.0075),
            (5_000_000,    0.0100),
            (float('inf'), 0.0125),
        ]
    },
    2010: {
        'tipo_minimo': 'exento',
        'mni': 305_000,
        'vivienda_exenta': True,
        'tramos': [
            (750_000,       0.0050),
            (2_000_000,    0.0075),
            (5_000_000,    0.0100),
            (float('inf'), 0.0125),
        ]
    },
    2011: {
        'tipo_minimo': 'exento',
        'mni': 305_000,
        'vivienda_exenta': True,
        'tramos': [
            (750_000,       0.0050),
            (2_000_000,    0.0075),
            (5_000_000,    0.0100),
            (float('inf'), 0.0125),
        ]
    },
    2012: {
        'tipo_minimo': 'exento',
        'mni': 305_000,
        'vivienda_exenta': True,
        'tramos': [
            (750_000,       0.0050),
            (2_000_000,    0.0075),
            (5_000_000,    0.0100),
            (float('inf'), 0.0125),
        ]
    },
    2013: {
        'tipo_minimo': 'exento',
        'mni': 305_000,
        'vivienda_exenta': True,
        'tramos': [
            (750_000,       0.0050),
            (2_000_000,    0.0075),
            (5_000_000,    0.0100),
            (float('inf'), 0.0125),
        ]
    },
    2014: {
        'tipo_minimo': 'exento',
        'mni': 305_000,
        'vivienda_exenta': True,
        'tramos': [
            (750_000,       0.0050),
            (2_000_000,    0.0075),
            (5_000_000,    0.0100),
            (float('inf'), 0.0125),
        ]
    },
    2015: {
        'tipo_minimo': 'exento',
        'mni': 305_000,
        'vivienda_exenta': True,
        'tramos': [
            (750_000,       0.0050),
            (2_000_000,    0.0075),
            (5_000_000,    0.0100),
            (float('inf'), 0.0125),
        ]
    },

    # ------------------------------------------------------------------
    # 2016–2018: Ley 27.260 — alícuota FLAT, sin tramos
    # Cambia a MNI real (solo paga sobre excedente)
    # Un único tramo cubre todo el excedente
    # Fuente: PDF Hacienda + APROCAM (verificado)
    # ------------------------------------------------------------------
    2016: {
        'tipo_minimo': 'flat',
        'mni': 800_000,
        'vivienda_exenta': True,
        'tramos': [
            (float('inf'), 0.0075),
        ]
    },
    2017: {
        'tipo_minimo': 'flat',
        'mni': 950_000,
        'vivienda_exenta': True,
        'tramos': [
            (float('inf'), 0.0050),
        ]
    },
    2018: {
        'tipo_minimo': 'flat',
        'mni': 1_050_000,
        'vivienda_exenta': True,
        'tramos': [
            (float('inf'), 0.0025),
        ]
    },

    # ------------------------------------------------------------------
    # 2019–2022: Ley 27.541 + Ley 27.667
    # MECANISMO: los tramos se miden sobre el VALOR TOTAL de los bienes
    # (no sobre el excedente del MNI). El MNI es solo umbral de entrada.
    # Fuente: PDF Hacienda — "Más de $0 a $3.000.000 → 0.50%"
    #
    # 2019-2020: cuatro tramos, tope 1.25% (Ley 27.541)
    # 2021:      seis tramos, tope 1.75% (Ley 27.667, dic 2021)
    # 2022:      seis tramos, tope 1.75%, valores actualizados por IPC
    # ------------------------------------------------------------------
    2019: {
        'tipo_minimo': 'total_desde_cero',
        'mni': 2_000_000,
        'vivienda_exenta': True,
        'tramos': [
            (3_000_000,    0.0025),  # 0% a $2M (MNI) luego 0.25%
            (6_500_000,    0.0050),
            (18_000_000,   0.0075),
            (float('inf'), 0.0100),
        ]
    },
    2020: {
        'tipo_minimo': 'total_desde_cero',
        'mni': 2_000_000,
        'vivienda_exenta': True,
        'tramos': [
            (3_000_000,    0.0050),
            (6_500_000,    0.0075),
            (18_000_000,   0.0100),
            (float('inf'), 0.0125),
        ]
    },
    2021: {
        'tipo_minimo': 'total_desde_cero',
        'mni': 6_000_000,
        'vivienda_exenta': True,
        'tramos': [
            (3_000_000,    0.0050),
            (6_500_000,    0.0075),
            (18_000_000,   0.0100),
            (100_000_000,  0.0125),
            (300_000_000,  0.0150),
            (float('inf'), 0.0175),
        ]
    },
    2022: {
        'tipo_minimo': 'total_desde_cero',
        'mni': 11_282_141,
        'vivienda_exenta': True,
        'tramos': [
            # Valores exactos del PDF oficial Hacienda
            (5_641_071,    0.0050),
            (12_222_320,   0.0075),
            (33_846_423,   0.0100),
            (188_035_685,  0.0125),
            (564_107_054,  0.0150),
            (float('inf'), 0.0175),
        ]
    },

    # ------------------------------------------------------------------
    # 2023–2024: Ley 27.743 — vuelve a escala sobre el EXCEDENTE del MNI
    # Los tramos de ARCA están expresados como excedente del MNI.
    # Fuente: tablas oficiales ARCA (verificadas)
    # ------------------------------------------------------------------
    2023: {
        'tipo_minimo': 'no_imponible',
        'mni': 100_000_000,
        'vivienda_exenta': True,
        'tramos': [
            (13_688_704,   0.0050),
            (29_658_859,   0.0075),
            (82_132_225,   0.0100),
            (456_290_138,  0.0125),
            (float('inf'), 0.0150),
        ]
    },
    2024: {
        'tipo_minimo': 'no_imponible',
        'mni': 292_994_965,
        'vivienda_exenta': True,
        'tramos': [
            (40_107_214,   0.0050),
            (86_898_963,   0.0075),
            (240_643_283,  0.0100),
            (float('inf'), 0.0125),
        ]
    },
}

# Verificación rápida de integridad
assert len(BP_PARAMS) == 21, f"Faltan años: {set(range(2004,2025)) - set(BP_PARAMS)}"
tipos = {a: p['tipo_minimo'] for a, p in BP_PARAMS.items()}
print(f"✓ BP_PARAMS cargado: {len(BP_PARAMS)} años")
print(f"\nRégimen por período:")
print(f"  2004–2015: mínimo EXENTO (impuesto sobre totalidad si supera)")
print(f"  2016–2024: mínimo NO IMPONIBLE (impuesto solo sobre excedente)")
print(f"\nCambios de alícuota máxima:")
for anio, p in sorted(BP_PARAMS.items()):
    tope = p['tramos'][-1][1]
    mni  = p['mni']
    tipo = "EXENTO" if p['tipo_minimo'] == 'exento' else "MNI"
    print(f"  {anio}: tope {tope:.2%} | {tipo} ${mni:>15,.0f}")

✓ BP_PARAMS cargado: 21 años

Régimen por período:
  2004–2015: mínimo EXENTO (impuesto sobre totalidad si supera)
  2016–2024: mínimo NO IMPONIBLE (impuesto solo sobre excedente)

Cambios de alícuota máxima:
  2004: tope 0.75% | EXENTO $        102,300
  2005: tope 0.75% | EXENTO $        102,300
  2006: tope 0.75% | EXENTO $        102,300
  2007: tope 1.25% | EXENTO $        305,000
  2008: tope 1.25% | EXENTO $        305,000
  2009: tope 1.25% | EXENTO $        305,000
  2010: tope 1.25% | EXENTO $        305,000
  2011: tope 1.25% | EXENTO $        305,000
  2012: tope 1.25% | EXENTO $        305,000
  2013: tope 1.25% | EXENTO $        305,000
  2014: tope 1.25% | EXENTO $        305,000
  2015: tope 1.25% | EXENTO $        305,000
  2016: tope 0.75% | MNI $        800,000
  2017: tope 0.50% | MNI $        950,000
  2018: tope 0.25% | MNI $      1,050,000
  2019: tope 1.00% | MNI $      2,000,000
  2020: tope 1.25% | MNI $      2,000,000
  2021: tope 1.75% | MNI $      6,000,000

In [ ]:
# -----------------------------------------------------------------------------
# PARTE B — DISTRIBUCIÓN POR DECIL
# -----------------------------------------------------------------------------
# La EPH no captura patrimonio. Usamos la distribución de la recaudación
# de BP por tramo de patrimonio publicada por AFIP (Estadísticas Tributarias
# Cuadro 22: "Impuesto sobre los Bienes Personales — declarantes por tramo")
# y la mapeamos a deciles de ingreso EPH.
#
# MAPEO DECIL ↔ TRAMO PATRIMONIO:
#   Basado en Rossignolo (2014) y CEQ Argentina (Lustig et al. 2017).
#   El supuesto clave es que la distribución del patrimonio sigue
#   aproximadamente la del ingreso, con mayor concentración en la cima.
#
# DISTRIBUCIÓN DE LA RECAUDACIÓN POR DECIL (participación %):
#   Estas participaciones se derivan de los datos públicos de AFIP sobre
#   declarantes por tramo. Son estables en el tiempo (la distribución del
#   patrimonio cambia lentamente). Se usa una tabla única con ajuste leve
#   para los períodos de mayor erosión del MNI (2008–2015).
#
# FUENTE DIRECTA VERIFICABLE:
#   AFIP — "Estadísticas Tributarias" (publicación anual)
#   Cuadro: "Impuesto sobre los Bienes Personales — por tramo de base imponible"
#   URL: https://www.afip.gob.ar/institucional/estudios/
#
# Participaciones aproximadas (validadas contra Rossignolo 2014 y CEQ 2017):
#   D1–D7:   0%   (patrimonio neto < MNI, o vivienda única exenta)
#   D8:      3%   (algunos propietarios con inmuebles > MNI)
#   D9:     12%   (propietarios con 2+ inmuebles, pequeños inversores)
#   D10:    85%   (principales contribuyentes, activos financieros, exterior)
#
# NOTA sobre el período 2008–2015 (MNI congelado):
#   La erosión inflacionaria del MNI ($305.000 sin actualizar) incorporó
#   gradualmente a contribuyentes del D7–D8. Se ajusta la participación:
#   D7: 1%, D8: 5%, D9: 14%, D10: 80%.
# -----------------------------------------------------------------------------

### VERIFICAR MANUALMENTE!

DIST_DECILES_BP = {
    # Período normal (MNI actualizado o alto relativo al patrimonio mediano)
    'normal': {
        1: 0.00, 2: 0.00, 3: 0.00, 4: 0.00, 5: 0.00,
        6: 0.00, 7: 0.00, 8: 0.03, 9: 0.12, 10: 0.85,
    },
    # Período de MNI congelado (2008-2015): erosión inflacionaria
    # incorporó propietarios del D7-D8 al padrón de contribuyentes
    'mni_congelado': {
        1: 0.00, 2: 0.00, 3: 0.00, 4: 0.00, 5: 0.00,
        6: 0.00, 7: 0.01, 8: 0.05, 9: 0.14, 10: 0.80,
    },
}

def get_dist_bp(anio):
    """Devuelve la distribución por decil correspondiente al año."""
    if 2008 <= anio <= 2015:
        return DIST_DECILES_BP['mni_congelado']
    return DIST_DECILES_BP['normal']

print("✓ Distribuciones por decil cargadas.")
print()
print("Verificación de sumas (deben ser 1.0):")
for nombre, dist in DIST_DECILES_BP.items():
    total = sum(dist.values())
    estado = "✓" if abs(total - 1.0) < 0.001 else "❌ ERROR"
    print(f"  {nombre}: {total:.3f} {estado}")

✓ Distribuciones por decil cargadas.

Verificación de sumas (deben ser 1.0):
  normal: 1.000 ✓
  mni_congelado: 1.000 ✓


In [ ]:
# -----------------------------------------------------------------------------
# PARTE C — RECAUDACIÓN OBSERVADA DE BP POR AÑO (para escalar la imputación)
# -----------------------------------------------------------------------------
# Fuente: ARCA — Series de Recaudación Tributaria
# URL: https://www.afip.gob.ar/institucional/estudios/archivos/recaudacion/
# Unidad: MILLONES de pesos nominales corrientes
#
# IMPORTANTE: estos datos son la recaudación TOTAL del impuesto en cada año.
# La distribución por decil de la Parte B se aplica sobre esta recaudación
# para obtener cuánto pagó (en términos absolutos) cada decil.
# En la Fase 3, ese monto se divide por el ingreso total del decil
# (suma de IPCF × personas representadas × 12 meses) para obtener la tasa.
#
# Fuente verificación: ARCA publica mensualmente en:
#   https://www.afip.gob.ar/institucional/estudios/archivos/recaudacion/
# Los datos anuales son suma de los 12 meses de cada año fiscal.
#
# NOTA: 2007–2015 usa datos AFIP directos (no INDEC).
#   La recaudación de BP no fue alterada por la intervención del INDEC.
# -----------------------------------------------------------------------------

### VERIFICAR MANUALMENTE!

RECAUDACION_BP_MILLONES = {
    2004:      474,
    2005:      618,
    2006:      757,
    2007:    1_095,
    2008:    1_687,
    2009:    2_185,
    2010:    3_032,
    2011:    4_408,
    2012:    5_879,
    2013:    7_650,
    2014:   10_789,
    2015:   14_522,
    2016:   20_817,
    2017:   31_058,
    2018:   45_124,
    2019:   84_390,
    2020:  131_447,
    2021:  196_528,
    2022:  468_133,
    2023: 1_452_000,
    2024: 5_820_000,  # Estimación provisoria — verificar con ARCA al cierre    TODOS ESTOS DATOS HAY QUE CHEQUEARLOS
}

# Verificación rápida: la recaudación debería crecer todos los años (inflación)
print(f"{'Año':<6} {'Recaudación (M$)':>18}  {'Var. %':>8}")
print("-" * 38)
prev = None
for anio in sorted(RECAUDACION_BP_MILLONES):
    rec = RECAUDACION_BP_MILLONES[anio]
    if prev is not None:
        var = (rec / prev - 1) * 100
        flag = " ⚠️ REVISAR" if var < 0 else ""
        print(f"{anio:<6} {rec:>18,.0f}  {var:>7.1f}%{flag}")
    else:
        print(f"{anio:<6} {rec:>18,.0f}  {'—':>8}")
    prev = rec

Año      Recaudación (M$)    Var. %
--------------------------------------
2004                  474         —
2005                  618     30.4%
2006                  757     22.5%
2007                1,095     44.6%
2008                1,687     54.1%
2009                2,185     29.5%
2010                3,032     38.8%
2011                4,408     45.4%
2012                5,879     33.4%
2013                7,650     30.1%
2014               10,789     41.0%
2015               14,522     34.6%
2016               20,817     43.3%
2017               31,058     49.2%
2018               45,124     45.3%
2019               84,390     87.0%
2020              131,447     55.8%
2021              196,528     49.5%
2022              468,133    138.2%
2023            1,452,000    210.2%
2024            5,820,000    300.8%


In [32]:
# -----------------------------------------------------------------------------
# PARTE D — CONSTRUCCIÓN DE LA TABLA FINAL: BP POR DECIL Y AÑO
# -----------------------------------------------------------------------------
# Output: bp_incidencia_2004_2024.csv
#
# Columnas:
#   anio | decil | recaudacion_total_M | participacion_decil | impuesto_decil_M
#
# Este archivo es el insumo directo de la Fase 3.
# La tasa efectiva (impuesto / ingreso del decil) se calcula en Fase 3
# donde tenemos el ingreso total del decil desde la EPH.
# -----------------------------------------------------------------------------
# PARTE D — CONSTRUCCIÓN DE LA TABLA FINAL: BP POR DECIL Y AÑO
# Corregido: claves JSON como str, campo bienes_exterior con .get()
# -----------------------------------------------------------------------------

registros_bp = []
for anio in range(2004, 2025):
    rec_total = RECAUDACION_BP_MILLONES[anio]
    dist      = get_dist_bp(anio)
    params    = BP_PARAMS[anio]

    for decil, participacion in dist.items():
        registros_bp.append({
            'anio':                anio,
            'decil':               decil,
            'recaudacion_total_M': rec_total,
            'participacion_decil': participacion,
            'impuesto_decil_M':    rec_total * participacion,
            'mni':                 params['mni'],
            'tipo_minimo':         params['tipo_minimo'],
            'vivienda_exenta':     params['vivienda_exenta'],
        })

df_bp = pd.DataFrame(registros_bp)
df_bp.to_csv(RUTAS['alicuotas'] / "bp_incidencia_2004_2024.csv", index=False)

# JSON: claves como str (requerimiento de la especificación JSON)
bp_json = {}
for anio, p in BP_PARAMS.items():
    bp_json[str(anio)] = {                          # Bug 1 corregido: str(anio)
        'tipo_minimo':              p['tipo_minimo'],
        'mni':                      p['mni'],
        'vivienda_exenta':          p['vivienda_exenta'],
        'bienes_exterior_agravado': p.get('bienes_exterior_agravado', False),  # Bug 2 corregido
        'tramos': [
            [lim if lim != float('inf') else 'inf', alic]
            for lim, alic in p['tramos']
        ]
    }

with open(RUTAS['alicuotas'] / "bp_params_2004_2024.json", 'w', encoding='utf-8') as f:
    json.dump(bp_json, f, indent=2, ensure_ascii=False)

print(f"✅ Bienes Personales guardado:")
print(f"   → bp_incidencia_2004_2024.csv ({len(df_bp)} filas)")
print(f"   → bp_params_2004_2024.json (parámetros formales)")
print()

# Verificación de integridad del CSV
assert df_bp['participacion_decil'].groupby(df_bp['anio']).sum().round(10).eq(1.0).all(), \
    "❌ Las participaciones no suman 1.0 para algún año"
print("✓ Integridad verificada: participaciones suman 1.0 en cada año")
print()

# Tabla resumen
print(f"{'Año':<6} {'Régimen':<20} {'MNI':>16} {'Rec. M$':>12}  "
      f"{'D8':>5} {'D9':>5} {'D10':>6}")
print("-" * 75)
for anio in range(2004, 2025):
    dist   = get_dist_bp(anio)
    rec    = RECAUDACION_BP_MILLONES[anio]
    params = BP_PARAMS[anio]
    print(f"{anio:<6} {params['tipo_minimo']:<20} ${params['mni']:>14,.0f} "
          f"${rec:>10,.0f}  "
          f"{dist[8]:>4.0%} {dist[9]:>4.0%} {dist[10]:>5.0%}")

✅ Bienes Personales guardado:
   → bp_incidencia_2004_2024.csv (210 filas)
   → bp_params_2004_2024.json (parámetros formales)

✓ Integridad verificada: participaciones suman 1.0 en cada año

Año    Régimen                           MNI      Rec. M$     D8    D9    D10
---------------------------------------------------------------------------
2004   exento               $       102,300 $       474    3%  12%   85%
2005   exento               $       102,300 $       618    3%  12%   85%
2006   exento               $       102,300 $       757    3%  12%   85%
2007   exento               $       305,000 $     1,095    3%  12%   85%
2008   exento               $       305,000 $     1,687    5%  14%   80%
2009   exento               $       305,000 $     2,185    5%  14%   80%
2010   exento               $       305,000 $     3,032    5%  14%   80%
2011   exento               $       305,000 $     4,408    5%  14%   80%
2012   exento               $       305,000 $     5,879    5%  14%   8

In [ ]:
# -----------------------------------------------------------------------------
# PARTE E — FUNCIÓN DE VALIDACIÓN
# Calcula la alícuota efectiva formal para un patrimonio dado en un año dado.
# Útil para verificar que los tramos estén bien cargados comparando con
# la calculadora de AFIP o con casos concretos conocidos.
# -----------------------------------------------------------------------------

def calcular_alicuota_efectiva_bp(patrimonio_neto, anio):
    """
    Cuatro regímenes históricos de BP:

    'exento'           (2004-2015): si patrimonio <= mni → 0; si supera →
                       impuesto sobre TOTALIDAD con escala desde el primer $
    'flat'             (2016-2018): impuesto solo sobre excedente, alícuota única
    'total_desde_cero' (2019-2022): si patrimonio <= mni → 0; si supera →
                       impuesto sobre TOTALIDAD con escala progresiva desde $0
                       (mecánicamente igual que 'exento' pero con más tramos)
    'no_imponible'     (2023-2024): impuesto solo sobre excedente del MNI,
                       con escala progresiva sobre ese excedente
    """
    p    = BP_PARAMS[anio]
    mni  = p['mni']
    tipo = p['tipo_minimo']

    def aplicar_escala_sobre(base):
        """Aplica la escala progresiva sobre un monto base."""
        impuesto   = 0.0
        tramo_prev = 0
        for limite, alicuota in p['tramos']:
            if base <= tramo_prev:
                break
            tope       = min(base, limite)
            impuesto  += (tope - tramo_prev) * alicuota
            tramo_prev = limite
            if limite == float('inf'):
                break
        return impuesto

    # --- Régimen 'exento' (2004-2015) ---
    if tipo == 'exento':
        if patrimonio_neto <= mni:
            return {'impuesto': 0, 'alicuota_efectiva': 0.0,
                    'base_imponible': 0, 'razon': 'bajo_minimo_exento'}
        base     = patrimonio_neto          # escala sobre la totalidad
        impuesto = aplicar_escala_sobre(base)

    # --- Régimen 'flat' (2016-2018) ---
    elif tipo == 'flat':
        if patrimonio_neto <= mni:
            return {'impuesto': 0, 'alicuota_efectiva': 0.0,
                    'base_imponible': 0, 'razon': 'bajo_mni'}
        base     = patrimonio_neto - mni    # solo sobre el excedente
        impuesto = aplicar_escala_sobre(base)

    # --- Régimen 'total_desde_cero' (2019-2022) ---
    elif tipo == 'total_desde_cero':
        if patrimonio_neto <= mni:
            return {'impuesto': 0, 'alicuota_efectiva': 0.0,
                    'base_imponible': 0, 'razon': 'bajo_mni'}
        base     = patrimonio_neto          # escala sobre la totalidad
        impuesto = aplicar_escala_sobre(base)

    # --- Régimen 'no_imponible' (2023-2024) ---
    elif tipo == 'no_imponible':
        if patrimonio_neto <= mni:
            return {'impuesto': 0, 'alicuota_efectiva': 0.0,
                    'base_imponible': 0, 'razon': 'bajo_mni'}
        base     = patrimonio_neto - mni    # solo sobre el excedente
        impuesto = aplicar_escala_sobre(base)

    else:
        raise ValueError(f"tipo_minimo desconocido: {tipo}")

    alicuota_ef = impuesto / patrimonio_neto if patrimonio_neto > 0 else 0.0
    return {
        'impuesto':          round(impuesto, 2),
        'alicuota_efectiva': round(alicuota_ef, 5),
        'base_imponible':    round(base, 2),
        'razon':             'calculado',
    }


# --- Validaciones con casos ilustrativos ---
print(f"{'Año':<6} {'Régimen':<20} {'Patrimonio':>16}  {'Alíc. ef.':>10}  {'Impuesto':>14}  Caso")
print("-" * 100)

### VERIFICAR MANUALMENTE!

casos_bp = [
    # Pre-2007: escala de dos tramos Ley 25.239
    (2005,    102_000, "Bajo mínimo exento $102.3k → 0%"),
    (2005,    150_000, "Sobre mínimo, primer tramo 0.50%"),
    (2005,    250_000, "Sobre $200k, segundo tramo 0.75%"),
    # 2007-2015: cuatro tramos
    (2010,    304_999, "Justo bajo $305k → exento"),
    (2010,    305_001, "Un peso arriba → paga sobre TODO"),
    (2010,  3_000_000, "Tercer tramo 1.00%"),
    (2015, 10_000_000, "Tramo máximo 1.25%"),
    # Transición a flat
    (2016,    799_999, "Bajo MNI $800k → exento"),
    (2016,  2_000_000, "Flat 0.75% sobre excedente $1.2M"),
    (2017,  2_000_000, "Flat 0.50%"),
    (2018,  2_000_000, "Flat 0.25%"),
    # 2019-2022: total desde cero
    (2019,  1_999_999, "Bajo MNI $2M → exento"),
    (2019,  5_000_000, "Sobre MNI, escala sobre el total"),
    (2020, 20_000_000, "Tope 1.25% (no 1.75%)"),
    (2021,150_000_000, "Tramo 1.50% (Ley 27.667)"),
    (2021,400_000_000, "Tope 1.75%"),
    (2022,200_000_000, "Tramo 1.50%, valores actualizados"),
    # 2023-2024: excedente del MNI
    (2023, 99_999_999, "Bajo MNI $100M → exento"),
    (2023,200_000_000, "Excedente $100M, tramo 1.25%"),
    (2023,500_000_000, "Tope 1.50%"),
    (2024,292_000_000, "Bajo MNI $293M → exento"),
    (2024,500_000_000, "Excedente, tramo 1.00%-1.25%"),
    (2024,  1_000_000_000, "Tope 1.25%"),
]

for anio, pat, desc in casos_bp:
    r     = calcular_alicuota_efectiva_bp(pat, anio)
    regim = BP_PARAMS[anio]['tipo_minimo']
    print(f"{anio:<6} {regim:<20} ${pat:>14,.0f}  "
          f"{r['alicuota_efectiva']:>9.3%}  ${r['impuesto']:>12,.0f}  {desc}")

Año    Régimen                    Patrimonio   Alíc. ef.        Impuesto  Caso
----------------------------------------------------------------------------------------------------
2005   exento               $       102,000     0.000%  $           0  Bajo mínimo exento $102.3k → 0%
2005   exento               $       150,000     0.500%  $         750  Sobre mínimo, primer tramo 0.50%
2005   exento               $       250,000     0.550%  $       1,375  Sobre $200k, segundo tramo 0.75%
2010   exento               $       304,999     0.000%  $           0  Justo bajo $305k → exento
2010   exento               $       305,001     0.500%  $       1,525  Un peso arriba → paga sobre TODO
2010   exento               $     3,000,000     0.771%  $      23,125  Tercer tramo 1.00%
2015   exento               $    10,000,000     1.056%  $     105,625  Tramo máximo 1.25%
2016   flat                 $       799,999     0.000%  $           0  Bajo MNI $800k → exento
2016   flat                 $    

In [ ]:
# =============================================================================
# EFECTO DE LICUACIÓN INFLACIONARIA — Análisis complementario
#
# Bienes Personales se devenga el 31/12 del año N pero se paga en junio
# del año N+1 (con variaciones históricas). En un contexto inflacionario,
# el valor real del impuesto se licúa durante ese rezago.
#
# DECISIÓN METODOLÓGICA (documentar en README):
#   El total tax rate usa criterio DEVENGADO (año N), consistente con
#   toda la literatura de incidencia fiscal (CEQ, CEPAL, Rossignolo).
#   Este análisis complementario cuantifica el "descuento real" que
#   obtiene el contribuyente por el rezago de pago.
#
# FÓRMULA:
#   impuesto_real_al_pago = impuesto_nominal / (1 + inflacion_rezago)
#   licuacion_pct = (impuesto_nominal - impuesto_real) / impuesto_nominal
#                = inflacion_rezago / (1 + inflacion_rezago)
#
# FUENTE INFLACIÓN: IPC INDEC (con corrección por años intervenidos).
#   2007-2015: se usa IPC-Congreso (promedio de estimaciones privadas).
#   El rezago estándar es de 6 meses (dic → jun siguiente).
# =============================================================================

import pandas as pd
import numpy as np

# Inflación acumulada en el período de rezago (31/12/año N → ~30/06/año N+1)
# Aproximación: mitad de la inflación anual del año N+1
# Para años de INDEC intervenido (2007-2015): se usa estimación privada
# Fuentes: INDEC para años confiables; IPC-Congreso para 2007-2015

### VERIFICAR MANUALMENTE!

INFLACION_REZAGO_SEMESTRAL = {
    # año N: inflación acumulada dic(N) a jun(N+1), aprox.
    # = inflación interanual del año N+1 / 2 (proxy)
    2004: 0.045,   # IPC INDEC: inf. 2005 ~9.1% → semestre ~4.5%
    2005: 0.055,   # IPC INDEC: inf. 2006 ~10.9% → ~5.5%
    2006: 0.045,   # IPC INDEC: inf. 2007 ~8.8% → ~4.5%
    2007: 0.115,   # IPC Congreso: inf. 2008 ~23% → ~11.5%
    2008: 0.075,   # IPC Congreso: inf. 2009 ~15% → ~7.5%
    2009: 0.130,   # IPC Congreso: inf. 2010 ~26% → ~13%
    2010: 0.125,   # IPC Congreso: inf. 2011 ~25% → ~12.5%
    2011: 0.120,   # IPC Congreso: inf. 2012 ~24% → ~12%
    2012: 0.130,   # IPC Congreso: inf. 2013 ~26% → ~13%
    2013: 0.175,   # IPC Congreso: inf. 2014 ~35% → ~17.5%
    2014: 0.150,   # IPC Congreso: inf. 2015 ~30% → ~15%
    2015: 0.200,   # IPC Congreso: inf. 2016 ~40% → ~20%
    2016: 0.125,   # IPC INDEC (nueva metodología): inf. 2017 ~25% → ~12.5%
    2017: 0.190,   # IPC INDEC: inf. 2018 ~47.6% → ~19%  (aceleración)
    2018: 0.270,   # IPC INDEC: inf. 2019 ~53.8% → ~27%
    2019: 0.185,   # IPC INDEC: inf. 2020 ~36.1% → ~18.5%
    2020: 0.240,   # IPC INDEC: inf. 2021 ~50.9% → ~24%
    2021: 0.290,   # IPC INDEC: inf. 2022 ~94.8% → ~29% (aceleración H1)
    2022: 0.490,   # IPC INDEC: inf. 2023 ~211% → ene-jun 2023 ~49%
    2023: 0.630,   # IPC INDEC: inf. acum. dic23-jun24 ~63% (hiperinflación)
    # 2024 no aplica (aún no se devengó el pago en el período analizado)
}

# Calcular el descuento real para cada año
registros_licuacion = []
for anio, inf_semestral in sorted(INFLACION_REZAGO_SEMESTRAL.items()):
    licuacion_pct = inf_semestral / (1 + inf_semestral)
    registros_licuacion.append({
        'anio':                anio,
        'inflacion_rezago':    inf_semestral,
        'licuacion_pct':       licuacion_pct,
        # Ejemplo concreto: impuesto devengado hipotético de $1.000 nominales
        'valor_real_de_1000':  round(1000 / (1 + inf_semestral), 1),
    })

df_licuacion = pd.DataFrame(registros_licuacion)
df_licuacion.to_csv(
    RUTAS['alicuotas'] / "bp_licuacion_inflacionaria_2004_2023.csv",
    index=False
)

# Tabla de resultados
print("=" * 70)
print("EFECTO DE LICUACIÓN INFLACIONARIA — BIENES PERSONALES")
print("Criterio del modelo: DEVENGADO (año N)")
print("Este análisis es complementario — no modifica el total tax rate")
print("=" * 70)
print()
print(f"{'Año':<6} {'Inf. rezago':>12}  {'Licuación':>11}  "
      f"{'$1.000 devengados':>19}  {'valen en jun N+1':>17}")
print("-" * 70)

for _, row in df_licuacion.iterrows():
    anio   = int(row['anio'])
    inf    = row['inflacion_rezago']
    licu   = row['licuacion_pct']
    valor  = row['valor_real_de_1000']
    # Resaltar años de licuación extrema (>25%)
    flag   = " ◄ ALTA" if licu > 0.25 else ""
    print(f"{anio:<6} {inf:>11.1%}  {licu:>10.1%}  "
          f"{'$1.000':>19}  ${valor:>14.1f}{flag}")

print()
print("Interpretación:")
print("  'Licuación %' = fracción del impuesto nominal que se evaporó")
print("  en términos reales por el rezago entre devengamiento y pago.")
print()
print("Años de mayor licuación (>25% del impuesto): "
      f"{[int(r['anio']) for _, r in df_licuacion.iterrows() if r['licuacion_pct'] > 0.25]}")
print()
print("Nota metodológica para el paper:")
print("  El modelo usa criterio devengado, consistente con CEQ/CEPAL.")
print("  La licuación aquí calculada representa el 'beneficio fiscal'")
print("  implícito que obtuvieron los contribuyentes de BP por el rezago")
print("  de pago en contextos de alta inflación. Este efecto es regresivo")
print("  en sentido inverso: beneficia exclusivamente a los deciles 9-10,")
print("  únicos pagadores del impuesto, y es creciente con el nivel")
print("  patrimonial absoluto (mayor impuesto nominal = mayor beneficio).")

EFECTO DE LICUACIÓN INFLACIONARIA — BIENES PERSONALES
Criterio del modelo: DEVENGADO (año N)
Este análisis es complementario — no modifica el total tax rate

Año     Inf. rezago    Licuación    $1.000 devengados   valen en jun N+1
----------------------------------------------------------------------
2004          4.5%        4.3%               $1.000  $         956.9
2005          5.5%        5.2%               $1.000  $         947.9
2006          4.5%        4.3%               $1.000  $         956.9
2007         11.5%       10.3%               $1.000  $         896.9
2008          7.5%        7.0%               $1.000  $         930.2
2009         13.0%       11.5%               $1.000  $         885.0
2010         12.5%       11.1%               $1.000  $         888.9
2011         12.0%       10.7%               $1.000  $         892.9
2012         13.0%       11.5%               $1.000  $         885.0
2013         17.5%       14.9%               $1.000  $         851.1
2014    

### Sección 13

In [ ]:
# =============================================================================
# SECCIÓN 13 — SERIES DE RECAUDACIÓN ARCA POR IMPUESTO (2004–2024)
# =============================================================================
# Propósito: construir la tabla de recaudación total observada por impuesto
# y año, que se usará en la Fase 5 para VALIDAR que la incidencia estimada
# por el modelo es consistente con la recaudación real agregada.
#
# LÓGICA DE VALIDACIÓN (Fase 5):
#   Para cada impuesto X y año Y:
#     incidencia_estimada(X, Y) = Σ deciles [tasa_decil(X,Y) × ingreso_decil(Y)]
#   Ese número debe aproximarse a recaudacion_observada(X, Y).
#   Si la divergencia supera el ±15%, revisar los supuestos de incidencia.
#
# FUENTES PRIMARIAS:
#   - ARCA (ex-AFIP): "Recaudación tributaria" — series mensuales publicadas en:
#     https://www.afip.gob.ar/institucional/estudios/archivos/recaudacion/
#   - Los datos anuales son SUMA de los 12 meses del año calendario.
#   - Para años anteriores a 2008, algunos datos provienen del Informe Anual
#     de Recaudación (PDFs históricos disponibles en el mismo sitio).
#
# UNIDAD: millones de pesos corrientes (nominales).
#
# COBERTURA:
#   - IVA: recaudación total (incluye IVA DGI + IVA aduana)
#   - Ganancias: total (personas físicas + sociedades). Se incluye el total
#     porque la EPH no permite separar la parte de personas físicas con
#     exactitud; la validación es contra el total y se documenta la diferencia.
#   - Aportes personales SIPA: recaudación de aportes de trabajadores
#     (no incluye contribuciones patronales).
#   - Contribuciones patronales SIPA: por separado, para validar
#     independientemente del supuesto de incidencia 50/50.
#   - Impuestos Internos: total (tabaco + bebidas + combustibles + otros).
#   - Bienes Personales: total (ya cargado en Sección 12, se replica aquí
#     por consistencia de la tabla de validación).
#   - IIBB: NO se incluye aquí. La recaudación de IIBB es provincial,
#     fragmentada en 24 jurisdicciones, y no está consolidada en ARCA.
#     La validación de IIBB se hace por estimación de presión tributaria
#     provincial total (ver nota al pie de esta sección).
#
# NOTA SOBRE GANANCIAS PERSONAS FÍSICAS VS. SOCIEDADES:
#   La ARCA publica Ganancias como total unificado. La participación de
#   personas físicas varía históricamente entre el 20% y el 35% del total,
#   según datos de declaraciones juradas (Estadísticas Tributarias AFIP,
#   Cuadros 5 y 6). Se carga el total aquí y se corrige en Fase 5 con
#   el coeficiente de participación de personas físicas por año.
# =============================================================================

import pandas as pd
import numpy as np

# -----------------------------------------------------------------------------
# A. TABLA DE RECAUDACIÓN OBSERVADA (millones de $ nominales)
# -----------------------------------------------------------------------------
# Fuente: ARCA — series mensuales sumadas por año calendario.
# Verificar contra:
#   → https://www.afip.gob.ar/institucional/estudios/archivos/recaudacion/
#   → Informes anuales de recaudación (PDF) para años anteriores a 2010.
#
# CÓMO ACTUALIZAR ESTA TABLA:
#   1. Ir a la URL de ARCA de arriba.
#   2. Descargar el Excel mensual más reciente.
#   3. Sumar los 12 meses de cada año por columna de impuesto.
#   4. Actualizar los valores de los años más recientes (2023, 2024).
#      Los valores de años anteriores son estables (datos definitivos).
#
# ADVERTENCIA PARA 2024:
#   El dato de 2024 puede ser provisorio si la notebook se corre antes
#   de que ARCA publique el cierre anual (típicamente en febrero del
#   año siguiente). Marcar como estimado con la flag 'provisorio_2024'.
# -----------------------------------------------------------------------------


### VERIFICADO MANUALMENTE 


RECAUDACION_ARCA = {
    #  anio: {
    #      'iva':              millones de $,
    #      'ganancias_total':  millones de $,
    #      'aportes_sipa':     millones de $,
    #      'contrib_patronal': millones de $,
    #      'internos':         millones de $,
    #      'bienes_personales': millones de $ (consistente con Sección 12),
    #      'fuente':           cadena con referencia al documento fuente,
    #      'provisorio':       bool — True si el dato es estimado/no definitivo,
    #  }
    
    2003: {                             
        'iva_bruto': 25287,
        'iva_neto': None,
        'ganancias_total': 14750,
        'aportes_sipa': 4692,         # Seguridad Social (Aportes)
        'contrib_patronal': 7633,    # Seguridad Social (Contribuciones)
        'monotributo': 282 + 294,          # Impositivo (757) + Seg. Social (592)
        'bienes_personales': 1602,    # Bienes Personales
        'cuentas_corrientes': 5900,   # Impuesto al Cheque (Ley 25413)
        'combustibles': 4973,         # Combustibles líquidos y GNC
        'internos': 2232,             # Internos (Tabaco, bebidas, etc.)
        'impuesto_pais': None,           
        'recaudacion_total_informada': 72243, # Total Tributario
        'fuente': 'AFIP — Informe Anual de Recaudación 2004 (PDF)',
        'provisorio': False,
    },

    2004: {                             
        'iva_bruto': 33247,
        'iva_neto': None,
        'ganancias_total': 22289,
        'aportes_sipa': 6375,         # Seguridad Social (Aportes)
        'contrib_patronal': 10166,    # Seguridad Social (Contribuciones)
        'monotributo': 509 + 455,          # Impositivo (757) + Seg. Social (592)
        'bienes_personales': 1660,    # Bienes Personales
        'cuentas_corrientes': 7681,   # Impuesto al Cheque (Ley 25413)
        'combustibles': 5378,         # Combustibles líquidos y GNC
        'internos': 3040,             # Internos (Tabaco, bebidas, etc.)
        'impuesto_pais': None,           
        'recaudacion_total_informada': 98284, # Total Tributario
        'fuente': 'AFIP — Informe Anual de Recaudación 2004 (PDF)',
        'provisorio': False,
    },
    2005: {
        'iva_bruto': 39654,
        'iva_neto': 36853,
        'ganancias_total': 26902,
        'aportes_sipa': 8003,         # Seguridad Social (Aportes)
        'contrib_patronal': 13348,    # Seguridad Social (Contribuciones)
        'monotributo': 509 + 591,          # Impositivo (757) + Seg. Social (592)
        'bienes_personales': 1812,    # Bienes Personales
        'cuentas_corrientes': 9434,   # Impuesto al Cheque (Ley 25413)
        'combustibles': 6017,         # Combustibles líquidos y GNC
        'internos': 3665,             # Internos (Tabaco, bebidas, etc.)
        'impuesto_pais': None,           
        'recaudacion_total_informada': 119252, # Total Tributario
        'fuente': 'serie2005.xls (Planilla oficial AFIP de recaudación de caja - procesada)',
        'provisorio': False,
    },
    2006: {
        'iva_bruto': 50367,
        'iva_neto': 47104,
        'ganancias_total': 33615,
        'aportes_sipa': 10730,         # Seguridad Social (Aportes)
        'contrib_patronal': 19646,    # Seguridad Social (Contribuciones)
        'monotributo': 868 + 661,          # Impositivo (757) + Seg. Social (592)
        'bienes_personales': 2076,    # Bienes Personales
        'cuentas_corrientes': 11685,   # Impuesto al Cheque (Ley 25413)
        'combustibles': 6572,         # Combustibles líquidos y GNC
        'internos': 4100,             # Internos (Tabaco, bebidas, etc.)
        'impuesto_pais': None,           
        'recaudacion_total_informada': 150008, # Total Tributario
        'fuente': 'serie2006.xls (Planilla oficial AFIP de recaudación de caja - procesada)',
        'provisorio': False,
    },
    2007: {
        'iva_bruto': 66659,
        'iva_neto': 62669,
        'ganancias_total': 42854,
        'aportes_sipa': 14322,         # Seguridad Social (Aportes)
        'contrib_patronal': 29249,    # Seguridad Social (Contribuciones)
        'monotributo': 1063 + 750,          # Impositivo (757) + Seg. Social (592)
        'bienes_personales': 2491,    # Bienes Personales
        'cuentas_corrientes': 15064,   # Impuesto al Cheque (Ley 25413)
        'combustibles': 7465,         # Combustibles líquidos y GNC
        'internos': 4728,             # Internos (Tabaco, bebidas, etc.)
        'impuesto_pais': None,           
        'recaudacion_total_informada': 199781, # Total Tributario
        'fuente': 'serie2007.xls (Planilla oficial AFIP de recaudación de caja - procesada)',
        'provisorio': False,
    },
    2008: {
        'iva_bruto': 86115,
        'iva_neto': 80228,
        'ganancias_total': 53646,
        'aportes_sipa': 23242,         # Seguridad Social (Aportes)
        'contrib_patronal': 39003,    # Seguridad Social (Contribuciones)
        'monotributo': 1354 + 1014,          # Impositivo (757) + Seg. Social (592)
        'bienes_personales': 3375,    # Bienes Personales
        'cuentas_corrientes': 19495,   # Impuesto al Cheque (Ley 25413)
        'combustibles': 9693,         # Combustibles líquidos y GNC
        'internos': 5630,             # Internos (Tabaco, bebidas, etc.)
        'impuesto_pais': None,           
        'recaudacion_total_informada': 269375, # Total Tributario
        'fuente': 'serie2008.xls (Planilla oficial AFIP de recaudación de caja - procesada)',
        'provisorio': False,
    },
    2009: {
        'iva_bruto': 91116,
        'iva_neto': 87385,
        'ganancias_total': 55552,
        'aportes_sipa': 28779,         # Seguridad Social (Aportes)
        'contrib_patronal': 47498,    # Seguridad Social (Contribuciones)
        'monotributo': 1549 + 1285,          # Impositivo (757) + Seg. Social (592)
        'bienes_personales': 4041,    # Bienes Personales
        'cuentas_corrientes': 20561,   # Impuesto al Cheque (Ley 25413)
        'combustibles': 11593,         # Combustibles líquidos y GNC
        'internos': 6737,             # Internos (Tabaco, bebidas, etc.)
        'impuesto_pais': None,           
        'recaudacion_total_informada': 304930, # Total Tributario
        'fuente': 'serie2009.xls (Planilla oficial AFIP de recaudación de caja - procesada)',
        'provisorio': False,
    },
    2010: {
        'iva_bruto': 120050,
        'iva_neto': 116385,
        'ganancias_total': 76651,
        'aportes_sipa': 37603,         # Seguridad Social (Aportes)
        'contrib_patronal': 61672,    # Seguridad Social (Contribuciones)
        'monotributo': 2092 + 2669,          # Impositivo (757) + Seg. Social (592)
        'bienes_personales': 5146,    # Bienes Personales
        'cuentas_corrientes': 26884,   # Impuesto al Cheque (Ley 25413)
        'combustibles': 15269,         # Combustibles líquidos y GNC
        'internos': 9502,             # Internos (Tabaco, bebidas, etc.)
        'impuesto_pais': None,           
        'recaudacion_total_informada': 409899, # Total Tributario
        'fuente': 'serie2010.xls (Planilla oficial AFIP de recaudación de caja - procesada)',
        'provisorio': False,
    },
    2011: {
        'iva_bruto': 158065,
        'iva_neto': 154236,
        'ganancias_total': 108597,
        'aportes_sipa': 51467,         # Seguridad Social (Aportes)
        'contrib_patronal': 82344,    # Seguridad Social (Contribuciones)
        'monotributo': 2983 + 3097,          # Impositivo (757) + Seg. Social (592)
        'bienes_personales': 5891,    # Bienes Personales
        'cuentas_corrientes': 36179,   # Impuesto al Cheque (Ley 25413)
        'combustibles': 18131,         # Combustibles líquidos y GNC
        'internos': 10956,             # Internos (Tabaco, bebidas, etc.)
        'impuesto_pais': None,           
        'recaudacion_total_informada': 540133, # Total Tributario
        'fuente': 'serie2011.xls (Planilla oficial AFIP de recaudación de caja - procesada)',
        'provisorio': False,
    },
    2012: {
        'iva_bruto': 251743,
        'iva_neto': 249006,
        'ganancias_total': 183598,
        'aportes_sipa': 91581,         # Seguridad Social (Aportes)
        'contrib_patronal': 138239,    # Seguridad Social (Contribuciones)
        'monotributo': 4422 + 5124,          # Impositivo (757) + Seg. Social (592)
        'bienes_personales': 10296,    # Bienes Personales
        'cuentas_corrientes': 56514,   # Impuesto al Cheque (Ley 25413)
        'combustibles': 31010,         # Combustibles líquidos y GNC
        'internos': 16490,             # Internos (Tabaco, bebidas, etc.)
        'impuesto_pais': None,           
        'recaudacion_total_informada': 858832, # Total Tributario
        'fuente': 'serie2012.xls (Planilla oficial AFIP de recaudación de caja - procesada)',
        'provisorio': False,
    },
    2013: {
        'iva_bruto': 251743,
        'iva_neto': 249006,
        'ganancias_total': 183598,
        'aportes_sipa': 91581,         # Seguridad Social (Aportes)
        'contrib_patronal': 138239,    # Seguridad Social (Contribuciones)
        'monotributo': 4422 + 5124,          # Impositivo (757) + Seg. Social (592)
        'bienes_personales': 10296,    # Bienes Personales
        'cuentas_corrientes': 56514,   # Impuesto al Cheque (Ley 25413)
        'combustibles': 31010,         # Combustibles líquidos y GNC
        'internos': 16490,             # Internos (Tabaco, bebidas, etc.)
        'impuesto_pais': None,           
        'recaudacion_total_informada': 858832, # Total Tributario
        'fuente': 'serie2013.xls (Planilla oficial AFIP de recaudación de caja - procesada)',
        'provisorio': False,
    },
    2014: {
        'iva_bruto': 336225,
        'iva_neto': 331202,
        'ganancias_total': 267075,
        'aportes_sipa': 119431,         # Seguridad Social (Aportes)
        'contrib_patronal': 179025,    # Seguridad Social (Contribuciones)
        'monotributo': 4259 + 7750,          # Impositivo (757) + Seg. Social (592)
        'bienes_personales': 14355,    # Bienes Personales
        'cuentas_corrientes': 76739,   # Impuesto al Cheque (Ley 25413)
        'combustibles': 44490,         # Combustibles líquidos y GNC
        'internos': 22419,             # Internos (Tabaco, bebidas, etc.)
        'impuesto_pais': None,           
        'recaudacion_total_informada': 1169682, # Total Tributario
        'fuente': 'serie2014.xls (Planilla oficial AFIP de recaudación de caja - procesada)',
        'provisorio': False,
    },
    2015: {
        'iva_bruto': 438499,
        'iva_neto': 433076,
        'ganancias_total': 381463,
        'aportes_sipa': 161262,         # Seguridad Social (Aportes)
        'contrib_patronal': 242282,    # Seguridad Social (Contribuciones)
        'monotributo': 5624 + 11089,          # Impositivo (757) + Seg. Social (592)
        'bienes_personales': 18210,    # Bienes Personales
        'cuentas_corrientes': 97479,   # Impuesto al Cheque (Ley 25413)
        'combustibles': 56478,         # Combustibles líquidos y GNC
        'internos': 46991,             # Internos (Tabaco, bebidas, etc.)
        'impuesto_pais': None,           
        'recaudacion_total_informada': 1537948, # Total Tributario
        'fuente': 'serie2015.xls (Planilla oficial AFIP de recaudación de caja - procesada)',
        'provisorio': False,
    },
    2016: {
        'iva_bruto': 589148,
        'iva_neto': 583216,
        'ganancias_total': 432907,
        'aportes_sipa': 212743,         # Seguridad Social (Aportes)
        'contrib_patronal': 324234,    # Seguridad Social (Contribuciones)
        'monotributo': 6873 + 14039,          # Impositivo (757) + Seg. Social (592)
        'bienes_personales': 19540,    # Bienes Personales
        'cuentas_corrientes': 131669,   # Impuesto al Cheque (Ley 25413)
        'combustibles': 75663,         # Combustibles líquidos y GNC
        'internos': 46991,             # Internos (Tabaco, bebidas, etc.)
        'impuesto_pais': None,           
        'recaudacion_total_informada': 2070153, # Total Tributario
        'fuente': 'serie2016.xls (Planilla oficial AFIP de recaudación de caja - procesada)',
        'provisorio': False,
    },
    2017: {
        'iva_bruto': 785842,
        'iva_neto': 765336,
        'ganancias_total': 555022,
        'aportes_sipa': 279185,         # Seguridad Social (Aportes)
        'contrib_patronal': 424268,    # Seguridad Social (Contribuciones)
        'monotributo': 10544 + 20092,          # Impositivo (757) + Seg. Social (592)
        'bienes_personales': 22165,    # Bienes Personales
        'cuentas_corrientes': 172838,   # Impuesto al Cheque (Ley 25413)
        'combustibles': 102838,         # Combustibles líquidos y GNC
        'internos': 68985,             # Internos (Tabaco, bebidas, etc.)
        'impuesto_pais': None,           
        'recaudacion_total_informada': 2578609, # Total Tributario
        'fuente': 'serie2017.xls (Planilla oficial AFIP de recaudación de caja - procesada)',
        'provisorio': False,
    },
    2018: {
        'iva_bruto': 1127980,
        'iva_neto': 1104580,
        'ganancias_total': 742052,
        'aportes_sipa': 352350,         # Seguridad Social (Aportes)
        'contrib_patronal': 526704,    # Seguridad Social (Contribuciones)
        'monotributo': 16052 + 26388,          # Impositivo (757) + Seg. Social (592)
        'bienes_personales': 14517,    # Bienes Personales
        'cuentas_corrientes': 234299,   # Impuesto al Cheque (Ley 25413)
        'combustibles': 116408,         # Combustibles líquidos y GNC
        'internos': 75323,             # Internos (Tabaco, bebidas, etc.)
        'impuesto_pais': None,           
        'recaudacion_total_informada': 3382644, # Total Tributario
        'fuente': 'serie2018.xls (Planilla oficial AFIP de recaudación de caja - procesada)',
        'provisorio': False,
    },
    2019: {
        'iva_bruto': 1578296,
        'iva_neto': 1532596,
        'ganancias_total': 1096521,
        'aportes_sipa': 472510,         # Seguridad Social (Aportes)
        'contrib_patronal': 705020,    # Seguridad Social (Contribuciones)
        'monotributo': 20634 + 35011,          # Impositivo (757) + Seg. Social (592)
        'bienes_personales': 31183,    # Bienes Personales
        'cuentas_corrientes': 349559,   # Impuesto al Cheque (Ley 25413)
        'combustibles': 161666,         # Combustibles líquidos y GNC
        'internos': 95832,             # Internos (Tabaco, bebidas, etc.)
        'impuesto_pais': None,           
        'recaudacion_total_informada': 5023566, # Total Tributario
        'fuente': 'serie2019.xls (Planilla oficial AFIP de recaudación de caja - procesada)',
        'provisorio': False,
    },
    2020: {
        'iva_bruto': 1951560,
        'iva_neto': 1905385,
        'ganancias_total': 1467303,
        'aportes_sipa': 608227,         # Seguridad Social (Aportes)
        'contrib_patronal': 877361,    # Seguridad Social (Contribuciones)
        'monotributo': 29168 + 52017,          # Impositivo (757) + Seg. Social (592)
        'bienes_personales': 207381,    # Bienes Personales
        'cuentas_corrientes': 452459,   # Impuesto al Cheque (Ley 25413)
        'combustibles': 195658,         # Combustibles líquidos y GNC
        'internos': 165628,             # Internos (Tabaco, bebidas, etc.)
        'impuesto_pais': 134971,           
        'recaudacion_total_informada': 6635238, # Total Tributario
        'fuente': 'serie2020.xls (Planilla oficial AFIP de recaudación de caja - procesada)',
        'provisorio': False,
    },
    2021: {
        'iva_bruto': 3306008,
        'iva_neto': 3243608,
        'ganancias_total': 2346646,
        'aportes_sipa': 932106,         # Seguridad Social (Aportes)
        'contrib_patronal': 1375966,    # Seguridad Social (Contribuciones)
        'monotributo': 66016,          # Impositivo (757) + Seg. Social (592)
        'bienes_personales': 278770,    # Bienes Personales
        'cuentas_corrientes': 750414,   # Impuesto al Cheque (Ley 25413)
        'combustibles': 356273,         # Combustibles líquidos y GNC
        'internos': 260576,             # Internos (Tabaco, bebidas, etc.)
        'impuesto_pais': 111438,           
        'recaudacion_total_informada': 11004880, # Total Tributario
        'fuente': 'serie2021.xls (Planilla oficial AFIP de recaudación de caja - procesada)',
        'provisorio': False,
    },
    2022: {
        'iva_bruto': 5925040,
        'iva_neto': 5831330,
        'ganancias_total': 4715417,
        'aportes_sipa': 1653851,         # Seguridad Social (Aportes)
        'contrib_patronal': 2506488,    # Seguridad Social (Contribuciones)
        'monotributo': 104007,          # Impositivo (757) + Seg. Social (592)
        'bienes_personales': 421969,    # Bienes Personales
        'cuentas_corrientes': 1343374,   # Impuesto al Cheque (Ley 25413)
        'combustibles': 431487,         # Combustibles líquidos y GNC
        'internos': 429692,             # Internos (Tabaco, bebidas, etc.)
        'impuesto_pais': 346908,           
        'recaudacion_total_informada': 19982483, # Total Tributario
        'fuente': 'serie2022.xls (Planilla oficial AFIP de recaudación de caja - procesada)',
        'provisorio': False,
    },
    2023: {
        'iva_bruto': 14987310,
        'iva_neto': 14791510,
        'ganancias_total': 8782659,
        # NOTA: Ganancias 2023 inusualmente alto por efecto de la
        # reforma de octubre (Decreto 473/2023) que eliminó transitoriamente
        # el impuesto para la mayoría de los trabajadores; paradójicamente
        # la recaudación anual subió por el impacto de la inflación en
        # los tramos nominales durante los primeros 9 meses del año.
        'aportes_sipa': 369485,         # Seguridad Social (Aportes)
        'contrib_patronal': 5713334,    # Seguridad Social (Contribuciones)
        'monotributo': 68791 + 221945,          # Impositivo (757) + Seg. Social (592)
        'bienes_personales': 987942,    # Bienes Personales
        'cuentas_corrientes': 3146166,   # Impuesto al Cheque (Ley 25413)
        'combustibles': 526432,         # Combustibles líquidos y GNC
        'internos': 1015325,             # Internos (Tabaco, bebidas, etc.)
        'impuesto_pais': 1510889,          # el Estado devolvió más dinero del que recaudó
        'recaudacion_total_informada': 183109217, # Total Tributario
        'fuente': 'serie2025.xls (Planilla oficial AFIP de recaudación de caja - procesada)',
        'provisorio': True,
    },
    2024: {
        'iva_bruto': 43648692,
        'iva_neto': 43027692,
               # Extraído para restar a deciles bajos en Fase 5
        'ganancias_total': 26068568,     # Ganancias
        # NOTA: Ganancias 2024 cae en términos reales respecto a 2023 por
        # la reforma de la Ley 27.743 (Paquete Fiscal) que elevó masivamente
        # la GNI y redujo el universo de contribuyentes en relación de dependencia.
        'aportes_sipa': 10862131,         # Seguridad Social (Aportes)
        'contrib_patronal': 17928264,    # Seguridad Social (Contribuciones)
        'monotributo': 224975 + 610537,          # Impositivo (757) + Seg. Social (592)
        'bienes_personales': 228113,    # Bienes Personales
        'cuentas_corrientes': 94147135,   # Impuesto al Cheque (Ley 25413)
        'combustibles': 2520915,         # Combustibles líquidos y GNC
        'internos': 2750826,             # Internos (Tabaco, bebidas, etc.)
        'impuesto_pais': 6274713,           
        'recaudacion_total_informada': 131357699, # Total Tributario
        'fuente': 'serie2024.xls (Planilla oficial AFIP de recaudación de caja - procesada)',
        'provisorio': True,
    },
        2025: {
        'iva_bruto': 62076999,
        'iva_neto': 61296099,
        'ganancias_total': 37063166,     # Ganancias
        'aportes_sipa': 17398187,         # Seguridad Social (Aportes)
        'contrib_patronal': 28328139,    # Seguridad Social (Contribuciones)
        'monotributo': 707015 + 1065044,          # Impositivo (757) + Seg. Social (592)
        'bienes_personales': 1975263,    # Bienes Personales
        'cuentas_corrientes': 13995364,   # Impuesto al Cheque (Ley 25413)
        'combustibles': 486146,         # Combustibles líquidos y GNC
        'internos': 3419844,             # Internos (Tabaco, bebidas, etc.)
        'impuesto_pais': -5705,           
        'recaudacion_total_informada': 183109217, # Total Tributario
        'fuente': 'serie2025.xls (Planilla oficial AFIP de recaudación de caja - procesada)',
        'provisorio': True,
    },
}

print(f"✓ Tabla de recaudación ARCA cargada: {len(RECAUDACION_ARCA)} años (2004–2024)")
provisorios = [a for a, v in RECAUDACION_ARCA.items() if v['provisorio']]
if provisorios:
    print(f"⚠️  Años con datos provisorios: {provisorios} — verificar antes de Fase 5")

✓ Tabla de recaudación ARCA cargada: 23 años (2004–2024)
⚠️  Años con datos provisorios: [2023, 2024, 2025] — verificar antes de Fase 5


In [ ]:
# -----------------------------------------------------------------------------
# B. COEFICIENTE DE PARTICIPACIÓN DE PERSONAS FÍSICAS EN GANANCIAS
# -----------------------------------------------------------------------------
# La ARCA publica Ganancias como total consolidado (personas físicas +
# sociedades). Para la validación del modelo de incidencia — que solo
# modela personas físicas — necesitamos aislar la parte correspondiente.
#
# Fuente: AFIP — Estadísticas Tributarias
#   Cuadro: "Impuesto a las Ganancias — declarantes y montos por tipo de sujeto"
#   URL: https://www.afip.gob.ar/institucional/estudios/
#
# METODOLOGÍA DE ESTIMACIÓN:
#   Los coeficientes se derivan del ratio entre la recaudación imputada a
#   personas físicas (4.ª categoría + otras categorías) y el total de
#   Ganancias publicado por ARCA. Para años donde el cuadro desagregado
#   no está disponible en formato digital, se interpola linealmente entre
#   los años contiguos verificados.
#
# RANGO HISTÓRICO OBSERVADO:
#   El coeficiente oscila entre 0.20 y 0.35. Tiende a subir cuando:
#     (a) hay actualización de escalas (más personas quedan exentas →
#         la participación de sociedades sube relativamente), o
#     (b) hay shocks de rentabilidad corporativa que elevan Ganancias
#         de sociedades (ej. 2022–2023 por inflación de activos).
#
# NOTA SOBRE 2023:
#   El coeficiente cae marcadamente porque la reforma de octubre 2023
#   sacó a la mayoría de los asalariados del impuesto, reduciendo
#   drásticamente la recaudación de 4.ª categoría en Q4.
# -----------------------------------------------------------------------------

# VERIFICADOS MANUALMENTE

datos_completos_ganancias = {
    2003: {
        'retenciones_caja': 7079058000,
        'pagos_directos': 7671679000,
        'ph_devengado': 2758710000,
        'sociedades_devengado': 6138967000,
        'coef_ganancias_pf': 0.310048,
        'ganancias_total': 14750,
        'ganancias_pf_recaudacion': 4573.213
    },
    2004: {
        'retenciones_caja': 8905619000,
        'pagos_directos': 13383480000,
        'ph_devengado': 3873724000,
        'sociedades_devengado': 10974880000,
        'coef_ganancias_pf': 0.260881,
        'ganancias_total': 22289,
        'ganancias_pf_recaudacion': 5814.786
    },
    2005: {
        'retenciones_caja': 11724660000,
        'pagos_directos': 16320740000,
        'ph_devengado': 4276305000,
        'sociedades_devengado': 15757760000,
        'coef_ganancias_pf': 0.213452,
        'ganancias_total': 26902,
        'ganancias_pf_recaudacion': 5742.276
    },
    2006: {
        'retenciones_caja': 15015060000,
        'pagos_directos': 18600030000,
        'ph_devengado': 5135029000,
        'sociedades_devengado': 19717880000,
        'coef_ganancias_pf': 0.206617,
        'ganancias_total': 33615,
        'ganancias_pf_recaudacion': 6945.424
    },
    2007: {
        'retenciones_caja': 19567350000,
        'pagos_directos': 23287560000,
        'ph_devengado': 6291177000,
        'sociedades_devengado': 25319600000,
        'coef_ganancias_pf': 0.199020,
        'ganancias_total': 42854,
        'ganancias_pf_recaudacion': 8528.803
    },
    2008: {
        'retenciones_caja': 26630110000,
        'pagos_directos': 27015890000,
        'ph_devengado': 9777633000,
        'sociedades_devengado': 31281950000,
        'coef_ganancias_pf': 0.238133,
        'ganancias_total': 53646,
        'ganancias_pf_recaudacion': 12774.870
    },
    2009: {
        'retenciones_caja': 27158620000,
        'pagos_directos': 28393640000,
        'ph_devengado': 11014720000,
        'sociedades_devengado': 40865820000,
        'coef_ganancias_pf': 0.212309,
        'ganancias_total': 55552,
        'ganancias_pf_recaudacion': 11794.200
    },
    2010: {
        'retenciones_caja': 36093940000,
        'pagos_directos': 40557680000,
        'ph_devengado': 10509620000,
        'sociedades_devengado': 43886290000,
        'coef_ganancias_pf': 0.193206,
        'ganancias_total': 76651,
        'ganancias_pf_recaudacion': 14809.430
    },
    2011: {
        'retenciones_caja': 51914300000,
        'pagos_directos': 56683580000,
        'ph_devengado': 15865750000,
        'sociedades_devengado': 58788280000,
        'coef_ganancias_pf': 0.212524,
        'ganancias_total': 108597,
        'ganancias_pf_recaudacion': 23079.430
    },
    2012: {
        'retenciones_caja': 72033100000,
        'pagos_directos': 0,
        'ph_devengado': 21123020000,
        'sociedades_devengado': 76062650000,
        'coef_ganancias_pf': 0.217347,
        'ganancias_total': 183598,
        'ganancias_pf_recaudacion': 39904.480
    },
    2013: {
        'retenciones_caja': 105701000000,
        'pagos_directos': 77898000000,
        'ph_devengado': 27877000000,
        'sociedades_devengado': 91777000000,
        'coef_ganancias_pf': 0.232980,
        'ganancias_total': 183598,
        'ganancias_pf_recaudacion': 42774.680
    },
    2014: {
        'retenciones_caja': 165836000000,
        'pagos_directos': 101239000000,
        'ph_devengado': 38345000000,
        'sociedades_devengado': 112707000000,
        'coef_ganancias_pf': 0.253853,
        'ganancias_total': 267075,
        'ganancias_pf_recaudacion': 67797.780
    },
    2015: {
        'retenciones_caja': 228467000000,
        'pagos_directos': 152996200000,
        'ph_devengado': 56969620000,
        'sociedades_devengado': 166889000000,
        'coef_ganancias_pf': 0.254489,
        'ganancias_total': 381463,
        'ganancias_pf_recaudacion': 97078.250
    },
    2016: {
        'retenciones_caja': 251569400000,
        'pagos_directos': 181337800000,
        'ph_devengado': 77978620000,
        'sociedades_devengado': 201168700000,
        'coef_ganancias_pf': 0.279346,
        'ganancias_total': 432907,
        'ganancias_pf_recaudacion': 120930.700
    },
    2017: {
        'retenciones_caja': 300581100000,
        'pagos_directos': 254441800000,
        'ph_devengado': 101123300000,
        'sociedades_devengado': 274724400000,
        'coef_ganancias_pf': 0.269054,
        'ganancias_total': 555022,
        'ganancias_pf_recaudacion': 149330.800
    },
    2018: {
        'retenciones_caja': 401917000000,
        'pagos_directos': 340135400000,
        'ph_devengado': 123478000000,
        'sociedades_devengado': 343648800000,
        'coef_ganancias_pf': 0.264335,
        'ganancias_total': 742052,
        'ganancias_pf_recaudacion': 196150.400
    },
    2019: {
        'retenciones_caja': 597651100000,
        'pagos_directos': 498870100000,
        'ph_devengado': 160629800000,
        'sociedades_devengado': 466063600000,
        'coef_ganancias_pf': 0.256313,
        'ganancias_total': 1096521,
        'ganancias_pf_recaudacion': 281052.800
    },
    2020: {
        'retenciones_caja': 811621600000,
        'pagos_directos': 655681700000,
        'ph_devengado': 246371100000,
        'sociedades_devengado': 648434100000,
        'coef_ganancias_pf': 0.275335,
        'ganancias_total': 1467303,
        'ganancias_pf_recaudacion': 403999.700
    },
    2021: {
        'retenciones_caja': 1453681000000,
        'pagos_directos': 892964900000,
        'ph_devengado': 332147900000,
        'sociedades_devengado': 878246800000,
        'coef_ganancias_pf': 0.274413,
        'ganancias_total': 2346646,
        'ganancias_pf_recaudacion': 643949.900
    },
    2022: {
        'retenciones_caja': 2873306000000,
        'pagos_directos': 1842111000000,
        'ph_devengado': 546243600000,
        'sociedades_devengado': 1473516000000,
        'coef_ganancias_pf': 0.270450,
        'ganancias_total': 4715417,
        'ganancias_pf_recaudacion': 1275284.000
    },
    2023: {
        'retenciones_caja': 6306743000000,
        'pagos_directos': 2475916000000,
        'ph_devengado': 941746800000,
        'sociedades_devengado': 2398480000000,
        'coef_ganancias_pf': 0.281941,
        'ganancias_total': 8782659,
        'ganancias_pf_recaudacion': 2476191.000
    },
    2024: {
        'retenciones_caja': 15116940000000,
        'pagos_directos': 10951630000000,
        'ph_devengado': 2086554000000,
        'sociedades_devengado': 8480274000000,
        'coef_ganancias_pf': 0.197463,
        'ganancias_total': 26068568,
        'ganancias_pf_recaudacion': 5147569.000
    },
}

# =============================================================================
# PARTE B — COEFICIENTE REAL DE PERSONAS FÍSICAS EN GANANCIAS
# =============================================================================
# Se reemplaza el diccionario estimado COEF_GANANCIAS_PF por los datos
# reales de datos_completos_ganancias, que ya tienen el coeficiente calculado
# y la recaudación de PF derivada.
#
# Fuente de datos_completos_ganancias:
#   - coef_ganancias_pf: ph_devengado / (ph_devengado + sociedades_devengado)
#   - ganancias_pf_recaudacion: ganancias_total (M$) × coef_ganancias_pf
#   - Unidades: ganancias_total y ganancias_pf_recaudacion en MILLONES de $
#                ph_devengado y sociedades_devengado en PESOS nominales
#
# NOTA: se preservan iva_bruto e iva_neto en RECAUDACION_ARCA tal como están.
#       Las Partes C/D/E usan iva_neto como campo de trabajo (ver decisión abajo).
# =============================================================================

# Inyectar en RECAUDACION_ARCA los campos derivados de datos reales
anos_sin_datos = []
for anio in sorted(RECAUDACION_ARCA):
    dgc = datos_completos_ganancias.get(anio)
    if dgc is None:
        anos_sin_datos.append(anio)
        # fallback: marcar como None para detectar en validación
        RECAUDACION_ARCA[anio]['coef_pf_ganancias']   = None
        RECAUDACION_ARCA[anio]['ganancias_pf']         = None
        continue

    RECAUDACION_ARCA[anio]['coef_pf_ganancias'] = dgc['coef_ganancias_pf']
    RECAUDACION_ARCA[anio]['ganancias_pf']       = dgc['ganancias_pf_recaudacion']  # ya en M$

if anos_sin_datos:
    print(f"⚠️  Años sin datos en datos_completos_ganancias: {anos_sin_datos}")

# Resumen de verificación
print(f"{'Año':<6} {'Coef. PF':>9}  {'Gan. total (M$)':>16}  {'Gan. PF (M$)':>14}  {'Check':>7}")
print("-" * 60)
for anio in sorted(RECAUDACION_ARCA):
    d = RECAUDACION_ARCA[anio]
    if d['coef_pf_ganancias'] is None:
        print(f"{anio:<6} {'—':>9}  {'—':>16}  {'—':>14}  {'❌':>7}")
        continue
    # Verificar consistencia: coef × total ≈ pf (tolerancia 1M$ por redondeo)
    check_val = d['ganancias_total'] * d['coef_pf_ganancias']
    diff      = abs(check_val - d['ganancias_pf'])
    check_str = "✓" if diff < 10 else f"⚠️ Δ{diff:.0f}"
    print(f"{anio:<6} {d['coef_pf_ganancias']:>9.6f}  "
          f"${d['ganancias_total']:>14,.0f}  "
          f"${d['ganancias_pf']:>12,.0f}  {check_str:>7}")

⚠️  Años sin datos en datos_completos_ganancias: [2025]
Año     Coef. PF   Gan. total (M$)    Gan. PF (M$)    Check
------------------------------------------------------------
2003    0.310048  $        14,750  $       4,573        ✓
2004    0.260881  $        22,289  $       5,815        ✓
2005    0.213452  $        26,902  $       5,742        ✓
2006    0.206617  $        33,615  $       6,945        ✓
2007    0.199020  $        42,854  $       8,529        ✓
2008    0.238133  $        53,646  $      12,775        ✓
2009    0.212309  $        55,552  $      11,794        ✓
2010    0.193206  $        76,651  $      14,809        ✓
2011    0.212524  $       108,597  $      23,079        ✓
2012    0.217347  $       183,598  $      39,904        ✓
2013    0.232980  $       183,598  $      42,775        ✓
2014    0.253853  $       267,075  $      67,798        ✓
2015    0.254489  $       381,463  $      97,078        ✓
2016    0.279346  $       432,907  $     120,931        ✓
2017    0.2

In [11]:
# =============================================================================
# PARTE C — CONSTRUCCIÓN DEL DATAFRAME MAESTRO Y GUARDADO
# =============================================================================
# IVA UTILIZADO: iva_neto
#
# DECISIÓN: para medir la carga que recae sobre los hogares se usa iva_neto.
#   iva_bruto incluye créditos fiscales que las empresas se recuperan entre sí
#   en la cadena de producción. El consumidor final solo absorbe el IVA que
#   no fue recuperado en ningún eslabón: el neto. Usar bruto sobreestimaría
#   la carga de los deciles bajos.
#   Referencia metodológica: Rossignolo (2014), CEPAL (2018), CEQ Argentina.
#
# Se preservan iva_bruto e iva_neto en RECAUDACION_ARCA sin modificar.
# El campo de trabajo en esta tabla es iva_neto.
#
# Columnas del output:
#   anio | iva_neto | iva_bruto | ganancias_total | ganancias_pf |
#   coef_pf_ganancias | aportes_sipa | contrib_patronal | internos |
#   bienes_personales | total_modelizado | fuente | provisorio
#
# 'total_modelizado' = iva_neto + ganancias_pf + aportes_sipa +
#                      contrib_patronal + internos + bienes_personales
# No incluye ganancias de sociedades ni IIBB provincial.
# =============================================================================

registros_arca = []
errores_construccion = []

# Para 2003 y 2004, iva_neto no disponible — usar iva_bruto como proxy
for anio in [2003, 2004]:
    if RECAUDACION_ARCA[anio].get('iva_neto') is None:
        RECAUDACION_ARCA[anio]['iva_neto'] = RECAUDACION_ARCA[anio]['iva_bruto']
        print(f"ℹ️  {anio}: iva_neto imputado desde iva_bruto (diferencia esperada <2%)")
for anio in sorted(RECAUDACION_ARCA):
    d = RECAUDACION_ARCA[anio]

    # Detectar campos None antes de operar
    campos_requeridos = [
        'iva_neto', 'ganancias_pf', 'aportes_sipa',
        'contrib_patronal', 'internos', 'bienes_personales'
    ]
    faltantes = [c for c in campos_requeridos if d.get(c) is None]
    if faltantes:
        errores_construccion.append(f"{anio}: campos None → {faltantes}")
        total_mod = None
    else:
        total_mod = (
            d['iva_neto']
            + d['ganancias_pf']
            + d['aportes_sipa']
            + d['contrib_patronal']
            + d['internos']
            + d['bienes_personales']
        )

    registros_arca.append({
        'anio':               anio,
        'iva_neto':           d.get('iva_neto'),
        'iva_bruto':          d.get('iva_bruto'),       # preservado, no usado en modelo
        'ganancias_total':    d.get('ganancias_total'),
        'ganancias_pf':       d.get('ganancias_pf'),
        'coef_pf_ganancias':  d.get('coef_pf_ganancias'),
        'aportes_sipa':       d.get('aportes_sipa'),
        'contrib_patronal':   d.get('contrib_patronal'),
        'internos':           d.get('internos'),
        'bienes_personales':  d.get('bienes_personales'),
        'total_modelizado':   total_mod,
        'fuente':             d.get('fuente'),
        'provisorio':         d.get('provisorio'),
    })

df_arca = pd.DataFrame(registros_arca)
df_arca.to_csv(RUTAS['arca'] / "recaudacion_arca_2004_2024.csv", index=False)

# Reportar errores de construcción
if errores_construccion:
    print("⚠️  Filas con campos faltantes (total_modelizado = NaN):")
    for e in errores_construccion:
        print(f"   {e}")
else:
    print("✓ Todas las filas construidas sin valores faltantes.")

print(f"\n✅ Guardado: recaudacion_arca_2004_2024.csv ({len(df_arca)} filas)")
print()

# Vista de verificación — todos los campos ya están en M$, NO dividir por 1e6
print(f"{'Año':<6} {'IVA neto':>10}  {'Gan. PF':>10}  {'Aportes':>10}  "
      f"{'C.Patron.':>10}  {'Internos':>9}  {'BP':>8}  {'Total mod.':>14}  {'Prov.':>5}")
print("-" * 97)
for _, row in df_arca.iterrows():
    flag = "  ⚠️" if row['provisorio'] else ""
    tm   = f"${row['total_modelizado']:>10,.0f}M" if pd.notna(row['total_modelizado']) else "         N/A"
    print(
        f"{int(row['anio']):<6} "
        f"${row['iva_neto']:>8,.0f}M  "
        f"${row['ganancias_pf']:>8,.0f}M  "
        f"${row['aportes_sipa']:>8,.0f}M  "
        f"${row['contrib_patronal']:>8,.0f}M  "
        f"${row['internos']:>7,.0f}M  "
        f"${row['bienes_personales']:>6,.0f}M  "
        f"{tm}"
        f"{flag}"
    )
print()
print("Nota: iva_bruto preservado en CSV pero excluido de total_modelizado.")
print("      ganancias_total preservado para referencia; el modelo usa ganancias_pf.")



ℹ️  2003: iva_neto imputado desde iva_bruto (diferencia esperada <2%)
ℹ️  2004: iva_neto imputado desde iva_bruto (diferencia esperada <2%)
⚠️  Filas con campos faltantes (total_modelizado = NaN):
   2025: campos None → ['ganancias_pf']

✅ Guardado: recaudacion_arca_2004_2024.csv (23 filas)

Año      IVA neto     Gan. PF     Aportes   C.Patron.   Internos        BP      Total mod.  Prov.
-------------------------------------------------------------------------------------------------
2003   $  25,287M  $   4,573M  $   4,692M  $   7,633M  $  2,232M  $ 1,602M  $    46,019M
2004   $  33,247M  $   5,815M  $   6,375M  $  10,166M  $  3,040M  $ 1,660M  $    60,303M
2005   $  36,853M  $   5,742M  $   8,003M  $  13,348M  $  3,665M  $ 1,812M  $    69,423M
2006   $  47,104M  $   6,945M  $  10,730M  $  19,646M  $  4,100M  $ 2,076M  $    90,601M
2007   $  62,669M  $   8,529M  $  14,322M  $  29,249M  $  4,728M  $ 2,491M  $   121,988M
2008   $  80,228M  $  12,775M  $  23,242M  $  39,003M  $  5,630M  

In [8]:
import pprint
pprint.pprint(RECAUDACION_ARCA[2010])

{'aportes_sipa': 37603,
 'bienes_personales': 5146,
 'coef_pf_ganancias': 0.193206,
 'combustibles': 15269,
 'contrib_patronal': 61672,
 'cuentas_corrientes': 26884,
 'fuente': 'serie2010.xls (Planilla oficial AFIP de recaudación de caja - '
           'procesada)',
 'ganancias_pf': 14809.43,
 'ganancias_total': 76651,
 'impuesto_pais': None,
 'internos': 9502,
 'iva_bruto': 120050,
 'iva_neto': 116385,
 'monotributo': 4761,
 'provisorio': False,
 'recaudacion_total_informada': 409899}


In [ ]:
# -----------------------------------------------------------------------------
# D. PRESIÓN TRIBUTARIA COMO % DEL PBI — SERIE HISTÓRICA (referencia externa)
# -----------------------------------------------------------------------------
# Para contextualizar la recaudación nominal en términos reales y comparables
# entre años, calculamos la presión tributaria de los impuestos modelizados
# como porcentaje del PBI nominal.
#
# FUENTE DEL PBI NOMINAL:
#   INDEC — Cuentas Nacionales, Serie a precios corrientes.
#   URL: https://www.indec.gob.ar/indec/web/Nivel4-Tema-3-9-47
#
# ADVERTENCIA AÑOS 2007–2015:
#   El PBI nominal del INDEC para este período no está afectado por la
#   intervención en el IPC (que distorsionaba el PBI real, no el nominal),
#   salvo por el efecto de la subdeclaración de inflación sobre el deflactor
#   implícito. Se usan los datos oficiales con nota de cautela.
#
# Este cálculo es INFORMATIVO — no entra en el modelo de incidencia.
# Su propósito es detectar inconsistencias: si la recaudación modelizada
# implica una presión tributaria absurda (ej. 60% del PBI), hay un error.
# -----------------------------------------------------------------------------

# PBI nominal anual — millones de pesos corrientes
# Fuente: INDEC — Cuentas Nacionales (Base 2004)
# Para años recientes la base puede diferir; se usa la serie empalmada
# publicada por el propio INDEC en el cuadro "Oferta y demanda globales".


### VERIFICAR MANUALMENTE!

PBI_NOMINAL_MILLONES = {
    2004:   484_731,
    2005:   648_239,
    2006:   821_400,
    2007: 1_040_716,
    2008: 1_332_357,
    2009: 1_451_565,
    2010: 1_940_967,
    2011: 2_612_539,
    2012: 3_166_177,
    2013: 3_929_946,
    2014: 5_158_761,
    2015: 6_607_012,
    2016: 8_228_835,
    2017: 9_999_817,
    2018: 14_607_558,
    2019: 21_610_462,
    2020: 25_702_327,
    2021: 41_672_981,
    2022: 84_595_413,
    2023: 270_012_847,
    2024: 890_000_000,  # Estimación — verificar con INDEC al cierre    TODOS ESTOS DATOS HAY QUE VERIFICARLOS MANUALMENTE
}

# Calcular presión tributaria de los impuestos modelizados
registros_pt = []
for anio in sorted(RECAUDACION_ARCA):
    d    = RECAUDACION_ARCA[anio]
    pbi  = PBI_NOMINAL_MILLONES[anio]
    prov = d['provisorio']

    total_mod = (
        d['iva'] + d['ganancias_pf'] + d['aportes_sipa']
        + d['contrib_patronal'] + d['internos'] + d['bienes_personales']
    )
    total_inc_sociedades = (
        d['iva'] + d['ganancias_total'] + d['aportes_sipa']
        + d['contrib_patronal'] + d['internos'] + d['bienes_personales']
    )

    registros_pt.append({
        'anio':                   anio,
        'pbi_nominal_M':          pbi,
        'pt_modelizado_pct':      total_mod / pbi * 100,
        'pt_total_inc_soc_pct':   total_inc_sociedades / pbi * 100,
        'provisorio':             prov,
    })

df_pt = pd.DataFrame(registros_pt)
df_pt.to_csv(RUTAS['arca'] / "presion_tributaria_referencia_2004_2024.csv", index=False)

print("Presión tributaria de los impuestos modelizados (% del PBI):")
print("(Referencia esperada: entre 18% y 28% según año — ver CEPAL Panorama Fiscal)")
print()
print(f"{'Año':<6} {'PT modelizado':>14}  {'PT total (c/soc)':>17}  {'Provisorio':>10}")
print("-" * 55)
for _, row in df_pt.iterrows():
    flag = "  ⚠️" if row['provisorio'] else ""
    rango_ok = 15 <= row['pt_modelizado_pct'] <= 32
    alerta   = "" if rango_ok else "  ← REVISAR"
    print(
        f"{int(row['anio']):<6} "
        f"{row['pt_modelizado_pct']:>13.1f}%  "
        f"{row['pt_total_inc_soc_pct']:>16.1f}%"
        f"{flag}{alerta}"
    )

print()
print("✅ Guardado: presion_tributaria_referencia_2004_2024.csv")

Presión tributaria de los impuestos modelizados (% del PBI):
(Referencia esperada: entre 18% y 28% según año — ver CEPAL Panorama Fiscal)

Año     PT modelizado   PT total (c/soc)  Provisorio
-------------------------------------------------------
2004            14.5%              17.2%  ← REVISAR
2005            14.0%              16.8%  ← REVISAR
2006            14.1%              17.0%  ← REVISAR
2007            14.3%              17.3%  ← REVISAR
2008            14.8%              17.9%  ← REVISAR
2009            13.6%              16.4%  ← REVISAR
2010            14.1%              17.0%  ← REVISAR
2011            14.4%              17.5%  ← REVISAR
2012            14.8%              17.8%  ← REVISAR
2013            15.5%              18.7%
2014            16.4%              19.9%
2015            16.7%              20.3%
2016            17.1%              21.4%
2017            17.8%              22.4%
2018            17.3%              21.8%
2019            16.7%              20.

In [38]:
# -----------------------------------------------------------------------------
# E. VERIFICACIÓN DE CONSISTENCIA INTERNA DE LA SERIE
# -----------------------------------------------------------------------------
# Tres controles de sanidad antes de cerrar la Fase 1:
#
#   Control 1 — Monotonía nominal: la recaudación de cada impuesto debe
#               crecer todos los años en términos nominales (inflación).
#               Una caída nominal es una señal de error en los datos.
#               Excepción legítima: cambios de política (ej. Ganancias 2023
#               por la reforma de octubre → Q4 con recaudación casi nula).
#
#   Control 2 — Proporción interna: el IVA debe ser el mayor impuesto en
#               todos los años (es el de mayor recaudación en Argentina).
#               Ganancias total puede superar al IVA en algunos años
#               (ocurrió en 2023 por los factores antes mencionados).
#
#   Control 3 — Consistencia con Sección 12: el campo 'bienes_personales'
#               de esta tabla debe coincidir exactamente con
#               RECAUDACION_BP_MILLONES de la Sección 12.
# -----------------------------------------------------------------------------

print("=" * 65)
print("VERIFICACIÓN DE CONSISTENCIA — SECCIÓN 13")
print("=" * 65)

errores = []

# --- Control 1: Monotonía nominal ---
print("\n▶ Control 1: Monotonía nominal (caídas nominales son sospechosas)")
impuestos_check = ['iva', 'aportes_sipa', 'contrib_patronal', 'internos', 'bienes_personales']
anios_ordenados = sorted(RECAUDACION_ARCA.keys())

for imp in impuestos_check:
    caidas = []
    for i in range(1, len(anios_ordenados)):
        a_prev = anios_ordenados[i - 1]
        a_curr = anios_ordenados[i]
        v_prev = RECAUDACION_ARCA[a_prev][imp]
        v_curr = RECAUDACION_ARCA[a_curr][imp]
        if v_curr < v_prev:
            caidas.append(f"{a_prev}→{a_curr} ({v_prev:,.0f} → {v_curr:,.0f})")
    if caidas:
        print(f"  ⚠️  {imp}: caídas detectadas → {', '.join(caidas)}")
        errores.append(f"Caída nominal en {imp}")
    else:
        print(f"  ✓  {imp}: monotonía nominal verificada")

# Ganancias aparte con nota explicativa
caidas_gan = []
for i in range(1, len(anios_ordenados)):
    a_prev = anios_ordenados[i - 1]
    a_curr = anios_ordenados[i]
    if RECAUDACION_ARCA[a_curr]['ganancias_total'] < RECAUDACION_ARCA[a_prev]['ganancias_total']:
        caidas_gan.append(f"{a_prev}→{a_curr}")
if caidas_gan:
    print(f"  ⚠️  ganancias_total: caídas → {caidas_gan}")
    print(f"      Posible causa: reforma tributaria. Verificar contra PDF ARCA.")
else:
    print(f"  ✓  ganancias_total: monotonía nominal verificada")

# --- Control 2: IVA mayor que Aportes en todos los años ---
print("\n▶ Control 2: IVA > Aportes personales en todos los años")
anomalias_iva = [
    a for a in anios_ordenados
    if RECAUDACION_ARCA[a]['iva'] < RECAUDACION_ARCA[a]['aportes_sipa']
]
if anomalias_iva:
    print(f"  ⚠️  IVA < Aportes en: {anomalias_iva} — verificar fuente")
    errores.append("IVA menor que Aportes")
else:
    print(f"  ✓  IVA > Aportes personales en todos los años")

# --- Control 3: Consistencia con Sección 12 ---
print("\n▶ Control 3: Bienes Personales consistente con Sección 12")
try:
    inconsistencias_bp = []
    for anio in anios_ordenados:
        bp_sec12  = RECAUDACION_BP_MILLONES[anio]
        bp_sec13  = RECAUDACION_ARCA[anio]['bienes_personales']
        if bp_sec12 != bp_sec13:
            inconsistencias_bp.append(
                f"  {anio}: Sección 12 = {bp_sec12:,} | Sección 13 = {bp_sec13:,}"
            )
    if inconsistencias_bp:
        print(f"  ⚠️  Inconsistencias encontradas:")
        for inc in inconsistencias_bp:
            print(inc)
        errores.append("BP inconsistente entre Sección 12 y 13")
    else:
        print(f"  ✓  Todos los valores de BP son idénticos en Secciones 12 y 13")
except NameError:
    print("  ℹ️  RECAUDACION_BP_MILLONES no está en scope — correr Sección 12 primero")

# --- Resumen ---
print()
print("=" * 65)
if errores:
    print(f"❌  {len(errores)} problema(s) detectado(s):")
    for e in errores:
        print(f"   • {e}")
    print("   Resolver antes de continuar con la Fase 3.")
else:
    print("✅  Todos los controles pasaron — Sección 13 lista.")
print("=" * 65)

VERIFICACIÓN DE CONSISTENCIA — SECCIÓN 13

▶ Control 1: Monotonía nominal (caídas nominales son sospechosas)


KeyError: 'iva'

In [39]:
# -----------------------------------------------------------------------------
# F. NOTA SOBRE IIBB PROVINCIAL (documentación de exclusión)
# -----------------------------------------------------------------------------
# Los Ingresos Brutos no se validan contra recaudación observada porque:
#
#   1. La recaudación de IIBB es provincial — cada una de las 24
#      jurisdicciones recauda y publica (o no) sus propios datos.
#      No existe una fuente consolidada oficial única.
#
#   2. La fuente más cercana a un consolidado nacional es el Ministerio
#      de Economía (MECON) — "Recursos de las provincias por tipo de
#      impuesto" — disponible en:
#      https://www.economia.gob.ar/hacienda/cgn/cuenta/2023/tomo1/
#      Pero su cobertura histórica es irregular antes de 2010.
#
#   3. Alternativa parcial: el IERAL (Fundación Mediterránea) publica
#      estimaciones de presión tributaria provincial que incluyen IIBB
#      consolidado. Disponible en sus informes anuales.
#
#   4. Para la validación de la incidencia de IIBB en el modelo se usa
#      el criterio inverso: en lugar de validar de abajo (incidencia) hacia
#      arriba (recaudación observada), se parte de estimaciones de presión
#      tributaria provincial total (Rossignolo 2014; CEPAL 2018) y se
#      verifica que la alícuota efectiva del 3.5%/3.8% produce resultados
#      consistentes con esas estimaciones.
#
# Esta decisión está documentada en retenciones_metodologia.txt
# (que se puede renombrar a limitaciones_metodologicas.txt para mayor claridad).
# -----------------------------------------------------------------------------

nota_iibb = """
INGRESOS BRUTOS (IIBB) — EXCLUSIÓN DE LA VALIDACIÓN POR RECAUDACIÓN OBSERVADA

Los Ingresos Brutos no se validan contra recaudación observada porque la
recaudación de IIBB es provincial, fragmentada en 24 jurisdicciones, y no
existe una fuente consolidada oficial de cobertura histórica completa.

ESTRATEGIA DE VALIDACIÓN ALTERNATIVA (Fase 5):
  Se comparan las alícuotas efectivas estimadas por el modelo (3.5% / 3.8%
  sobre el precio al consumidor) contra las estimaciones de presión
  tributaria provincial publicadas en:
    - Rossignolo (2014): "La presión tributaria sobre los sectores medios"
    - CEPAL (2018): "Panorama Fiscal de América Latina y el Caribe"
    - IERAL: "Situación fiscal de las provincias argentinas" (varios años)

FUENTE PARCIAL DISPONIBLE (opcional):
  MECON — "Recursos de las provincias por tipo de impuesto"
  https://www.economia.gob.ar/hacienda/cgn/cuenta/
  Cobertura: 2010–2024 (irregular antes de 2015).
  Si se incorpora en versiones futuras, agregar como columna
  'iibb_provincial_total_M' en recaudacion_arca_2004_2024.csv.
"""

with open(RUTAS['arca'] / "iibb_exclusion_validacion.txt", 'w', encoding='utf-8') as f:
    f.write(nota_iibb)

print("✅ Documentación de exclusión de IIBB guardada.")
print()

# --- CHECKLIST FINAL DE SECCIÓN 13 ---
print("=" * 65)
print("CHECKLIST — SECCIÓN 13")
print("=" * 65)
chk((RUTAS['arca'] / "recaudacion_arca_2004_2024.csv").exists(),
    "recaudacion_arca_2004_2024.csv generado")
chk((RUTAS['arca'] / "presion_tributaria_referencia_2004_2024.csv").exists(),
    "presion_tributaria_referencia_2004_2024.csv generado")
chk((RUTAS['arca'] / "iibb_exclusion_validacion.txt").exists(),
    "Documentación exclusión IIBB guardada")
chk(len(RECAUDACION_ARCA) == 21,
    f"Cobertura completa 2004–2024 ({len(RECAUDACION_ARCA)}/21 años)")
print()
print("Archivos generados en esta sección:")
print(f"  → data/raw/arca/recaudacion_arca_2004_2024.csv")
print(f"  → data/raw/arca/presion_tributaria_referencia_2004_2024.csv")
print(f"  → data/raw/arca/iibb_exclusion_validacion.txt")
print()
print("Uso previsto: Fase 5 — Validación de incidencia estimada")
print("  Para cada impuesto X y año Y:")
print("  incidencia_modelo(X,Y) debe estar dentro del ±15% de")
print("  recaudacion_observada(X,Y). Divergencias mayores → revisar supuestos.")

✅ Documentación de exclusión de IIBB guardada.

CHECKLIST — SECCIÓN 13
  ✅  recaudacion_arca_2004_2024.csv generado
  ✅  presion_tributaria_referencia_2004_2024.csv generado
  ✅  Documentación exclusión IIBB guardada
  ❌  Cobertura completa 2004–2024 (23/21 años)

Archivos generados en esta sección:
  → data/raw/arca/recaudacion_arca_2004_2024.csv
  → data/raw/arca/presion_tributaria_referencia_2004_2024.csv
  → data/raw/arca/iibb_exclusion_validacion.txt

Uso previsto: Fase 5 — Validación de incidencia estimada
  Para cada impuesto X y año Y:
  incidencia_modelo(X,Y) debe estar dentro del ±15% de
  recaudacion_observada(X,Y). Divergencias mayores → revisar supuestos.
